In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "destinations_clean.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset siap untuk eksperimen:", df.shape)
display(
    df[["Place_Name", "Category", "City", "search_text"]].head()
)

Dataset siap untuk eksperimen: (437, 12)


,Place_Name,Category,City,search_text
0,Monumen Nasional,Budaya,Jakarta,Monumen Nasional Budaya Jakarta Monumen Nasion...
1,Kota Tua,Budaya,Jakarta,"Kota Tua Budaya Jakarta Kota tua di Jakarta, y..."
2,Dunia Fantasi,Taman Hiburan,Jakarta,Dunia Fantasi Taman Hiburan Jakarta Dunia Fant...
3,Taman Mini Indonesia Indah (TMII),Taman Hiburan,Jakarta,Taman Mini Indonesia Indah (TMII) Taman Hibura...
4,Atlantis Water Adventure,Taman Hiburan,Jakarta,Atlantis Water Adventure Taman Hiburan Jakarta...


In [2]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(
    df["search_text"].fillna("")
)

print("Ukuran matriks TF-IDF:", tfidf_matrix.shape)
print("Jumlah fitur teks:", len(vectorizer.get_feature_names_out()))
print("Jumlah destinasi terindeks:", tfidf_matrix.shape[0])

Ukuran matriks TF-IDF: (437, 35063)
Jumlah fitur teks: 35063
Jumlah destinasi terindeks: 437


In [14]:
def cari_destinasi(query, top_k=5):
    query = query.strip()

    if not query:
        raise ValueError("Kueri pencarian tidak boleh kosong.")

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    hasil = df.copy()
    hasil["similarity_score"] = scores

    hasil = hasil.sort_values(
        by="similarity_score",
        ascending=False
    ).head(top_k)

    kolom_hasil = [
        "Place_Name",
        "Category",
        "City",
        "Price",
        "Rating",
        "similarity_score"
    ]

    hasil = hasil[kolom_hasil].copy()
    hasil["similarity_score"] = hasil["similarity_score"].round(4)

    return hasil.reset_index(drop=True)

In [15]:
def cari_destinasi_filter(
    query,
    kota=None,
    kategori=None,
    harga_maks=None,
    top_k=5
):
    query = query.strip()

    if not query:
        raise ValueError("Kueri pencarian tidak boleh kosong.")

    # Hitung kemiripan teks dengan metode yang sama
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    hasil = df.copy()
    hasil["similarity_score"] = scores

    # Filter kota jika dipilih
    if kota is not None:
        hasil = hasil[
            hasil["City"].str.casefold() == kota.casefold()
        ]

    # Filter kategori jika dipilih
    if kategori is not None:
        hasil = hasil[
            hasil["Category"].str.casefold() == kategori.casefold()
        ]

    # Filter batas harga jika ditentukan
    if harga_maks is not None:
        hasil = hasil[
            pd.to_numeric(hasil["Price"], errors="coerce")
            .le(harga_maks)
        ]

    # Urutkan berdasarkan skor kemiripan
    hasil = hasil.sort_values(
        "similarity_score",
        ascending=False
    ).head(top_k)

    kolom = [
        "Place_Name",
        "Category",
        "City",
        "Price",
        "Rating",
        "similarity_score"
    ]

    hasil = hasil[kolom].copy()
    hasil["similarity_score"] = hasil["similarity_score"].round(4)

    return hasil.reset_index(drop=True)

In [16]:
display(cari_destinasi("wisata alam di Yogyakarta"))

,Place_Name,Category,City,Price,Rating,similarity_score
0,Curug Cilengkrang,Cagar Alam,Bandung,7500,4.0,0.1277
1,Wisata Alam Kalibiru,Cagar Alam,Yogyakarta,20000,4.4,0.1001
2,Wisata Alam Mangrove Angke,Cagar Alam,Jakarta,25000,4.3,0.0951
3,Wisata Kaliurang,Cagar Alam,Yogyakarta,8000,4.4,0.0837
4,Pantai Parangtritis,Bahari,Yogyakarta,10000,4.5,0.0806


In [17]:
print("UJI 1: Taman hiburan di Bandung")
display(
    cari_destinasi_filter(
        "taman hiburan di Bandung",
        kota="Bandung",
        kategori="Taman Hiburan"
    )
)

print("UJI 2: Wisata murah di Semarang")
display(
    cari_destinasi_filter(
        "wisata murah di Semarang",
        kota="Semarang",
        harga_maks=20000
    )
)

print("UJI 3: Wisata budaya di Jakarta")
display(
    cari_destinasi_filter(
        "wisata budaya di Jakarta",
        kota="Jakarta",
        kategori="Budaya"
    )
)

UJI 1: Taman hiburan di Bandung


,Place_Name,Category,City,Price,Rating,similarity_score
0,Panama Park 825,Taman Hiburan,Bandung,50000,4.4,0.1158
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.1101
2,Taman Jomblo,Taman Hiburan,Bandung,10000,4.1,0.1079
3,Trans Studio Bandung,Taman Hiburan,Bandung,280000,4.5,0.0897
4,Taman Balai Kota Bandung,Taman Hiburan,Bandung,0,4.6,0.0873


UJI 2: Wisata murah di Semarang


,Place_Name,Category,City,Price,Rating,similarity_score
0,Pura Giri Natha,Budaya,Semarang,20000,4.7,0.1164
1,Brown Canyon,Cagar Alam,Semarang,0,4.2,0.1126
2,Semarang Contemporary Art Gallery,Budaya,Semarang,10000,4.6,0.1026
3,Lawang Sewu,Budaya,Semarang,10000,4.6,0.0993
4,Grand Maerakaca,Taman Hiburan,Semarang,15000,4.4,0.0972


UJI 3: Wisata budaya di Jakarta


,Place_Name,Category,City,Price,Rating,similarity_score
0,Museum Nasional,Budaya,Jakarta,5000,4.6,0.0756
1,Museum Joang 45,Budaya,Jakarta,2000,4.0,0.0667
2,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,4.4,0.0661
3,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,4.5,0.0655
4,Museum Wayang,Budaya,Jakarta,5000,4.5,0.0577


In [18]:
# UJI 4: Kueri kosong
try:
    cari_destinasi_filter("")
except ValueError as e:
    print("Kueri kosong ditangani:", e)

# UJI 5: Kota yang tidak tersedia
hasil_kota = cari_destinasi_filter(
    "wisata budaya",
    kota="Kota Contoh Tidak Ada"
)

print("\nJumlah hasil untuk kota yang tidak tersedia:", len(hasil_kota))
display(hasil_kota)

# UJI 6: Filter harga
hasil_harga = cari_destinasi_filter(
    "wisata",
    kota="Semarang",
    harga_maks=10000
)

print("\nJumlah hasil Semarang dengan harga maksimal Rp10.000:",
      len(hasil_harga))

display(hasil_harga)

Kueri kosong ditangani: Kueri pencarian tidak boleh kosong.

Jumlah hasil untuk kota yang tidak tersedia: 0


,Place_Name,Category,City,Price,Rating,similarity_score



Jumlah hasil Semarang dengan harga maksimal Rp10.000: 5


,Place_Name,Category,City,Price,Rating,similarity_score
0,Desa Wisata Lembah Kalipancur,Taman Hiburan,Semarang,0,3.9,0.0839
1,Wisata Mangrove Tapak,Cagar Alam,Semarang,5000,4.3,0.0685
2,Wisata Alam Wana Wisata Penggaron,Cagar Alam,Semarang,10000,4.1,0.0639
3,Wisata Lereng Kelir,Cagar Alam,Semarang,7000,4.3,0.0633
4,Hutan Wisata Tinjomoyo Semarang,Cagar Alam,Semarang,3000,4.3,0.0604


In [19]:

# UJI 7: Menguji relevansi untuk beberapa jenis kueri

daftar_kueri = [
    "pantai di Yogyakarta",
    "museum budaya di Jakarta",
    "wisata alam di Semarang",
    "taman hiburan untuk keluarga",
    "qzxwxyz tidak ada"
]

for query in daftar_kueri:
    print("\n" + "=" * 70)
    print("KUERI:", query)

    hasil = cari_destinasi_filter(query, top_k=5)

    if hasil.empty:
        print("Tidak ada destinasi yang cocok dengan filter.")
    else:
        display(
            hasil[
                [
                    "Place_Name",
                    "Category",
                    "City",
                    "Price",
                    "Rating",
                    "similarity_score"
                ]
            ]
        )

        skor_tertinggi = hasil["similarity_score"].max()
        print("Skor tertinggi:", skor_tertinggi)

        if skor_tertinggi == 0:
            print("PERHATIAN: Tidak ada kecocokan kata yang ditemukan.")



KUERI: pantai di Yogyakarta


,Place_Name,Category,City,Price,Rating,similarity_score
0,Pantai Glagah,Bahari,Yogyakarta,5000,4.4,0.2245
1,Pantai Parangtritis,Bahari,Yogyakarta,10000,4.5,0.2034
2,Pantai Timang,Bahari,Yogyakarta,10000,4.7,0.1776
3,Pantai Sepanjang,Bahari,Yogyakarta,10000,4.4,0.1504
4,Pantai Jungwok,Bahari,Yogyakarta,5000,4.6,0.1375


Skor tertinggi: 0.2245

KUERI: museum budaya di Jakarta


,Place_Name,Category,City,Price,Rating,similarity_score
0,Museum Nasional,Budaya,Jakarta,5000,4.6,0.1319
1,Bentara Budaya Yogyakarta (BBY),Budaya,Yogyakarta,0,4.5,0.1260
2,Museum Wayang,Budaya,Jakarta,5000,4.5,0.1204
3,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,4.4,0.1171
4,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,4.5,0.1105


Skor tertinggi: 0.1319

KUERI: wisata alam di Semarang


...

Skor tertinggi: 0.1521

KUERI: taman hiburan untuk keluarga


,Place_Name,Category,City,Price,Rating,similarity_score
0,Rainbow Garden,Cagar Alam,Bandung,20000,4.6,0.1507
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.0785
2,Taman Hiburan Rakyat,Taman Hiburan,Surabaya,5000,4.2,0.0769
3,Taman Impian Jaya Ancol,Taman Hiburan,Jakarta,25000,4.5,0.0732
4,Grand Maerakaca,Taman Hiburan,Semarang,15000,4.4,0.0713


Skor tertinggi: 0.1507

KUERI: qzxwxyz tidak ada


,Place_Name,Category,City,Price,Rating,similarity_score
0,Pantai Baruna,Bahari,Semarang,3000,4.0,0.1784
1,Air Terjun Kali Pancur,Cagar Alam,Semarang,5000,4.1,0.1166
2,Kampung Batik Gedong Semarang,Budaya,Semarang,0,4.4,0.1165
3,Monumen Serangan Umum 1 Maret,Taman Hiburan,Yogyakarta,3000,4.6,0.0943
4,Museum De Javasche Bank,Budaya,Surabaya,5000,4.6,0.0900


Skor tertinggi: 0.1784


In [20]:

import re
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Kata umum yang tidak cukup untuk menentukan kebutuhan wisata.
STOPWORDS_KUERi = {
    "di", "ke", "dari", "dan", "atau", "untuk", "yang",
    "dengan", "ada", "tidak", "ini", "itu", "pada",
    "adalah", "wisata"
}

KOLOM_HASIL = [
    "Place_Name", "Category", "City",
    "Price", "Rating", "similarity_score"
]

def cari_destinasi_v2(
    query,
    kota=None,
    kategori=None,
    harga_maks=None,
    top_k=5
):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Kueri pencarian tidak boleh kosong.")

    # Ambil kata-kata yang bermakna dari kueri.
    tokens = re.findall(r"\b\w+\b", query.lower())
    token_penting = [
        token for token in tokens
        if token not in STOPWORDS_KUERi
        and not token.isdigit()
    ]

    # Tolak kueri jika semua kata penting tidak dikenal.
    vocab = vectorizer.vocabulary_
    token_dikenal = [token for token in token_penting if token in vocab]

    if not token_penting or not token_dikenal:
        return pd.DataFrame(columns=KOLOM_HASIL)

    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    hasil = df.copy()
    hasil["similarity_score"] = scores

    # Jangan tampilkan destinasi dengan skor nol.
    hasil = hasil[hasil["similarity_score"] > 0].copy()

    if kota is not None:
        hasil = hasil[
            hasil["City"].str.casefold() == kota.casefold()
        ]

    if kategori is not None:
        hasil = hasil[
            hasil["Category"].str.casefold() == kategori.casefold()
        ]

    if harga_maks is not None:
        hasil = hasil[
            pd.to_numeric(hasil["Price"], errors="coerce")
            .le(harga_maks)
        ]

    hasil = hasil.sort_values(
        "similarity_score", ascending=False
    ).head(top_k)

    hasil = hasil[KOLOM_HASIL].copy()
    hasil["similarity_score"] = hasil["similarity_score"].round(4)

    return hasil.reset_index(drop=True)

print("Fungsi pencarian versi 2 berhasil dibuat.")


Fungsi pencarian versi 2 berhasil dibuat.


In [21]:

# UJI A: Kueri dengan kata acak
hasil_acak = cari_destinasi_v2("qzxwxyz tidak ada")
print("Hasil kueri acak:", len(hasil_acak))
display(hasil_acak)

# UJI B: Kueri pantai yang relevan
hasil_pantai = cari_destinasi_v2("pantai di Yogyakarta")
print("\nHasil kueri pantai:")
display(hasil_pantai)

# UJI C: Kueri dengan filter kota dan kategori
hasil_museum = cari_destinasi_v2(
    "museum budaya di Jakarta",
    kota="Jakarta",
    kategori="Budaya"
)
print("\nHasil museum di Jakarta:")
display(hasil_museum)

# UJI D: Kueri kosong
try:
    cari_destinasi_v2("")
except ValueError as e:
    print("\nValidasi kueri kosong:", e)


Hasil kueri acak: 0


,Place_Name,Category,City,Price,Rating,similarity_score



Hasil kueri pantai:


,Place_Name,Category,City,Price,Rating,similarity_score
0,Pantai Glagah,Bahari,Yogyakarta,5000,4.4,0.2245
1,Pantai Parangtritis,Bahari,Yogyakarta,10000,4.5,0.2034
2,Pantai Timang,Bahari,Yogyakarta,10000,4.7,0.1776
3,Pantai Sepanjang,Bahari,Yogyakarta,10000,4.4,0.1504
4,Pantai Jungwok,Bahari,Yogyakarta,5000,4.6,0.1375



Hasil museum di Jakarta:


,Place_Name,Category,City,Price,Rating,similarity_score
0,Museum Nasional,Budaya,Jakarta,5000,4.6,0.1319
1,Museum Wayang,Budaya,Jakarta,5000,4.5,0.1204
2,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,4.4,0.1171
3,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,4.5,0.1105
4,Museum Joang 45,Budaya,Jakarta,2000,4.0,0.1077



Validasi kueri kosong: Kueri pencarian tidak boleh kosong.


In [22]:

# Perbaikan: kata "wisata" tetap boleh menjadi kueri pencarian
STOPWORDS_KUERi.discard("wisata")

print("Kata 'wisata' sudah dikeluarkan dari stopwords.")


Kata 'wisata' sudah dikeluarkan dari stopwords.


In [23]:

daftar_uji_relevansi = [
    "wisata",
    "pantai di Yogyakarta",
    "museum budaya di Jakarta",
    "wisata untuk keluarga",
    "qzxwxyz"
]

for query in daftar_uji_relevansi:
    print("\n" + "=" * 60)
    print("KUERI:", query)

    hasil = cari_destinasi_v2(query, top_k=3)

    if hasil.empty:
        print("Tidak ada hasil.")
        continue

    print("Skor tertinggi:", hasil["similarity_score"].max())
    display(hasil)



KUERI: wisata
Skor tertinggi: 0.091


,Place_Name,Category,City,Price,Rating,similarity_score
0,Wisata Kaliurang,Cagar Alam,Yogyakarta,8000,4.4,0.0910
1,Ekowisata Mangrove Wonorejo,Cagar Alam,Surabaya,0,4.3,0.0860
2,Desa Wisata Lembah Kalipancur,Taman Hiburan,Semarang,0,3.9,0.0839



KUERI: pantai di Yogyakarta
Skor tertinggi: 0.2245


,Place_Name,Category,City,Price,Rating,similarity_score
0,Pantai Glagah,Bahari,Yogyakarta,5000,4.4,0.2245
1,Pantai Parangtritis,Bahari,Yogyakarta,10000,4.5,0.2034
2,Pantai Timang,Bahari,Yogyakarta,10000,4.7,0.1776



KUERI: museum budaya di Jakarta
Skor tertinggi: 0.1319


,Place_Name,Category,City,Price,Rating,similarity_score
0,Museum Nasional,Budaya,Jakarta,5000,4.6,0.1319
1,Bentara Budaya Yogyakarta (BBY),Budaya,Yogyakarta,0,4.5,0.1260
2,Museum Wayang,Budaya,Jakarta,5000,4.5,0.1204



KUERI: wisata untuk keluarga
Skor tertinggi: 0.1265


,Place_Name,Category,City,Price,Rating,similarity_score
0,Rainbow Garden,Cagar Alam,Bandung,20000,4.6,0.1265
1,Tafso Barn,Cagar Alam,Bandung,0,4.2,0.0709
2,Kebun Teh Nglinggo,Cagar Alam,Yogyakarta,6000,4.5,0.0505



KUERI: qzxwxyz
Tidak ada hasil.


In [33]:
def cari_destinasi_v3(query, top_k=5, harga_maks=None):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Kueri pencarian tidak boleh kosong.")

    # Deteksi kota dan kategori dari kueri
    kota_terdeteksi = deteksi_kota(query)
    kategori_terdeteksi = deteksi_kategori(query)

    # Jika pengguna mengetik "murah", gunakan batas Rp20.000
    query_lower = query.casefold()

    if harga_maks is None and "murah" in query_lower.split():
        harga_maks = 20000

    # Jalankan pencarian beserta filter yang terdeteksi
    hasil = cari_destinasi_v2(
        query,
        kota=kota_terdeteksi,
        kategori=kategori_terdeteksi,
        harga_maks=harga_maks,
        top_k=top_k
    )

    print("Kota terdeteksi:", kota_terdeteksi or "Tidak ada")
    print("Kategori terdeteksi:", kategori_terdeteksi or "Tidak ada")
    print("Batas harga:", f"Rp{harga_maks:,}" if harga_maks is not None else "Tidak ada")

    return hasil

In [25]:

kueri_uji_v3 = [
    "museum budaya di Jakarta",
    "wisata alam di Semarang",
    "pantai di Yogyakarta",
    "taman hiburan di Bandung"
]

for query in kueri_uji_v3:
    print("\n" + "=" * 60)
    print("KUERI:", query)

    hasil = cari_destinasi_v3(query, top_k=5)
    display(hasil)



KUERI: museum budaya di Jakarta
Kota terdeteksi: Jakarta
Kategori terdeteksi: Budaya


,Place_Name,Category,City,Price,Rating,similarity_score
0,Museum Nasional,Budaya,Jakarta,5000,4.6,0.1319
1,Museum Wayang,Budaya,Jakarta,5000,4.5,0.1204
2,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,4.4,0.1171
3,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,4.5,0.1105
4,Museum Joang 45,Budaya,Jakarta,2000,4.0,0.1077



KUERI: wisata alam di Semarang
Kota terdeteksi: Semarang
Kategori terdeteksi: Cagar Alam


,Place_Name,Category,City,Price,Rating,similarity_score
0,Umbul Sidomukti,Cagar Alam,Semarang,75000,4.6,0.1521
1,Brown Canyon,Cagar Alam,Semarang,0,4.2,0.1060
2,Danau Rawa Pening,Cagar Alam,Semarang,15000,4.4,0.0982
3,Air Terjun Kali Pancur,Cagar Alam,Semarang,5000,4.1,0.0929
4,Watu Gunung Ungaran,Cagar Alam,Semarang,25000,4.4,0.0919



KUERI: pantai di Yogyakarta
Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Bahari


,Place_Name,Category,City,Price,Rating,similarity_score
0,Pantai Glagah,Bahari,Yogyakarta,5000,4.4,0.2245
1,Pantai Parangtritis,Bahari,Yogyakarta,10000,4.5,0.2034
2,Pantai Timang,Bahari,Yogyakarta,10000,4.7,0.1776
3,Pantai Sepanjang,Bahari,Yogyakarta,10000,4.4,0.1504
4,Pantai Jungwok,Bahari,Yogyakarta,5000,4.6,0.1375



KUERI: taman hiburan di Bandung
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan


,Place_Name,Category,City,Price,Rating,similarity_score
0,Panama Park 825,Taman Hiburan,Bandung,50000,4.4,0.1158
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.1101
2,Taman Jomblo,Taman Hiburan,Bandung,10000,4.1,0.1079
3,Trans Studio Bandung,Taman Hiburan,Bandung,280000,4.5,0.0897
4,Taman Balai Kota Bandung,Taman Hiburan,Bandung,0,4.6,0.0873


In [26]:

from pathlib import Path
import pandas as pd

# Kumpulan kueri untuk evaluasi relevansi.
kueri_evaluasi = [
    "museum di Jakarta",
    "pantai di Yogyakarta",
    "wisata alam di Semarang",
    "taman hiburan di Bandung",
    "wisata di Yogyakarta",
    "budaya di Bandung",
    "tempat ibadah di Surabaya",
    "wisata murah di Semarang"
]

daftar_kandidat = []

for query in kueri_evaluasi:
    hasil = cari_destinasi_v3(query, top_k=5)

    for peringkat, (_, baris) in enumerate(
        hasil.iterrows(), start=1
    ):
        daftar_kandidat.append({
            "query": query,
            "rank": peringkat,
            "Place_Id": df.loc[
                df["Place_Name"] == baris["Place_Name"],
                "Place_Id"
            ].iloc[0],
            "Place_Name": baris["Place_Name"],
            "Category": baris["Category"],
            "City": baris["City"],
            "Price": baris["Price"],
            "Rating": baris["Rating"],
            "similarity_score": baris["similarity_score"],
            "relevant": ""
        })

kandidat_df = pd.DataFrame(daftar_kandidat)

folder_output = Path("../data/processed")
folder_output.mkdir(parents=True, exist_ok=True)

file_output = folder_output / "kandidat_evaluasi.csv"
kandidat_df.to_csv(file_output, index=False)

print("Jumlah kueri:", len(kueri_evaluasi))
print("Jumlah kandidat:", len(kandidat_df))
print("File tersimpan di:", file_output.resolve())

display(kandidat_df.head(10))


Kota terdeteksi: Jakarta
Kategori terdeteksi: Budaya
Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Bahari
Kota terdeteksi: Semarang
Kategori terdeteksi: Cagar Alam
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Tidak ada
Kota terdeteksi: Bandung
Kategori terdeteksi: Budaya
Kota terdeteksi: Surabaya
Kategori terdeteksi: Tempat Ibadah
Kota terdeteksi: Semarang
Kategori terdeteksi: Tidak ada
Jumlah kueri: 8
Jumlah kandidat: 38
File tersimpan di: D:\WisataFinder\data\processed\kandidat_evaluasi.csv


,query,rank,Place_Id,Place_Name,Category,City,Price,Rating,similarity_score,relevant
0,museum di Jakarta,1,42,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,4.4,0.2207,
1,museum di Jakarta,2,24,Museum Nasional,Budaya,Jakarta,5000,4.6,0.1666,
2,museum di Jakarta,3,21,Museum Wayang,Budaya,Jakarta,5000,4.5,0.1436,
3,museum di Jakarta,4,39,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,4.5,0.1393,
4,museum di Jakarta,5,50,Museum Joang 45,Budaya,Jakarta,2000,4.0,0.1366,
5,pantai di Yogyakarta,1,186,Pantai Glagah,Bahari,Yogyakarta,5000,4.4,0.2245,
6,pantai di Yogyakarta,2,177,Pantai Parangtritis,Bahari,Yogyakarta,10000,4.5,0.2034,
7,pantai di Yogyakarta,3,168,Pantai Timang,Bahari,Yogyakarta,10000,4.7,0.1776,
8,pantai di Yogyakarta,4,184,Pantai Sepanjang,Bahari,Yogyakarta,10000,4.4,0.1504,
9,pantai di Yogyakarta,5,197,Pantai Jungwok,Bahari,Yogyakarta,5000,4.6,0.1375,


In [27]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
FILE_INPUT = PROJECT_ROOT / "data" / "processed" / "kandidat_evaluasi.csv"

evaluasi = pd.read_csv(FILE_INPUT)

print("Jumlah baris:", len(evaluasi))
print("Nama kolom:", evaluasi.columns.tolist())

display(evaluasi.head(10))

Jumlah baris: 38
Nama kolom: ['query', 'rank', 'Place_Id', 'Place_Name', 'Category', 'City', 'Price', 'Rating', 'similarity_score', 'relevant']


,query,rank,Place_Id,Place_Name,Category,City,Price,Rating,similarity_score,relevant
0,museum di Jakarta,1,42,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,4.4,0.2207,NaN
1,museum di Jakarta,2,24,Museum Nasional,Budaya,Jakarta,5000,4.6,0.1666,NaN
2,museum di Jakarta,3,21,Museum Wayang,Budaya,Jakarta,5000,4.5,0.1436,NaN
3,museum di Jakarta,4,39,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,4.5,0.1393,NaN
4,museum di Jakarta,5,50,Museum Joang 45,Budaya,Jakarta,2000,4.0,0.1366,NaN
5,pantai di Yogyakarta,1,186,Pantai Glagah,Bahari,Yogyakarta,5000,4.4,0.2245,NaN
6,pantai di Yogyakarta,2,177,Pantai Parangtritis,Bahari,Yogyakarta,10000,4.5,0.2034,NaN
7,pantai di Yogyakarta,3,168,Pantai Timang,Bahari,Yogyakarta,10000,4.7,0.1776,NaN
8,pantai di Yogyakarta,4,184,Pantai Sepanjang,Bahari,Yogyakarta,10000,4.4,0.1504,NaN
9,pantai di Yogyakarta,5,197,Pantai Jungwok,Bahari,Yogyakarta,5000,4.6,0.1375,NaN


In [28]:
# Isi label relevansi berdasarkan kueri dan hasil pencarian.
# 1 = relevan, 0 = tidak relevan.

evaluasi["relevant"] = pd.NA

for i, baris in evaluasi.iterrows():
    query = str(baris["query"]).strip().casefold()
    nama = str(baris["Place_Name"]).strip().casefold()
    kategori = str(baris["Category"]).strip().casefold()
    kota = str(baris["City"]).strip().casefold()
    harga = pd.to_numeric(baris["Price"], errors="coerce")

    relevan = False

    if query == "museum di jakarta":
        relevan = kota == "jakarta" and "museum" in nama

    elif query == "pantai di yogyakarta":
        relevan = kota == "yogyakarta" and "pantai" in nama

    elif query == "wisata alam di semarang":
        relevan = (
            kota == "semarang"
            and kategori == "cagar alam"
        )

    elif query == "taman hiburan di bandung":
        relevan = (
            kota == "bandung"
            and kategori == "taman hiburan"
        )

    elif query == "wisata di yogyakarta":
        relevan = kota == "yogyakarta"

    elif query == "budaya di bandung":
        relevan = (
            kota == "bandung"
            and kategori == "budaya"
        )

    elif query == "tempat ibadah di surabaya":
        relevan = (
            kota == "surabaya"
            and kategori == "tempat ibadah"
        )

    elif query == "wisata murah di semarang":
        relevan = (
            kota == "semarang"
            and pd.notna(harga)
            and harga <= 20000
        )

    evaluasi.at[i, "relevant"] = int(relevan)

evaluasi["relevant"] = evaluasi["relevant"].astype(int)

print("Jumlah label 1:", (evaluasi["relevant"] == 1).sum())
print("Jumlah label 0:", (evaluasi["relevant"] == 0).sum())

display(
    evaluasi[["query", "rank", "Place_Name", "Price", "relevant"]]
)

Jumlah label 1: 37
Jumlah label 0: 1


,query,rank,Place_Name,Price,relevant
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,5000,1
1,museum di Jakarta,2,Museum Nasional,5000,1
2,museum di Jakarta,3,Museum Wayang,5000,1
3,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,100000,1
4,museum di Jakarta,5,Museum Joang 45,2000,1
5,pantai di Yogyakarta,1,Pantai Glagah,5000,1
6,pantai di Yogyakarta,2,Pantai Parangtritis,10000,1
7,pantai di Yogyakarta,3,Pantai Timang,10000,1
8,pantai di Yogyakarta,4,Pantai Sepanjang,10000,1
9,pantai di Yogyakarta,5,Pantai Jungwok,5000,1


In [30]:
FILE_OUTPUT_BARU = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "kandidat_evaluasi_berlabel.csv"
)

evaluasi.to_csv(
    FILE_OUTPUT_BARU,
    index=False,
    encoding="utf-8-sig"
)

print("File berhasil disimpan!")
print(FILE_OUTPUT_BARU)

File berhasil disimpan!
D:\WisataFinder\data\processed\kandidat_evaluasi_berlabel.csv


In [31]:
cek = pd.read_csv(FILE_OUTPUT_BARU)

print("Jumlah kandidat:", len(cek))
print("Label kosong:", cek["relevant"].isna().sum())
print("Label 1:", (cek["relevant"] == 1).sum())
print("Label 0:", (cek["relevant"] == 0).sum())

display(
    cek[["query", "rank", "Place_Name", "relevant"]].head(10)
)

Jumlah kandidat: 38
Label kosong: 0
Label 1: 37
Label 0: 1


,query,rank,Place_Name,relevant
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,1
1,museum di Jakarta,2,Museum Nasional,1
2,museum di Jakarta,3,Museum Wayang,1
3,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,1
4,museum di Jakarta,5,Museum Joang 45,1
5,pantai di Yogyakarta,1,Pantai Glagah,1
6,pantai di Yogyakarta,2,Pantai Parangtritis,1
7,pantai di Yogyakarta,3,Pantai Timang,1
8,pantai di Yogyakarta,4,Pantai Sepanjang,1
9,pantai di Yogyakarta,5,Pantai Jungwok,1


In [32]:
import pandas as pd

# Baca file evaluasi yang sudah diberi label
evaluasi = pd.read_csv(FILE_OUTPUT_BARU)

evaluasi["relevant"] = pd.to_numeric(
    evaluasi["relevant"], errors="raise"
)

# Pastikan label hanya 0 atau 1
assert evaluasi["relevant"].isin([0, 1]).all()
assert evaluasi["relevant"].notna().all()

# Hitung Precision@5 untuk setiap kueri
hasil_evaluasi = (
    evaluasi.groupby("query")
    .agg(
        jumlah_hasil=("relevant", "size"),
        jumlah_relevan=("relevant", "sum")
    )
    .reset_index()
)

# Precision@5 menggunakan penyebut 5,
# meskipun sistem menghasilkan kurang dari 5 hasil.
hasil_evaluasi["Precision@5"] = (
    hasil_evaluasi["jumlah_relevan"] / 5
)

print("HASIL EVALUASI PER KUERI")
display(hasil_evaluasi)

# Rata-rata Precision@5 dari seluruh kueri
rata_rata = hasil_evaluasi["Precision@5"].mean()

print(f"Rata-rata Precision@5: {rata_rata:.4f}")
print(f"Rata-rata dalam persen: {rata_rata * 100:.2f}%")

HASIL EVALUASI PER KUERI


,query,jumlah_hasil,jumlah_relevan,Precision@5
0,budaya di Bandung,5,5,1.0
1,museum di Jakarta,5,5,1.0
2,pantai di Yogyakarta,5,5,1.0
3,taman hiburan di Bandung,5,5,1.0
4,tempat ibadah di Surabaya,3,3,0.6
5,wisata alam di Semarang,5,5,1.0
6,wisata di Yogyakarta,5,5,1.0
7,wisata murah di Semarang,5,4,0.8


Rata-rata Precision@5: 0.9250
Rata-rata dalam persen: 92.50%


In [34]:
hasil_murah = cari_destinasi_v3(
    "wisata murah di Semarang",
    top_k=5
)

display(hasil_murah)

Kota terdeteksi: Semarang
Kategori terdeteksi: Tidak ada
Batas harga: Rp20,000


,Place_Name,Category,City,Price,Rating,similarity_score
0,Pura Giri Natha,Budaya,Semarang,20000,4.7,0.1164
1,Brown Canyon,Cagar Alam,Semarang,0,4.2,0.1126
2,Semarang Contemporary Art Gallery,Budaya,Semarang,10000,4.6,0.1026
3,Lawang Sewu,Budaya,Semarang,10000,4.6,0.0993
4,Grand Maerakaca,Taman Hiburan,Semarang,15000,4.4,0.0972


In [35]:
hasil_pantai = cari_destinasi_v3(
    "pantai di Yogyakarta",
    top_k=5
)

display(hasil_pantai)

Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Bahari
Batas harga: Tidak ada


,Place_Name,Category,City,Price,Rating,similarity_score
0,Pantai Glagah,Bahari,Yogyakarta,5000,4.4,0.2245
1,Pantai Parangtritis,Bahari,Yogyakarta,10000,4.5,0.2034
2,Pantai Timang,Bahari,Yogyakarta,10000,4.7,0.1776
3,Pantai Sepanjang,Bahari,Yogyakarta,10000,4.4,0.1504
4,Pantai Jungwok,Bahari,Yogyakarta,5000,4.6,0.1375


In [36]:
from pathlib import Path
import pandas as pd

kueri_evaluasi = [
    "museum di Jakarta",
    "pantai di Yogyakarta",
    "wisata alam di Semarang",
    "taman hiburan di Bandung",
    "wisata di Yogyakarta",
    "budaya di Bandung",
    "tempat ibadah di Surabaya",
    "wisata murah di Semarang"
]

daftar_kandidat_baru = []

for query in kueri_evaluasi:
    hasil = cari_destinasi_v3(query, top_k=5)

    for peringkat, (_, baris) in enumerate(
        hasil.iterrows(), start=1
    ):
        daftar_kandidat_baru.append({
            "query": query,
            "rank": peringkat,
            "Place_Id": df.loc[
                df["Place_Name"] == baris["Place_Name"],
                "Place_Id"
            ].iloc[0],
            "Place_Name": baris["Place_Name"],
            "Category": baris["Category"],
            "City": baris["City"],
            "Price": baris["Price"],
            "Rating": baris["Rating"],
            "similarity_score": baris["similarity_score"]
        })

kandidat_baru = pd.DataFrame(daftar_kandidat_baru)

# Simpan sebagai file baru, tanpa menimpa file sebelumnya
file_baru = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "kandidat_evaluasi_perbaikan.csv"
)

kandidat_baru.to_csv(
    file_baru,
    index=False,
    encoding="utf-8-sig"
)

print("Jumlah kueri:", len(kueri_evaluasi))
print("Jumlah kandidat:", len(kandidat_baru))
print("File tersimpan di:", file_baru.resolve())

display(
    kandidat_baru[
        ["query", "rank", "Place_Name", "City", "Price"]
    ]
)

Kota terdeteksi: Jakarta
Kategori terdeteksi: Budaya
Batas harga: Tidak ada
Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Bahari
Batas harga: Tidak ada
Kota terdeteksi: Semarang
Kategori terdeteksi: Cagar Alam
Batas harga: Tidak ada
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada
Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Tidak ada
Batas harga: Tidak ada
Kota terdeteksi: Bandung
Kategori terdeteksi: Budaya
Batas harga: Tidak ada
Kota terdeteksi: Surabaya
Kategori terdeteksi: Tempat Ibadah
Batas harga: Tidak ada
Kota terdeteksi: Semarang
Kategori terdeteksi: Tidak ada
Batas harga: Rp20,000
Jumlah kueri: 8
Jumlah kandidat: 38
File tersimpan di: D:\WisataFinder\data\processed\kandidat_evaluasi_perbaikan.csv


,query,rank,Place_Name,City,Price
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,Jakarta,5000
1,museum di Jakarta,2,Museum Nasional,Jakarta,5000
2,museum di Jakarta,3,Museum Wayang,Jakarta,5000
3,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,Jakarta,100000
4,museum di Jakarta,5,Museum Joang 45,Jakarta,2000
5,pantai di Yogyakarta,1,Pantai Glagah,Yogyakarta,5000
6,pantai di Yogyakarta,2,Pantai Parangtritis,Yogyakarta,10000
7,pantai di Yogyakarta,3,Pantai Timang,Yogyakarta,10000
8,pantai di Yogyakarta,4,Pantai Sepanjang,Yogyakarta,10000
9,pantai di Yogyakarta,5,Pantai Jungwok,Yogyakarta,5000


In [37]:
# Beri label awal pada kandidat hasil pencarian terbaru
kandidat_baru["relevant"] = 0

for i, baris in kandidat_baru.iterrows():
    query = baris["query"].casefold()
    nama = str(baris["Place_Name"]).casefold()
    kategori = str(baris["Category"]).casefold()
    kota = str(baris["City"]).casefold()
    harga = pd.to_numeric(baris["Price"], errors="coerce")

    if query == "museum di jakarta":
        relevan = kota == "jakarta" and "museum" in nama

    elif query == "pantai di yogyakarta":
        relevan = kota == "yogyakarta" and "pantai" in nama

    elif query == "wisata alam di semarang":
        relevan = kota == "semarang" and kategori == "cagar alam"

    elif query == "taman hiburan di bandung":
        relevan = kota == "bandung" and kategori == "taman hiburan"

    elif query == "wisata di yogyakarta":
        relevan = kota == "yogyakarta"

    elif query == "budaya di bandung":
        relevan = kota == "bandung" and kategori == "budaya"

    elif query == "tempat ibadah di surabaya":
        relevan = kota == "surabaya" and kategori == "tempat ibadah"

    elif query == "wisata murah di semarang":
        relevan = (
            kota == "semarang"
            and pd.notna(harga)
            and harga <= 20000
        )

    else:
        relevan = False

    kandidat_baru.at[i, "relevant"] = int(relevan)

# Simpan kandidat yang sudah diberi label
kandidat_baru["relevant"] = kandidat_baru["relevant"].astype(int)

kandidat_baru.to_csv(
    file_baru,
    index=False,
    encoding="utf-8-sig"
)

print("Label kosong:", kandidat_baru["relevant"].isna().sum())
print("Jumlah relevan:", kandidat_baru["relevant"].sum())
print("Jumlah tidak relevan:", (kandidat_baru["relevant"] == 0).sum())

display(
    kandidat_baru[
        ["query", "rank", "Place_Name", "Price", "relevant"]
    ]
)

Label kosong: 0
Jumlah relevan: 38
Jumlah tidak relevan: 0


,query,rank,Place_Name,Price,relevant
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,5000,1
1,museum di Jakarta,2,Museum Nasional,5000,1
2,museum di Jakarta,3,Museum Wayang,5000,1
3,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,100000,1
4,museum di Jakarta,5,Museum Joang 45,2000,1
5,pantai di Yogyakarta,1,Pantai Glagah,5000,1
6,pantai di Yogyakarta,2,Pantai Parangtritis,10000,1
7,pantai di Yogyakarta,3,Pantai Timang,10000,1
8,pantai di Yogyakarta,4,Pantai Sepanjang,10000,1
9,pantai di Yogyakarta,5,Pantai Jungwok,5000,1


In [38]:
# Membuat kandidat untuk menguji relevansi TF-IDF
# tanpa filter kota dan kategori otomatis.

kueri_uji_relevansi = [
    "museum di Jakarta",
    "pantai di Yogyakarta",
    "wisata alam di Semarang",
    "taman hiburan di Bandung",
    "wisata di Yogyakarta",
    "budaya di Bandung",
    "tempat ibadah di Surabaya",
    "wisata murah di Semarang"
]

kandidat_relevansi = []

for query in kueri_uji_relevansi:
    # Menggunakan pencarian dasar tanpa filter kota/kategori/harga
    hasil = cari_destinasi_v2(query, top_k=10)

    for rank, (_, baris) in enumerate(hasil.iterrows(), start=1):
        kandidat_relevansi.append({
            "query": query,
            "rank": rank,
            "Place_Name": baris["Place_Name"],
            "Category": baris["Category"],
            "City": baris["City"],
            "Price": baris["Price"],
            "similarity_score": baris["similarity_score"],
            "relevant": ""
        })

kandidat_relevansi_df = pd.DataFrame(kandidat_relevansi)

file_uji = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "kandidat_uji_relevansi.csv"
)

kandidat_relevansi_df.to_csv(
    file_uji,
    index=False,
    encoding="utf-8-sig"
)

print("Jumlah kueri:", len(kueri_uji_relevansi))
print("Jumlah kandidat:", len(kandidat_relevansi_df))
print("File tersimpan di:", file_uji.resolve())

display(kandidat_relevansi_df.head(15))

Jumlah kueri: 8
Jumlah kandidat: 80
File tersimpan di: D:\WisataFinder\data\processed\kandidat_uji_relevansi.csv


,query,rank,Place_Name,Category,City,Price,similarity_score,relevant
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,0.2207,
1,museum di Jakarta,2,Museum Nasional,Budaya,Jakarta,5000,0.1666,
2,museum di Jakarta,3,Museum Wayang,Budaya,Jakarta,5000,0.1436,
3,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,0.1393,
4,museum di Jakarta,5,Museum Benteng Vredeburg Yogyakarta,Budaya,Yogyakarta,3000,0.1368,
5,museum di Jakarta,6,Museum Joang 45,Budaya,Jakarta,2000,0.1366,
6,museum di Jakarta,7,Taman Impian Jaya Ancol,Taman Hiburan,Jakarta,25000,0.1232,
7,museum di Jakarta,8,Museum Tengah Kebun,Budaya,Jakarta,0,0.1218,
8,museum di Jakarta,9,Museum Basoeki Abdullah,Budaya,Jakarta,1000,0.1075,
9,museum di Jakarta,10,Museum Bahari Jakarta,Budaya,Jakarta,2000,0.1040,


In [39]:
def nilai_relevansi_awal(baris):
    query = baris["query"].casefold()
    nama = str(baris["Place_Name"]).casefold()
    kategori = str(baris["Category"]).casefold()
    kota = str(baris["City"]).casefold()
    harga = pd.to_numeric(baris["Price"], errors="coerce")

    if query == "museum di jakarta":
        return int(kota == "jakarta" and "museum" in nama)

    elif query == "pantai di yogyakarta":
        return int(kota == "yogyakarta" and "pantai" in nama)

    elif query == "wisata alam di semarang":
        return int(kota == "semarang" and kategori == "cagar alam")

    elif query == "taman hiburan di bandung":
        return int(kota == "bandung" and kategori == "taman hiburan")

    elif query == "wisata di yogyakarta":
        return int(kota == "yogyakarta")

    elif query == "budaya di bandung":
        return int(kota == "bandung" and kategori == "budaya")

    elif query == "tempat ibadah di surabaya":
        return int(kota == "surabaya" and kategori == "tempat ibadah")

    elif query == "wisata murah di semarang":
        return int(
            kota == "semarang"
            and pd.notna(harga)
            and harga <= 20000
        )

    return 0


kandidat_relevansi_df["relevant"] = (
    kandidat_relevansi_df.apply(nilai_relevansi_awal, axis=1)
)

print("Total kandidat:", len(kandidat_relevansi_df))
print("Label relevan:", int(kandidat_relevansi_df["relevant"].sum()))
print("Label tidak relevan:", int((kandidat_relevansi_df["relevant"] == 0).sum()))

display(
    kandidat_relevansi_df[
        ["query", "rank", "Place_Name", "City", "Price", "relevant"]
    ].head(20)
)

Total kandidat: 80
Label relevan: 54
Label tidak relevan: 26


,query,rank,Place_Name,City,Price,relevant
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,Jakarta,5000,1
1,museum di Jakarta,2,Museum Nasional,Jakarta,5000,1
2,museum di Jakarta,3,Museum Wayang,Jakarta,5000,1
3,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,Jakarta,100000,1
4,museum di Jakarta,5,Museum Benteng Vredeburg Yogyakarta,Yogyakarta,3000,0
5,museum di Jakarta,6,Museum Joang 45,Jakarta,2000,1
6,museum di Jakarta,7,Taman Impian Jaya Ancol,Jakarta,25000,0
7,museum di Jakarta,8,Museum Tengah Kebun,Jakarta,0,1
8,museum di Jakarta,9,Museum Basoeki Abdullah,Jakarta,1000,1
9,museum di Jakarta,10,Museum Bahari Jakarta,Jakarta,2000,1


In [40]:
file_uji_berlabel = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "kandidat_uji_relevansi_berlabel.csv"
)

kandidat_relevansi_df.to_csv(
    file_uji_berlabel,
    index=False,
    encoding="utf-8-sig"
)

print("File tersimpan:", file_uji_berlabel.resolve())

File tersimpan: D:\WisataFinder\data\processed\kandidat_uji_relevansi_berlabel.csv


In [41]:
# Ambil maksimal 5 hasil teratas untuk setiap kueri
top5 = (
    kandidat_relevansi_df
    .sort_values(["query", "rank"])
    .groupby("query", group_keys=False)
    .head(5)
    .copy()
)

# Hitung jumlah hasil relevan pada top 5
evaluasi_p5 = (
    top5.groupby("query")
    .agg(
        jumlah_hasil=("relevant", "count"),
        jumlah_relevan=("relevant", "sum")
    )
    .reset_index()
)

# Precision@5 menggunakan k = 5
evaluasi_p5["Precision@5"] = (
    evaluasi_p5["jumlah_relevan"] / 5
)

display(evaluasi_p5)

precision_macro = evaluasi_p5["Precision@5"].mean()

print(f"Macro-average Precision@5: {precision_macro:.4f}")
print(f"Persentase Precision@5: {precision_macro * 100:.2f}%")

,query,jumlah_hasil,jumlah_relevan,Precision@5
0,budaya di Bandung,5,1,0.2
1,museum di Jakarta,5,4,0.8
2,pantai di Yogyakarta,5,5,1.0
3,taman hiburan di Bandung,5,5,1.0
4,tempat ibadah di Surabaya,5,1,0.2
5,wisata alam di Semarang,5,2,0.4
6,wisata di Yogyakarta,5,4,0.8
7,wisata murah di Semarang,5,4,0.8


Macro-average Precision@5: 0.6500
Persentase Precision@5: 65.00%


In [42]:
# Periksa kolom dan hasil evaluasi
print("Kolom:", evaluasi_p5.columns.tolist())
display(evaluasi_p5)

# Hitung ulang Precision@5 dari jumlah relevan
evaluasi_p5["Precision@5"] = (
    evaluasi_p5["jumlah_relevan"] / 5
)

macro_p5 = evaluasi_p5["Precision@5"].mean()

print(f"\nMacro-average Precision@5: {macro_p5:.4f}")
print(f"Persentase: {macro_p5 * 100:.2f}%")

Kolom: ['query', 'jumlah_hasil', 'jumlah_relevan', 'Precision@5']


,query,jumlah_hasil,jumlah_relevan,Precision@5
0,budaya di Bandung,5,1,0.2
1,museum di Jakarta,5,4,0.8
2,pantai di Yogyakarta,5,5,1.0
3,taman hiburan di Bandung,5,5,1.0
4,tempat ibadah di Surabaya,5,1,0.2
5,wisata alam di Semarang,5,2,0.4
6,wisata di Yogyakarta,5,4,0.8
7,wisata murah di Semarang,5,4,0.8



Macro-average Precision@5: 0.6500
Persentase: 65.00%


In [43]:
kueri_prioritas = [
    "budaya di Bandung",
    "tempat ibadah di Surabaya",
    "wisata alam di Semarang"
]

audit_hasil = kandidat_relevansi_df[
    kandidat_relevansi_df["query"].isin(kueri_prioritas)
].sort_values(["query", "rank"])

display(
    audit_hasil[
        [
            "query",
            "rank",
            "Place_Name",
            "Category",
            "City",
            "Price",
            "similarity_score",
            "relevant"
        ]
    ]
)

,query,rank,Place_Name,Category,City,Price,similarity_score,relevant
50,budaya di Bandung,1,Bentara Budaya Yogyakarta (BBY),Budaya,Yogyakarta,0,0.1155,0
51,budaya di Bandung,2,Panama Park 825,Taman Hiburan,Bandung,50000,0.0960,0
52,budaya di Bandung,3,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,0.0829,0
53,budaya di Bandung,4,Museum Pendidikan Nasional,Budaya,Bandung,5000,0.0768,1
54,budaya di Bandung,5,NuArt Sculpture Park,Taman Hiburan,Bandung,50000,0.0744,0
55,budaya di Bandung,6,Tafso Barn,Cagar Alam,Bandung,0,0.0737,0
56,budaya di Bandung,7,Monumen Bandung Lautan Api,Budaya,Bandung,0,0.0726,1
57,budaya di Bandung,8,Taman Jomblo,Taman Hiburan,Bandung,10000,0.0687,0
58,budaya di Bandung,9,Taman Kupu-Kupu Cihanjuang,Cagar Alam,Bandung,10000,0.0682,0
59,budaya di Bandung,10,Amazing Art World,Budaya,Bandung,75000,0.0665,1


In [44]:
kueri_uji = [
    "museum di Jakarta",
    "pantai di Yogyakarta",
    "wisata alam di Semarang",
    "taman hiburan di Bandung",
    "wisata di Yogyakarta",
    "budaya di Bandung",
    "tempat ibadah di Surabaya",
    "wisata murah di Semarang"
]

hasil_filtered = []

for query in kueri_uji:
    hasil = cari_destinasi_v3(query, top_k=5)

    for rank, (_, baris) in enumerate(hasil.iterrows(), start=1):
        hasil_filtered.append({
            "query": query,
            "rank": rank,
            "Place_Name": baris["Place_Name"],
            "Category": baris["Category"],
            "City": baris["City"],
            "Price": baris["Price"],
            "similarity_score": baris["similarity_score"]
        })

hasil_filtered_df = pd.DataFrame(hasil_filtered)

# Terapkan aturan label awal yang sama
hasil_filtered_df["relevant"] = (
    hasil_filtered_df.apply(nilai_relevansi_awal, axis=1)
)

# Hitung Precision@5 dengan penyebut tetap 5
evaluasi_filtered = (
    hasil_filtered_df.groupby("query")
    .agg(
        jumlah_hasil=("relevant", "count"),
        jumlah_relevan=("relevant", "sum")
    )
    .reset_index()
)

evaluasi_filtered["Precision@5"] = (
    evaluasi_filtered["jumlah_relevan"] / 5
)

display(evaluasi_filtered)

macro_filtered = evaluasi_filtered["Precision@5"].mean()
print(f"Macro-average Precision@5: {macro_filtered:.4f}")
print(f"Persentase: {macro_filtered * 100:.2f}%")

Kota terdeteksi: Jakarta
Kategori terdeteksi: Budaya
Batas harga: Tidak ada
Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Bahari
Batas harga: Tidak ada
Kota terdeteksi: Semarang
Kategori terdeteksi: Cagar Alam
Batas harga: Tidak ada
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada
Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Tidak ada
Batas harga: Tidak ada
Kota terdeteksi: Bandung
Kategori terdeteksi: Budaya
Batas harga: Tidak ada
Kota terdeteksi: Surabaya
Kategori terdeteksi: Tempat Ibadah
Batas harga: Tidak ada
Kota terdeteksi: Semarang
Kategori terdeteksi: Tidak ada
Batas harga: Rp20,000


,query,jumlah_hasil,jumlah_relevan,Precision@5
0,budaya di Bandung,5,5,1.0
1,museum di Jakarta,5,5,1.0
2,pantai di Yogyakarta,5,5,1.0
3,taman hiburan di Bandung,5,5,1.0
4,tempat ibadah di Surabaya,3,3,0.6
5,wisata alam di Semarang,5,5,1.0
6,wisata di Yogyakarta,5,5,1.0
7,wisata murah di Semarang,5,5,1.0


Macro-average Precision@5: 0.9500
Persentase: 95.00%


In [45]:
# Siapkan hasil pencarian untuk pemeriksaan manual
audit_manual = hasil_filtered_df.copy()

audit_manual["relevant_manual"] = ""

file_audit = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "audit_relevansi_manual.csv"
)

audit_manual.to_csv(
    file_audit,
    index=False,
    encoding="utf-8-sig"
)

print("Jumlah hasil:", len(audit_manual))
print("File audit tersimpan:", file_audit.resolve())

display(
    audit_manual[
        [
            "query",
            "rank",
            "Place_Name",
            "Category",
            "City",
            "Price",
            "similarity_score",
            "relevant_manual"
        ]
    ]
)

Jumlah hasil: 38
File audit tersimpan: D:\WisataFinder\data\processed\audit_relevansi_manual.csv


,query,rank,Place_Name,Category,City,Price,similarity_score,relevant_manual
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,0.2207,
1,museum di Jakarta,2,Museum Nasional,Budaya,Jakarta,5000,0.1666,
2,museum di Jakarta,3,Museum Wayang,Budaya,Jakarta,5000,0.1436,
3,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,0.1393,
4,museum di Jakarta,5,Museum Joang 45,Budaya,Jakarta,2000,0.1366,
5,pantai di Yogyakarta,1,Pantai Glagah,Bahari,Yogyakarta,5000,0.2245,
6,pantai di Yogyakarta,2,Pantai Parangtritis,Bahari,Yogyakarta,10000,0.2034,
7,pantai di Yogyakarta,3,Pantai Timang,Bahari,Yogyakarta,10000,0.1776,
8,pantai di Yogyakarta,4,Pantai Sepanjang,Bahari,Yogyakarta,10000,0.1504,
9,pantai di Yogyakarta,5,Pantai Jungwok,Bahari,Yogyakarta,5000,0.1375,


In [46]:

# Baca hasil pencarian yang sudah difilter
file_audit = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "audit_relevansi_manual.csv"
)

audit_manual = pd.read_csv(file_audit)

def label_awal(baris):
    query = baris["query"].casefold()
    nama = str(baris["Place_Name"]).casefold()
    kategori = str(baris["Category"]).casefold()
    kota = str(baris["City"]).casefold()
    harga = pd.to_numeric(baris["Price"], errors="coerce")

    if query == "museum di jakarta":
        return int(kota == "jakarta" and "museum" in nama)

    if query == "pantai di yogyakarta":
        return int(kota == "yogyakarta" and "pantai" in nama)

    if query == "wisata alam di semarang":
        return int(
            kota == "semarang"
            and kategori == "cagar alam"
        )

    if query == "taman hiburan di bandung":
        return int(
            kota == "bandung"
            and kategori == "taman hiburan"
        )

    if query == "wisata di yogyakarta":
        return int(kota == "yogyakarta")

    if query == "budaya di bandung":
        return int(
            kota == "bandung"
            and kategori == "budaya"
        )

    if query == "tempat ibadah di surabaya":
        return int(
            kota == "surabaya"
            and kategori == "tempat ibadah"
        )

    if query == "wisata murah di semarang":
        return int(
            kota == "semarang"
            and pd.notna(harga)
            and harga <= 20000
        )

    return 0

audit_manual["relevant_manual"] = audit_manual.apply(
    label_awal, axis=1
)

print("Jumlah kandidat:", len(audit_manual))
print("Relevan:", int(audit_manual["relevant_manual"].sum()))
print("Tidak relevan:", int((audit_manual["relevant_manual"] == 0).sum()))

display(
    audit_manual[
        ["query", "rank", "Place_Name", "City",
         "Category", "Price", "relevant_manual"]
    ]
)


Jumlah kandidat: 38
Relevan: 38
Tidak relevan: 0


,query,rank,Place_Name,City,Category,Price,relevant_manual
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,Jakarta,Budaya,5000,1
1,museum di Jakarta,2,Museum Nasional,Jakarta,Budaya,5000,1
2,museum di Jakarta,3,Museum Wayang,Jakarta,Budaya,5000,1
3,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,Jakarta,Budaya,100000,1
4,museum di Jakarta,5,Museum Joang 45,Jakarta,Budaya,2000,1
5,pantai di Yogyakarta,1,Pantai Glagah,Yogyakarta,Bahari,5000,1
6,pantai di Yogyakarta,2,Pantai Parangtritis,Yogyakarta,Bahari,10000,1
7,pantai di Yogyakarta,3,Pantai Timang,Yogyakarta,Bahari,10000,1
8,pantai di Yogyakarta,4,Pantai Sepanjang,Yogyakarta,Bahari,10000,1
9,pantai di Yogyakarta,5,Pantai Jungwok,Yogyakarta,Bahari,5000,1


In [47]:

file_label = (
    Path.cwd().parent
    / "data"
    / "processed"
    / "audit_relevansi_semiotomatis.csv"
)

audit_manual.to_csv(
    file_label,
    index=False,
    encoding="utf-8-sig"
)

print("File tersimpan:", file_label.resolve())


File tersimpan: D:\WisataFinder\data\processed\audit_relevansi_semiotomatis.csv


In [48]:
# Ambil contoh hasil dari setiap kueri
sampel_audit = (
    audit_manual
    .groupby("query", group_keys=False)
    .head(2)
    .copy()
)

kolom_audit = [
    "query",
    "rank",
    "Place_Name",
    "City",
    "Category",
    "Price",
    "similarity_score",
    "relevant_manual"
]

display(sampel_audit[kolom_audit])

,query,rank,Place_Name,City,Category,Price,similarity_score,relevant_manual
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,Jakarta,Budaya,5000,0.2207,1
1,museum di Jakarta,2,Museum Nasional,Jakarta,Budaya,5000,0.1666,1
5,pantai di Yogyakarta,1,Pantai Glagah,Yogyakarta,Bahari,5000,0.2245,1
6,pantai di Yogyakarta,2,Pantai Parangtritis,Yogyakarta,Bahari,10000,0.2034,1
10,wisata alam di Semarang,1,Umbul Sidomukti,Semarang,Cagar Alam,75000,0.1521,1
11,wisata alam di Semarang,2,Brown Canyon,Semarang,Cagar Alam,0,0.1060,1
15,taman hiburan di Bandung,1,Panama Park 825,Bandung,Taman Hiburan,50000,0.1158,1
16,taman hiburan di Bandung,2,Water Park Bandung Indah,Bandung,Taman Hiburan,50000,0.1101,1
20,wisata di Yogyakarta,1,Pantai Glagah,Yogyakarta,Bahari,5000,0.1585,1
21,wisata di Yogyakarta,2,The World Landmarks - Merapi Park Yogyakarta,Yogyakarta,Taman Hiburan,22000,0.1557,1


In [49]:
sampel_audit["relevant_manual"] = ""
sampel_audit["alasan_penilaian"] = ""

display(
    sampel_audit[
        [
            "query",
            "rank",
            "Place_Name",
            "City",
            "Category",
            "Price",
            "relevant_manual",
            "alasan_penilaian",
        ]
    ]
)

,query,rank,Place_Name,City,Category,Price,relevant_manual,alasan_penilaian
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,Jakarta,Budaya,5000,,
1,museum di Jakarta,2,Museum Nasional,Jakarta,Budaya,5000,,
5,pantai di Yogyakarta,1,Pantai Glagah,Yogyakarta,Bahari,5000,,
6,pantai di Yogyakarta,2,Pantai Parangtritis,Yogyakarta,Bahari,10000,,
10,wisata alam di Semarang,1,Umbul Sidomukti,Semarang,Cagar Alam,75000,,
11,wisata alam di Semarang,2,Brown Canyon,Semarang,Cagar Alam,0,,
15,taman hiburan di Bandung,1,Panama Park 825,Bandung,Taman Hiburan,50000,,
16,taman hiburan di Bandung,2,Water Park Bandung Indah,Bandung,Taman Hiburan,50000,,
20,wisata di Yogyakarta,1,Pantai Glagah,Yogyakarta,Bahari,5000,,
21,wisata di Yogyakarta,2,The World Landmarks - Merapi Park Yogyakarta,Yogyakarta,Taman Hiburan,22000,,


In [50]:
label_awal_manual = {
    "Museum Seni Rupa dan Kramik": 1,
    "Museum Nasional": 1,
    "Pantai Glagah": 1,
    "Pantai Parangtritis": 1,
    "Umbul Sidomukti": 1,
    "Brown Canyon": 1,
    "Panama Park 825": 1,
    "Water Park Bandung Indah": 1,
    "The World Landmarks - Merapi Park Yogyakarta": 1,
    "Museum Pendidikan Nasional": 1,
    "Monumen Bandung Lautan Api": 1,
    "Gereja Perawan Maria Tak Berdosa Surabaya": 1,
    "Masjid Muhammad Cheng Hoo": 1,
    "Pura Giri Natha": 1,
}

sampel_audit["relevant_manual"] = (
    sampel_audit["Place_Name"]
    .map(label_awal_manual)
)

display(
    sampel_audit[
        ["query", "rank", "Place_Name", "relevant_manual"]
    ]
)

,query,rank,Place_Name,relevant_manual
0,museum di Jakarta,1,Museum Seni Rupa dan Kramik,1
1,museum di Jakarta,2,Museum Nasional,1
5,pantai di Yogyakarta,1,Pantai Glagah,1
6,pantai di Yogyakarta,2,Pantai Parangtritis,1
10,wisata alam di Semarang,1,Umbul Sidomukti,1
11,wisata alam di Semarang,2,Brown Canyon,1
15,taman hiburan di Bandung,1,Panama Park 825,1
16,taman hiburan di Bandung,2,Water Park Bandung Indah,1
20,wisata di Yogyakarta,1,Pantai Glagah,1
21,wisata di Yogyakarta,2,The World Landmarks - Merapi Park Yogyakarta,1


In [51]:
print("Jumlah sampel:", len(sampel_audit))
print(
    "Sudah dinilai:",
    sampel_audit["relevant_manual"].notna().sum()
)
print(
    "Belum dinilai:",
    sampel_audit["relevant_manual"].isna().sum()
)

Jumlah sampel: 16
Sudah dinilai: 16
Belum dinilai: 0


In [52]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "audit_sampel_awal.csv"
)

sampel_audit.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("File tersimpan di:", OUTPUT_PATH)

File tersimpan di: D:\WisataFinder\data\processed\audit_sampel_awal.csv


In [53]:
kueri_tambahan = [
    "museum di Jakarta",
    "tempat rekreasi keluarga di Bandung",
    "wisata alam murah di Semarang",
    "tempat wisata yang menarik",
    "pantai dengan pemandangan indah",
    "wisata budaya di Yogyakarta",
    "tempat ibadah di Surabaya",
]

for query in kueri_tambahan:
    print("\n" + "=" * 70)
    print("KUERI:", query)

    hasil = cari_destinasi_v3(query, top_k=5)

    if hasil.empty:
        print("Tidak ada hasil yang cocok.")
    else:
        display(
            hasil[
                [
                    "Place_Name",
                    "Category",
                    "City",
                    "Price",
                    "similarity_score",
                ]
            ]
        )


KUERI: museum di Jakarta
Kota terdeteksi: Jakarta
Kategori terdeteksi: Budaya
Batas harga: Tidak ada


,Place_Name,Category,City,Price,similarity_score
0,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,0.2207
1,Museum Nasional,Budaya,Jakarta,5000,0.1666
2,Museum Wayang,Budaya,Jakarta,5000,0.1436
3,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,0.1393
4,Museum Joang 45,Budaya,Jakarta,2000,0.1366



KUERI: tempat rekreasi keluarga di Bandung
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada


,Place_Name,Category,City,Price,similarity_score
0,Taman Jomblo,Taman Hiburan,Bandung,10000,0.1120
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,0.1032
2,Panama Park 825,Taman Hiburan,Bandung,50000,0.0899
3,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,75000,0.0731
4,Teras Cikapundung BBWS,Taman Hiburan,Bandung,0,0.0548



KUERI: wisata alam murah di Semarang
Kota terdeteksi: Semarang
Kategori terdeteksi: Cagar Alam
Batas harga: Rp20,000


...

,Place_Name,Category,City,Price,similarity_score
0,Pantai Glagah,Bahari,Yogyakarta,5000,0.0884
1,Pantai Drini,Bahari,Yogyakarta,10000,0.0667
2,Pantai Siung,Bahari,Yogyakarta,10000,0.0653
3,Pantai Cipta,Bahari,Semarang,5000,0.0653
4,Pantai Jogan,Bahari,Yogyakarta,5000,0.0579



KUERI: wisata budaya di Yogyakarta
Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Budaya
Batas harga: Tidak ada


,Place_Name,Category,City,Price,similarity_score
0,Bentara Budaya Yogyakarta (BBY),Budaya,Yogyakarta,0,0.1361
1,Monumen Sanapati,Budaya,Yogyakarta,15000,0.0569
2,Museum Sonobudoyo Unit I,Budaya,Yogyakarta,5000,0.0547
3,Taman Budaya Yogyakarta,Budaya,Yogyakarta,0,0.0479
4,Tebing Breksi,Budaya,Yogyakarta,20000,0.0427



KUERI: tempat ibadah di Surabaya
Kota terdeteksi: Surabaya
Kategori terdeteksi: Tempat Ibadah
Batas harga: Tidak ada


,Place_Name,Category,City,Price,similarity_score
0,Gereja Perawan Maria Tak Berdosa Surabaya,Tempat Ibadah,Surabaya,10000,0.1675
1,Masjid Muhammad Cheng Hoo,Tempat Ibadah,Surabaya,0,0.0998
2,Masjid Nasional Al-Akbar,Tempat Ibadah,Surabaya,0,0.0971


In [54]:
# Uji pencarian dengan kueri sinonim
query = "tempat rekreasi keluarga di Bandung"

print("=" * 70)
print("KUERI:", query)

hasil = cari_destinasi_v3(query, top_k=5)

if hasil.empty:
    print("Tidak ada hasil yang cocok.")
else:
    display(
        hasil[
            [
                "Place_Name",
                "Category",
                "City",
                "Price",
                "similarity_score",
            ]
        ]
    )

KUERI: tempat rekreasi keluarga di Bandung
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada


,Place_Name,Category,City,Price,similarity_score
0,Taman Jomblo,Taman Hiburan,Bandung,10000,0.1120
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,0.1032
2,Panama Park 825,Taman Hiburan,Bandung,50000,0.0899
3,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,75000,0.0731
4,Teras Cikapundung BBWS,Taman Hiburan,Bandung,0,0.0548


In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

query = "tempat rekreasi keluarga di Bandung"

# Filter kandidat berdasarkan kota dan kategori
kandidat = df[
    (df["City"].str.casefold() == "bandung") &
    (df["Category"].str.casefold() == "taman hiburan")
].copy()

# Metode 1: TF-IDF dari search_text yang sudah digunakan
vectorizer_search = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True
)

matrix_search = vectorizer_search.fit_transform(
    df["search_text"].fillna("").astype(str)
)

query_search = vectorizer_search.transform([query])
score_search = cosine_similarity(
    query_search, matrix_search
).flatten()

# Metode 2: TF-IDF dari Description saja
vectorizer_desc = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True
)

matrix_desc = vectorizer_desc.fit_transform(
    df["Description"].fillna("").astype(str)
)

query_desc = vectorizer_desc.transform([query])
score_desc = cosine_similarity(
    query_desc, matrix_desc
).flatten()

# Tambahkan kedua skor pada kandidat
kandidat["score_search_text"] = score_search[kandidat.index]
kandidat["score_description"] = score_desc[kandidat.index]

# Tampilkan hasil perbandingan
hasil_perbandingan = kandidat[
    [
        "Place_Name",
        "City",
        "Category",
        "score_search_text",
        "score_description"
    ]
].copy()

hasil_perbandingan = hasil_perbandingan.sort_values(
    "score_description", ascending=False
).head(10)

display(hasil_perbandingan)

,Place_Name,City,Category,score_search_text,score_description
275,Taman Jomblo,Bandung,Taman Hiburan,0.112024,0.113143
290,Water Park Bandung Indah,Bandung,Taman Hiburan,0.103224,0.108976
266,Panama Park 825,Bandung,Taman Hiburan,0.089883,0.097827
236,Panghegar Waterboom Bandung,Bandung,Taman Hiburan,0.073106,0.073217
253,Teras Cikapundung BBWS,Bandung,Taman Hiburan,0.054771,0.058779
247,NuArt Sculpture Park,Bandung,Taman Hiburan,0.049974,0.052100
276,Peta Park,Bandung,Taman Hiburan,0.046955,0.050449
213,Trans Studio Bandung,Bandung,Taman Hiburan,0.042260,0.045015
246,Kiara Artha Park,Bandung,Taman Hiburan,0.039841,0.043580
215,Farm House Susu Lembang,Bandung,Taman Hiburan,0.042234,0.041829


In [56]:
# Eksperimen perluasan kueri dengan sinonim
perluasan_kueri = {
    "rekreasi keluarga": (
        "rekreasi keluarga taman bermain wahana "
        "waterpark taman rekreasi"
    )
}

query_asli = "tempat rekreasi keluarga di Bandung"
query_diperluas = query_asli

for frasa, tambahan in perluasan_kueri.items():
    if frasa in query_asli.casefold():
        query_diperluas += " " + tambahan

print("Kueri asli     :", query_asli)
print("Kueri diperluas:", query_diperluas)

# Uji melalui fungsi pencarian yang sudah ada
print("\n--- HASIL KUERI ASLI ---")
display(cari_destinasi_v3(query_asli, top_k=5))

print("\n--- HASIL KUERI DIPERLUAS ---")
display(cari_destinasi_v3(query_diperluas, top_k=5))

Kueri asli     : tempat rekreasi keluarga di Bandung
Kueri diperluas: tempat rekreasi keluarga di Bandung rekreasi keluarga taman bermain wahana waterpark taman rekreasi

--- HASIL KUERI ASLI ---
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada


,Place_Name,Category,City,Price,Rating,similarity_score
0,Taman Jomblo,Taman Hiburan,Bandung,10000,4.1,0.1120
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.1032
2,Panama Park 825,Taman Hiburan,Bandung,50000,4.4,0.0899
3,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,75000,4.3,0.0731
4,Teras Cikapundung BBWS,Taman Hiburan,Bandung,0,4.3,0.0548



--- HASIL KUERI DIPERLUAS ---
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada


,Place_Name,Category,City,Price,Rating,similarity_score
0,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.1285
1,Tektona Waterpark,Taman Hiburan,Bandung,60000,3.8,0.1016
2,Panama Park 825,Taman Hiburan,Bandung,50000,4.4,0.0941
3,Taman Jomblo,Taman Hiburan,Bandung,10000,4.1,0.0926
4,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,75000,4.3,0.0861


In [57]:
query_asli = "tempat rekreasi keluarga di Bandung"

query_diperluas = (
    "tempat rekreasi keluarga di Bandung "
    "taman bermain wahana waterpark taman rekreasi"
)

hasil_asli = cari_destinasi_v3(query_asli, top_k=5)
hasil_perluasan = cari_destinasi_v3(query_diperluas, top_k=5)

print("HASIL KUERI ASLI")
display(
    hasil_asli[
        ["Place_Name", "City", "Category", "similarity_score"]
    ]
)

print("HASIL KUERI DIPERLUAS")
display(
    hasil_perluasan[
        ["Place_Name", "City", "Category", "similarity_score"]
    ]
)

Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada
HASIL KUERI ASLI


,Place_Name,City,Category,similarity_score
0,Taman Jomblo,Bandung,Taman Hiburan,0.1120
1,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1032
2,Panama Park 825,Bandung,Taman Hiburan,0.0899
3,Panghegar Waterboom Bandung,Bandung,Taman Hiburan,0.0731
4,Teras Cikapundung BBWS,Bandung,Taman Hiburan,0.0548


HASIL KUERI DIPERLUAS


,Place_Name,City,Category,similarity_score
0,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1328
1,Taman Jomblo,Bandung,Taman Hiburan,0.1219
2,Tektona Waterpark,Bandung,Taman Hiburan,0.1125
3,Panama Park 825,Bandung,Taman Hiburan,0.1045
4,Trans Studio Bandung,Bandung,Taman Hiburan,0.0999


In [58]:
import pandas as pd
from pathlib import Path

def siapkan_audit(hasil, versi):
    audit = hasil.reset_index(drop=True).copy()
    audit["rank"] = range(1, len(audit) + 1)
    audit["versi_kueri"] = versi
    audit["relevant_manual"] = pd.NA

    return audit[
        [
            "versi_kueri",
            "rank",
            "Place_Name",
            "City",
            "Category",
            "similarity_score",
            "relevant_manual",
        ]
    ]

audit_perbandingan = pd.concat(
    [
        siapkan_audit(hasil_asli, "Asli"),
        siapkan_audit(hasil_perluasan, "Diperluas"),
    ],
    ignore_index=True,
)

display(audit_perbandingan)

,versi_kueri,rank,Place_Name,City,Category,similarity_score,relevant_manual
0,Asli,1,Taman Jomblo,Bandung,Taman Hiburan,0.1120,NaN
1,Asli,2,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1032,NaN
2,Asli,3,Panama Park 825,Bandung,Taman Hiburan,0.0899,NaN
3,Asli,4,Panghegar Waterboom Bandung,Bandung,Taman Hiburan,0.0731,NaN
4,Asli,5,Teras Cikapundung BBWS,Bandung,Taman Hiburan,0.0548,NaN
5,Diperluas,1,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1328,NaN
6,Diperluas,2,Taman Jomblo,Bandung,Taman Hiburan,0.1219,NaN
7,Diperluas,3,Tektona Waterpark,Bandung,Taman Hiburan,0.1125,NaN
8,Diperluas,4,Panama Park 825,Bandung,Taman Hiburan,0.1045,NaN
9,Diperluas,5,Trans Studio Bandung,Bandung,Taman Hiburan,0.0999,NaN


In [59]:
kolom_info = [
    "Place_Name",
    "Description",
    "City",
    "Category",
]

nama_destinasi_audit = audit_perbandingan["Place_Name"].unique()

display(
    df.loc[
        df["Place_Name"].isin(nama_destinasi_audit),
        kolom_info
    ].drop_duplicates("Place_Name").reset_index(drop=True)
)

,Place_Name,Description,City,Category
0,Trans Studio Bandung,Trans Studio Bandung adalah kawasan wisata ter...,Bandung,Taman Hiburan
1,Panghegar Waterboom Bandung,Panghegar Waterboom Bandung merupakan salah sa...,Bandung,Taman Hiburan
2,Teras Cikapundung BBWS,Teras Cikapundung Bandung sebelumnya merupakan...,Bandung,Taman Hiburan
3,Panama Park 825,Panama Park Bandung menjadi salah satu tempat ...,Bandung,Taman Hiburan
4,Taman Jomblo,Taman Jomblo dapat mengacu kepada:\n\nTaman Pa...,Bandung,Taman Hiburan
5,Tektona Waterpark,Tektona Waterpark sebuah wahana yang memberika...,Bandung,Taman Hiburan
6,Water Park Bandung Indah,Bandung Indah Waterpark merupakan salah satu k...,Bandung,Taman Hiburan


In [60]:
label_relevansi = {
    "Taman Jomblo": None,
    "Water Park Bandung Indah": None,
    "Panama Park 825": None,
    "Panghegar Waterboom Bandung": None,
    "Teras Cikapundung BBWS": None,
    "Tektona Waterpark": None,
    "Trans Studio Bandung": None,
}

audit_perbandingan["relevant_manual"] = (
    audit_perbandingan["Place_Name"]
    .map(label_relevansi)
)

display(audit_perbandingan)

,versi_kueri,rank,Place_Name,City,Category,similarity_score,relevant_manual
0,Asli,1,Taman Jomblo,Bandung,Taman Hiburan,0.1120,None
1,Asli,2,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1032,None
2,Asli,3,Panama Park 825,Bandung,Taman Hiburan,0.0899,None
3,Asli,4,Panghegar Waterboom Bandung,Bandung,Taman Hiburan,0.0731,None
4,Asli,5,Teras Cikapundung BBWS,Bandung,Taman Hiburan,0.0548,None
5,Diperluas,1,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1328,None
6,Diperluas,2,Taman Jomblo,Bandung,Taman Hiburan,0.1219,None
7,Diperluas,3,Tektona Waterpark,Bandung,Taman Hiburan,0.1125,None
8,Diperluas,4,Panama Park 825,Bandung,Taman Hiburan,0.1045,None
9,Diperluas,5,Trans Studio Bandung,Bandung,Taman Hiburan,0.0999,None


In [61]:
if audit_perbandingan["relevant_manual"].isna().any():
    print("Masih ada destinasi yang belum dinilai.")
else:
    evaluasi_perbandingan = (
        audit_perbandingan
        .groupby("versi_kueri")
        .agg(
            jumlah_hasil=("Place_Name", "count"),
            jumlah_relevan=("relevant_manual", "sum"),
        )
        .reset_index()
    )

    evaluasi_perbandingan["Precision@5"] = (
        evaluasi_perbandingan["jumlah_relevan"] / 5
    )

    display(evaluasi_perbandingan)

Masih ada destinasi yang belum dinilai.


In [62]:
nama_destinasi = [
    "Taman Jomblo",
    "Water Park Bandung Indah",
    "Panama Park 825",
    "Panghegar Waterboom Bandung",
    "Teras Cikapundung BBWS",
    "Tektona Waterpark",
    "Trans Studio Bandung",
]

kolom = ["Place_Name", "Description", "Category", "City"]

display(
    df.loc[
        df["Place_Name"].isin(nama_destinasi),
        kolom
    ].drop_duplicates("Place_Name").reset_index(drop=True)
)

,Place_Name,Description,Category,City
0,Trans Studio Bandung,Trans Studio Bandung adalah kawasan wisata ter...,Taman Hiburan,Bandung
1,Panghegar Waterboom Bandung,Panghegar Waterboom Bandung merupakan salah sa...,Taman Hiburan,Bandung
2,Teras Cikapundung BBWS,Teras Cikapundung Bandung sebelumnya merupakan...,Taman Hiburan,Bandung
3,Panama Park 825,Panama Park Bandung menjadi salah satu tempat ...,Taman Hiburan,Bandung
4,Taman Jomblo,Taman Jomblo dapat mengacu kepada:\n\nTaman Pa...,Taman Hiburan,Bandung
5,Tektona Waterpark,Tektona Waterpark sebuah wahana yang memberika...,Taman Hiburan,Bandung
6,Water Park Bandung Indah,Bandung Indah Waterpark merupakan salah satu k...,Taman Hiburan,Bandung


In [63]:
label_relevansi = {
    "Taman Jomblo": 0,  # Sementara, verifikasi deskripsi lengkap
    "Water Park Bandung Indah": 1,
    "Panama Park 825": 1,
    "Panghegar Waterboom Bandung": 1,
    "Teras Cikapundung BBWS": 1,
    "Tektona Waterpark": 1,
    "Trans Studio Bandung": 1,
}

audit_perbandingan["relevant_manual"] = (
    audit_perbandingan["Place_Name"].map(label_relevansi)
)

display(audit_perbandingan)

,versi_kueri,rank,Place_Name,City,Category,similarity_score,relevant_manual
0,Asli,1,Taman Jomblo,Bandung,Taman Hiburan,0.1120,0
1,Asli,2,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1032,1
2,Asli,3,Panama Park 825,Bandung,Taman Hiburan,0.0899,1
3,Asli,4,Panghegar Waterboom Bandung,Bandung,Taman Hiburan,0.0731,1
4,Asli,5,Teras Cikapundung BBWS,Bandung,Taman Hiburan,0.0548,1
5,Diperluas,1,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1328,1
6,Diperluas,2,Taman Jomblo,Bandung,Taman Hiburan,0.1219,0
7,Diperluas,3,Tektona Waterpark,Bandung,Taman Hiburan,0.1125,1
8,Diperluas,4,Panama Park 825,Bandung,Taman Hiburan,0.1045,1
9,Diperluas,5,Trans Studio Bandung,Bandung,Taman Hiburan,0.0999,1


In [64]:
evaluasi_perbandingan = (
    audit_perbandingan
    .groupby("versi_kueri")
    .agg(
        jumlah_hasil=("Place_Name", "count"),
        jumlah_relevan=("relevant_manual", "sum"),
    )
    .reset_index()
)

evaluasi_perbandingan["Precision@5"] = (
    evaluasi_perbandingan["jumlah_relevan"] / 5
)

display(evaluasi_perbandingan)

,versi_kueri,jumlah_hasil,jumlah_relevan,Precision@5
0,Asli,5,4,0.8
1,Diperluas,5,4,0.8


In [65]:
# Kueri tambahan untuk menguji variasi kebutuhan pengguna
kueri_tambahan = [
    "tempat rekreasi keluarga di Bandung",
    "rekreasi keluarga di Bandung",
    "taman bermain di Bandung",
    "waterpark di Bandung",
]

for query in kueri_tambahan:
    print(f"\nKUERI: {query}")
    hasil = cari_destinasi_v3(query, top_k=5)

    display(
        hasil[
            [
                "Place_Name",
                "City",
                "Category",
                "similarity_score",
            ]
        ]
    )


KUERI: tempat rekreasi keluarga di Bandung
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada


,Place_Name,City,Category,similarity_score
0,Taman Jomblo,Bandung,Taman Hiburan,0.1120
1,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1032
2,Panama Park 825,Bandung,Taman Hiburan,0.0899
3,Panghegar Waterboom Bandung,Bandung,Taman Hiburan,0.0731
4,Teras Cikapundung BBWS,Bandung,Taman Hiburan,0.0548



KUERI: rekreasi keluarga di Bandung
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada


,Place_Name,City,Category,similarity_score
0,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1140
1,Panama Park 825,Bandung,Taman Hiburan,0.0930
2,Panghegar Waterboom Bandung,Bandung,Taman Hiburan,0.0766
3,Taman Jomblo,Bandung,Taman Hiburan,0.0749
4,NuArt Sculpture Park,Bandung,Taman Hiburan,0.0552



KUERI: taman bermain di Bandung
Kota terdeteksi: Bandung
Kategori terdeteksi: Tidak ada
Batas harga: Tidak ada


,Place_Name,City,Category,similarity_score
0,Panama Park 825,Bandung,Taman Hiburan,0.1642
1,Trans Studio Bandung,Bandung,Taman Hiburan,0.1289
2,Happyfarm Ciwidey,Bandung,Cagar Alam,0.1138
3,Taman Lalu Lintas Ade Irma Suryani Nasution,Bandung,Taman Hiburan,0.0823
4,Taman Jomblo,Bandung,Taman Hiburan,0.0753



KUERI: waterpark di Bandung
Kota terdeteksi: Bandung
Kategori terdeteksi: Tidak ada
Batas harga: Tidak ada


,Place_Name,City,Category,similarity_score
0,Water Park Bandung Indah,Bandung,Taman Hiburan,0.1772
1,Pesona Nirwana Waterpark & Cottages,Bandung,Taman Hiburan,0.1306
2,Tektona Waterpark,Bandung,Taman Hiburan,0.1267
3,Panama Park 825,Bandung,Taman Hiburan,0.1151
4,NuArt Sculpture Park,Bandung,Taman Hiburan,0.0892


In [66]:
nama_audit = [
    "Taman Jomblo",
    "NuArt Sculpture Park",
    "Happyfarm Ciwidey",
    "Teras Cikapundung BBWS",
]

kolom_audit = [
    "Place_Name",
    "Description",
    "Category",
    "City",
]

display(
    df.loc[
        df["Place_Name"].isin(nama_audit),
        kolom_audit
    ].drop_duplicates("Place_Name").reset_index(drop=True)
)

,Place_Name,Description,Category,City
0,NuArt Sculpture Park,NuArt Sculpture Park ini merupakan sebuah gall...,Taman Hiburan,Bandung
1,Teras Cikapundung BBWS,Teras Cikapundung Bandung sebelumnya merupakan...,Taman Hiburan,Bandung
2,Taman Jomblo,Taman Jomblo dapat mengacu kepada:\n\nTaman Pa...,Taman Hiburan,Bandung
3,Happyfarm Ciwidey,Objek wisata alam dan edukasi tengah banyak me...,Cagar Alam,Bandung


In [67]:
audit_metadata = pd.DataFrame([
    {
        "Place_Name": "NuArt Sculpture Park",
        "kategori_sekarang": "Taman Hiburan",
        "kategori_usulan": "Perlu verifikasi",
        "catatan": "Deskripsi menyebut galeri seni",
    },
    {
        "Place_Name": "Taman Jomblo",
        "kategori_sekarang": "Taman Hiburan",
        "kategori_usulan": "Perlu verifikasi",
        "catatan": "Deskripsi ambigu dan perlu diperiksa",
    },
    {
        "Place_Name": "Happyfarm Ciwidey",
        "kategori_sekarang": "Cagar Alam",
        "kategori_usulan": "Perlu verifikasi",
        "catatan": "Wisata alam dan edukasi",
    },
])

display(audit_metadata)

,Place_Name,kategori_sekarang,kategori_usulan,catatan
0,NuArt Sculpture Park,Taman Hiburan,Perlu verifikasi,Deskripsi menyebut galeri seni
1,Taman Jomblo,Taman Hiburan,Perlu verifikasi,Deskripsi ambigu dan perlu diperiksa
2,Happyfarm Ciwidey,Cagar Alam,Perlu verifikasi,Wisata alam dan edukasi


In [68]:
from IPython.display import display, Markdown

nama_audit = [
    "NuArt Sculpture Park",
    "Taman Jomblo",
    "Happyfarm Ciwidey",
]

for nama in nama_audit:
    baris = df.loc[df["Place_Name"] == nama]

    if not baris.empty:
        deskripsi = str(baris.iloc[0]["Description"])
        kategori = baris.iloc[0]["Category"]

        display(Markdown(f"### {nama}"))
        display(Markdown(f"**Kategori saat ini:** {kategori}"))
        display(Markdown(f"**Deskripsi lengkap:** {deskripsi}"))

### NuArt Sculpture Park

**Kategori saat ini:** Taman Hiburan

**Deskripsi lengkap:** NuArt Sculpture Park ini merupakan sebuah gallery miliki seorang seniman ternama di Bandung yang bernama Nyoman Nuarta. Terlahir di Tabanan Bali, Nyoman Nuarta akhirnya menetap di Bandung. Dan ada fakta menarik dari seniman berkelas internasional tersebut, dimana ia dahulu rupanya lulusan dari Fakultas Seni Rupa dari perguruan tinggi ternama di Bandung yakni ITB pada tahun 1979 silam. Kehlian Nyoman Nuarta sebagai seorang maestro pemahat dan juga pematung hebat memang sudah dikenal secara internasional, maka dari itu ia pun membuat NuArt Sculpture Park ini untuk memamerkan semua karya seni yang ia miliki selama ini.

### Taman Jomblo

**Kategori saat ini:** Taman Hiburan

**Deskripsi lengkap:** Taman Jomblo dapat mengacu kepada:\n\nTaman Pasupati, sebuah taman yang didirikan oleh Ridwan Kamil di Bandung\nPedestrian Jomblo, sebuah tempat rekreasi yang didirikan di Jambi\nTaman Jomblo (Mataram), sebuah taman yang didirikan di Mataram, Nusa Tenggara Barat

### Happyfarm Ciwidey

**Kategori saat ini:** Cagar Alam

**Deskripsi lengkap:** Objek wisata alam dan edukasi tengah banyak menjamur akhir-akhir ini. Selain De Ranch dan Farmhouse Lembang, ada juga Happy Farm Ciwidey di Bandung. Objek wisata ini masih terbilang baru, karena baru bulan Juni 2019 ini berdiri. Konsepnya hampir sama, wisata alam dan peternakan dengan unsur edukasi. Sangat cocok menjadi tujuan wisata keluarga, terutama untuk anak-anak. Dan semakin seru karena bermain di area outdoor dengan udara sejuk.

In [69]:
# Salinan khusus eksperimen, tidak mengubah df asli
df_perbaikan = df.copy()

# 1. Bersihkan deskripsi Taman Jomblo berdasarkan informasi
# yang tersedia dalam dataset
mask_jomblo = df_perbaikan["Place_Name"].eq("Taman Jomblo")

df_perbaikan.loc[mask_jomblo, "Description"] = (
    "Taman Jomblo atau Taman Pasupati adalah taman publik "
    "di Bandung."
)

# 2. Koreksi kategori NuArt untuk eksperimen saja.
# "Budaya" tersedia sebagai kategori dalam dataset.
mask_nuart = df_perbaikan["Place_Name"].eq("NuArt Sculpture Park")

df_perbaikan.loc[mask_nuart, "Category"] = "Budaya"

# 3. Happyfarm tidak diubah karena deskripsinya sudah informatif.
display(
    df_perbaikan.loc[
        df_perbaikan["Place_Name"].isin([
            "Taman Jomblo",
            "NuArt Sculpture Park",
            "Happyfarm Ciwidey",
        ]),
        ["Place_Name", "Description", "Category", "City"],
    ]
)

,Place_Name,Description,Category,City
247,NuArt Sculpture Park,NuArt Sculpture Park ini merupakan sebuah gall...,Budaya,Bandung
275,Taman Jomblo,Taman Jomblo atau Taman Pasupati adalah taman ...,Taman Hiburan,Bandung
323,Happyfarm Ciwidey,Objek wisata alam dan edukasi tengah banyak me...,Cagar Alam,Bandung


In [70]:
print("Kolom dataset:", df_perbaikan.columns.tolist())
print("\nContoh search_text sebelum perbaikan:")
display(df[["Place_Name", "search_text"]].head(3))

Kolom dataset: ['Place_Id', 'Place_Name', 'Description', 'Category', 'City', 'Price', 'Rating', 'Time_Minutes', 'Coordinate', 'Lat', 'Long', 'search_text']

Contoh search_text sebelum perbaikan:


,Place_Name,search_text
0,Monumen Nasional,Monumen Nasional Budaya Jakarta Monumen Nasion...
1,Kota Tua,"Kota Tua Budaya Jakarta Kota tua di Jakarta, y..."
2,Dunia Fantasi,Dunia Fantasi Taman Hiburan Jakarta Dunia Fant...


In [71]:
# Buat ulang teks pencarian dari kolom yang sudah diperbaiki
kolom_teks = ["Place_Name", "Category", "City", "Description"]

df_perbaikan["search_text"] = (
    df_perbaikan[kolom_teks]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Periksa tiga destinasi yang sedang kita audit
nama_audit = [
    "Taman Jomblo",
    "NuArt Sculpture Park",
    "Happyfarm Ciwidey",
]

display(
    df_perbaikan.loc[
        df_perbaikan["Place_Name"].isin(nama_audit),
        ["Place_Name", "Category", "City", "search_text"],
    ].reset_index(drop=True)
)

,Place_Name,Category,City,search_text
0,NuArt Sculpture Park,Budaya,Bandung,NuArt Sculpture Park Budaya Bandung NuArt Scul...
1,Taman Jomblo,Taman Hiburan,Bandung,Taman Jomblo Taman Hiburan Bandung Taman Jombl...
2,Happyfarm Ciwidey,Cagar Alam,Bandung,Happyfarm Ciwidey Cagar Alam Bandung Objek wis...


In [72]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_perbaikan = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

tfidf_matrix_perbaikan = vectorizer_perbaikan.fit_transform(
    df_perbaikan["search_text"].fillna("")
)

print("Jumlah destinasi:", tfidf_matrix_perbaikan.shape[0])
print("Jumlah fitur TF-IDF:", tfidf_matrix_perbaikan.shape[1])

Jumlah destinasi: 437
Jumlah fitur TF-IDF: 35042


In [73]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def cari_destinasi_eksperimen(
    query,
    top_k=5,
    kota=None,
    kategori=None,
    harga_maks=None,
):
    if not query or not query.strip():
        return pd.DataFrame()

    # Ubah kueri menjadi vektor TF-IDF
    query_vector = vectorizer_perbaikan.transform([query])

    # Hitung cosine similarity terhadap seluruh destinasi
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix_perbaikan
    ).flatten()

    hasil = df_perbaikan.copy()
    hasil["similarity_score"] = scores

    # Terapkan filter jika diberikan
    if kota:
        hasil = hasil[hasil["City"].str.casefold() == kota.casefold()]

    if kategori:
        hasil = hasil[
            hasil["Category"].str.casefold() == kategori.casefold()
        ]

    if harga_maks is not None:
        hasil = hasil[hasil["Price"] <= harga_maks]

    # Hapus hasil yang tidak memiliki kemiripan dengan kueri
    hasil = hasil[hasil["similarity_score"] > 0]

    hasil = hasil.sort_values(
        "similarity_score",
        ascending=False
    ).head(top_k)

    kolom_hasil = [
        "Place_Name",
        "Category",
        "City",
        "Price",
        "Rating",
        "similarity_score",
    ]

    return hasil[kolom_hasil].reset_index(drop=True)

In [74]:
query_uji = "tempat rekreasi keluarga di Bandung"

print("HASIL LAMA")
display(
    cari_destinasi_v3(query_uji, top_k=5)[
        ["Place_Name", "Category", "City", "similarity_score"]
    ]
)

print("HASIL SETELAH PERBAIKAN DATA")
display(
    cari_destinasi_eksperimen(
        query_uji,
        top_k=5,
        kota="Bandung",
        kategori="Taman Hiburan",
    )[
        ["Place_Name", "Category", "City", "similarity_score"]
    ]
)

HASIL LAMA
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada


,Place_Name,Category,City,similarity_score
0,Taman Jomblo,Taman Hiburan,Bandung,0.1120
1,Water Park Bandung Indah,Taman Hiburan,Bandung,0.1032
2,Panama Park 825,Taman Hiburan,Bandung,0.0899
3,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,0.0731
4,Teras Cikapundung BBWS,Taman Hiburan,Bandung,0.0548


HASIL SETELAH PERBAIKAN DATA


,Place_Name,Category,City,similarity_score
0,Water Park Bandung Indah,Taman Hiburan,Bandung,0.103205
1,Panama Park 825,Taman Hiburan,Bandung,0.089870
2,Taman Jomblo,Taman Hiburan,Bandung,0.080213
3,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,0.073060
4,Teras Cikapundung BBWS,Taman Hiburan,Bandung,0.054815


In [75]:
kueri_perbandingan = [
    {
        "query": "tempat rekreasi keluarga di Bandung",
        "kota": "Bandung",
        "kategori": "Taman Hiburan",
    },
    {
        "query": "rekreasi keluarga di Bandung",
        "kota": "Bandung",
        "kategori": "Taman Hiburan",
    },
    {
        "query": "taman bermain di Bandung",
        "kota": "Bandung",
        "kategori": None,
    },
    {
        "query": "waterpark di Bandung",
        "kota": "Bandung",
        "kategori": None,
    },
]

for kasus in kueri_perbandingan:
    query = kasus["query"]
    kota = kasus["kota"]
    kategori = kasus["kategori"]

    print(f"\n{'=' * 60}")
    print("KUERI:", query)
    print("Filter kota:", kota)
    print("Filter kategori:", kategori)

    print("\nHASIL LAMA")
    hasil_lama = cari_destinasi_v2(
        query,
        kota=kota,
        kategori=kategori,
        top_k=5,
    )
    display(
        hasil_lama[
            ["Place_Name", "Category", "City", "similarity_score"]
        ]
    )

    print("\nHASIL EKSPERIMEN")
    hasil_baru = cari_destinasi_eksperimen(
        query,
        kota=kota,
        kategori=kategori,
        top_k=5,
    )
    display(
        hasil_baru[
            ["Place_Name", "Category", "City", "similarity_score"]
        ]
    )


KUERI: tempat rekreasi keluarga di Bandung
Filter kota: Bandung
Filter kategori: Taman Hiburan

HASIL LAMA


,Place_Name,Category,City,similarity_score
0,Taman Jomblo,Taman Hiburan,Bandung,0.1120
1,Water Park Bandung Indah,Taman Hiburan,Bandung,0.1032
2,Panama Park 825,Taman Hiburan,Bandung,0.0899
3,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,0.0731
4,Teras Cikapundung BBWS,Taman Hiburan,Bandung,0.0548



HASIL EKSPERIMEN


,Place_Name,Category,City,similarity_score
0,Water Park Bandung Indah,Taman Hiburan,Bandung,0.103205
1,Panama Park 825,Taman Hiburan,Bandung,0.089870
2,Taman Jomblo,Taman Hiburan,Bandung,0.080213
3,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,0.073060
4,Teras Cikapundung BBWS,Taman Hiburan,Bandung,0.054815



KUERI: rekreasi keluarga di Bandung
Filter kota: Bandung
Filter kategori: Taman Hiburan

HASIL LAMA


...

,Place_Name,Category,City,similarity_score
0,Panama Park 825,Taman Hiburan,Bandung,0.164163
1,Taman Jomblo,Taman Hiburan,Bandung,0.133295
2,Trans Studio Bandung,Taman Hiburan,Bandung,0.128907
3,Happyfarm Ciwidey,Cagar Alam,Bandung,0.113764
4,Taman Lalu Lintas Ade Irma Suryani Nasution,Taman Hiburan,Bandung,0.082386



KUERI: waterpark di Bandung
Filter kota: Bandung
Filter kategori: None

HASIL LAMA


,Place_Name,Category,City,similarity_score
0,Water Park Bandung Indah,Taman Hiburan,Bandung,0.1772
1,Pesona Nirwana Waterpark & Cottages,Taman Hiburan,Bandung,0.1306
2,Tektona Waterpark,Taman Hiburan,Bandung,0.1267
3,Panama Park 825,Taman Hiburan,Bandung,0.1151
4,NuArt Sculpture Park,Taman Hiburan,Bandung,0.0892



HASIL EKSPERIMEN


,Place_Name,Category,City,similarity_score
0,Water Park Bandung Indah,Taman Hiburan,Bandung,0.177133
1,Taman Jomblo,Taman Hiburan,Bandung,0.143631
2,Pesona Nirwana Waterpark & Cottages,Taman Hiburan,Bandung,0.130626
3,Tektona Waterpark,Taman Hiburan,Bandung,0.126665
4,Panama Park 825,Taman Hiburan,Bandung,0.115056


In [76]:
# Eksperimen: kota tetap menjadi filter,
# tetapi tidak dimasukkan ke teks TF-IDF.
kolom_teks_v2 = ["Place_Name", "Category", "Description"]

df_perbaikan["search_text_v2"] = (
    df_perbaikan[kolom_teks_v2]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

vectorizer_v2 = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
)

tfidf_matrix_v2 = vectorizer_v2.fit_transform(
    df_perbaikan["search_text_v2"]
)

print("Jumlah destinasi:", tfidf_matrix_v2.shape[0])
print("Jumlah fitur:", tfidf_matrix_v2.shape[1])

Jumlah destinasi: 437
Jumlah fitur: 35042


In [77]:
def cari_destinasi_eksperimen_v2(
    query,
    top_k=5,
    kota=None,
    kategori=None,
    harga_maks=None,
):
    if not query or not query.strip():
        return pd.DataFrame()

    query_vector = vectorizer_v2.transform([query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix_v2
    ).flatten()

    hasil = df_perbaikan.copy()
    hasil["similarity_score"] = scores

    if kota:
        hasil = hasil[
            hasil["City"].str.casefold() == kota.casefold()
        ]

    if kategori:
        hasil = hasil[
            hasil["Category"].str.casefold() == kategori.casefold()
        ]

    if harga_maks is not None:
        hasil = hasil[hasil["Price"] <= harga_maks]

    hasil = hasil[hasil["similarity_score"] > 0]

    return (
        hasil.sort_values("similarity_score", ascending=False)
        .head(top_k)[
            [
                "Place_Name", "Category", "City",
                "Price", "Rating", "similarity_score"
            ]
        ]
        .reset_index(drop=True)
    )

In [78]:
for query in [
    "tempat rekreasi keluarga di Bandung",
    "waterpark di Bandung",
]:
    print("\nKUERI:", query)

    display(
        cari_destinasi_eksperimen_v2(
            query,
            kota="Bandung",
            top_k=5,
        )
    )


KUERI: tempat rekreasi keluarga di Bandung


,Place_Name,Category,City,Price,Rating,similarity_score
0,Tafso Barn,Cagar Alam,Bandung,0,4.2,0.113467
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.104136
2,Taman Kupu-Kupu Cihanjuang,Cagar Alam,Bandung,10000,4.0,0.102252
3,Panama Park 825,Taman Hiburan,Bandung,50000,4.4,0.090659
4,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,75000,4.3,0.073424



KUERI: waterpark di Bandung


,Place_Name,Category,City,Price,Rating,similarity_score
0,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.177480
1,Pesona Nirwana Waterpark & Cottages,Taman Hiburan,Bandung,50000,4.1,0.130353
2,Taman Jomblo,Taman Hiburan,Bandung,10000,4.1,0.129501
3,Panama Park 825,Taman Hiburan,Bandung,50000,4.4,0.115643
4,Tektona Waterpark,Taman Hiburan,Bandung,60000,3.8,0.115599


In [79]:
import re

STOPWORDS_KUERI = {
    "di", "ke", "dari", "yang", "dan", "untuk",
    "dengan", "ada", "pada", "atau", "tempat"
}

def bersihkan_kueri( query, kota=None):
    teks = query.lower().strip()

    # Hapus nama kota dari kueri karena kota akan difilter terpisah.
    if kota:
        teks = re.sub(
            rf"\b{re.escape(kota.lower())}\b",
            " ",
            teks
        )

    # Hilangkan kata umum yang kurang membantu pencarian.
    tokens = re.findall(r"\b\w+\b", teks)
    tokens = [
        token for token in tokens
        if token not in STOPWORDS_KUERI
    ]

    return " ".join(tokens)

In [80]:
def cari_destinasi_eksperimen_v3(
    query,
    top_k=5,
    kota=None,
    kategori=None,
    harga_maks=None,
):
    if not query or not query.strip():
        return pd.DataFrame()

    query_bersih = bersihkan_kueri(query, kota=kota)
    if not query_bersih:
        return pd.DataFrame()

    query_vector = vectorizer_v2.transform([query_bersih])
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix_v2
    ).flatten()

    hasil = df_perbaikan.copy()
    hasil["similarity_score"] = scores

    if kota:
        hasil = hasil[
            hasil["City"].str.casefold() == kota.casefold()
        ]

    if kategori:
        hasil = hasil[
            hasil["Category"].str.casefold() == kategori.casefold()
        ]

    if harga_maks is not None:
        hasil = hasil[hasil["Price"] <= harga_maks]

    hasil = hasil[hasil["similarity_score"] > 0]

    return (
        hasil.sort_values("similarity_score", ascending=False)
        .head(top_k)[
            [
                "Place_Name", "Category", "City",
                "Price", "Rating", "similarity_score"
            ]
        ]
        .reset_index(drop=True)
    )

In [81]:
for query in [
    "tempat rekreasi keluarga di Bandung",
    "waterpark di Bandung",
]:
    print("\nKUERI ASLI:", query)
    print(
        "KUERI BERSIH:",
        bersihkan_kueri(query, kota="Bandung")
    )

    display(
        cari_destinasi_eksperimen_v3(
            query,
            kota="Bandung",
            top_k=5,
        )
    )


KUERI ASLI: tempat rekreasi keluarga di Bandung
KUERI BERSIH: rekreasi keluarga


,Place_Name,Category,City,Price,Rating,similarity_score
0,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,75000,4.3,0.089849
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.079010
2,Rainbow Garden,Cagar Alam,Bandung,20000,4.6,0.068628
3,Tafso Barn,Cagar Alam,Bandung,0,4.2,0.044439
4,Dusun Bambu,Taman Hiburan,Bandung,25000,4.5,0.035472



KUERI ASLI: waterpark di Bandung
KUERI BERSIH: waterpark


,Place_Name,Category,City,Price,Rating,similarity_score
0,Tektona Waterpark,Taman Hiburan,Bandung,60000,3.8,0.158543
1,Pesona Nirwana Waterpark & Cottages,Taman Hiburan,Bandung,50000,4.1,0.144690
2,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.109092


In [82]:
kueri_uji_v3 = [
    ("waterpark di Bandung", "Bandung"),
    ("museum di Jakarta", "Jakarta"),
    ("pantai di Yogyakarta", "Yogyakarta"),
    ("wisata alam di Semarang", "Semarang"),
    ("taman bermain di Bandung", "Bandung"),
    ("tempat rekreasi keluarga di Bandung", "Bandung"),
    ("wisata murah di Semarang", "Semarang"),
    ("budaya di Bandung", "Bandung"),
]

for query, kota in kueri_uji_v3:
    print(f"\n{'=' * 60}")
    print("Kueri:", query)
    print("Kueri bersih:", bersihkan_kueri(query, kota=kota))

    hasil = cari_destinasi_eksperimen_v3(
        query,
        kota=kota,
        top_k=5,
        harga_maks=20000 if "murah" in query.lower().split() else None,
    )

    display(hasil)


Kueri: waterpark di Bandung
Kueri bersih: waterpark


,Place_Name,Category,City,Price,Rating,similarity_score
0,Tektona Waterpark,Taman Hiburan,Bandung,60000,3.8,0.158543
1,Pesona Nirwana Waterpark & Cottages,Taman Hiburan,Bandung,50000,4.1,0.144690
2,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.109092



Kueri: museum di Jakarta
Kueri bersih: museum


,Place_Name,Category,City,Price,Rating,similarity_score
0,Museum Wayang,Budaya,Jakarta,5000,4.5,0.228504
1,Museum Nasional,Budaya,Jakarta,5000,4.6,0.205613
2,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,4.4,0.185302
3,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,4.5,0.164647
4,Museum Bahari Jakarta,Budaya,Jakarta,2000,4.4,0.160919



Kueri: pantai di Yogyakarta
Kueri bersih: pantai


...

,Place_Name,Category,City,Price,Rating,similarity_score
0,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,75000,4.3,0.089849
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.079010
2,Rainbow Garden,Cagar Alam,Bandung,20000,4.6,0.068628
3,Tafso Barn,Cagar Alam,Bandung,0,4.2,0.044439
4,Dusun Bambu,Taman Hiburan,Bandung,25000,4.5,0.035472



Kueri: wisata murah di Semarang
Kueri bersih: wisata murah


,Place_Name,Category,City,Price,Rating,similarity_score
0,Desa Wisata Lembah Kalipancur,Taman Hiburan,Semarang,0,3.9,0.026347
1,Wisata Mangrove Tapak,Cagar Alam,Semarang,5000,4.3,0.021532
2,Wisata Alam Wana Wisata Penggaron,Cagar Alam,Semarang,10000,4.1,0.019983
3,Wisata Lereng Kelir,Cagar Alam,Semarang,7000,4.3,0.019834
4,Hutan Wisata Tinjomoyo Semarang,Cagar Alam,Semarang,3000,4.3,0.018902



Kueri: budaya di Bandung
Kueri bersih: budaya


,Place_Name,Category,City,Price,Rating,similarity_score
0,Taman Budaya Jawa Barat,Budaya,Bandung,0,4.3,0.078045
1,Roemah Seni Sarasvati,Budaya,Bandung,0,4.4,0.059253
2,Museum Pendidikan Nasional,Budaya,Bandung,5000,4.6,0.053799
3,Monumen Perjuangan Rakyat Jawa Barat,Budaya,Bandung,0,4.5,0.047905
4,Taman Sejarah Bandung,Budaya,Bandung,0,4.6,0.047114


In [83]:
kasus_uji_kategori = [
    {
        "query": "waterpark di Bandung",
        "kota": "Bandung",
        "kategori": "Taman Hiburan",
    },
    {
        "query": "museum di Jakarta",
        "kota": "Jakarta",
        "kategori": "Budaya",
    },
    {
        "query": "pantai di Yogyakarta",
        "kota": "Yogyakarta",
        "kategori": "Bahari",
    },
    {
        "query": "wisata alam di Semarang",
        "kota": "Semarang",
        "kategori": "Cagar Alam",
    },
    {
        "query": "taman bermain di Bandung",
        "kota": "Bandung",
        "kategori": "Taman Hiburan",
    },
    {
        "query": "tempat rekreasi keluarga di Bandung",
        "kota": "Bandung",
        "kategori": None,
    },
    {
        "query": "wisata murah di Semarang",
        "kota": "Semarang",
        "kategori": None,
    },
    {
        "query": "budaya di Bandung",
        "kota": "Bandung",
        "kategori": "Budaya",
    },
]

for kasus in kasus_uji_kategori:
    query = kasus["query"]
    kota = kasus["kota"]
    kategori = kasus["kategori"]

    harga_maks = (
        20000
        if "murah" in query.lower().split()
        else None
    )

    print(f"\n{'=' * 60}")
    print("Kueri:", query)
    print("Kota:", kota)
    print("Kategori:", kategori)

    hasil = cari_destinasi_eksperimen_v3(
        query,
        kota=kota,
        kategori=kategori,
        harga_maks=harga_maks,
        top_k=5,
    )

    display(hasil)


Kueri: waterpark di Bandung
Kota: Bandung
Kategori: Taman Hiburan


,Place_Name,Category,City,Price,Rating,similarity_score
0,Tektona Waterpark,Taman Hiburan,Bandung,60000,3.8,0.158543
1,Pesona Nirwana Waterpark & Cottages,Taman Hiburan,Bandung,50000,4.1,0.144690
2,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.109092



Kueri: museum di Jakarta
Kota: Jakarta
Kategori: Budaya


,Place_Name,Category,City,Price,Rating,similarity_score
0,Museum Wayang,Budaya,Jakarta,5000,4.5,0.228504
1,Museum Nasional,Budaya,Jakarta,5000,4.6,0.205613
2,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,4.4,0.185302
3,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,4.5,0.164647
4,Museum Bahari Jakarta,Budaya,Jakarta,2000,4.4,0.160919



Kueri: pantai di Yogyakarta
Kota: Yogyakarta
Kategori: Bahari


...

,Place_Name,Category,City,Price,Rating,similarity_score
0,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,75000,4.3,0.089849
1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.079010
2,Rainbow Garden,Cagar Alam,Bandung,20000,4.6,0.068628
3,Tafso Barn,Cagar Alam,Bandung,0,4.2,0.044439
4,Dusun Bambu,Taman Hiburan,Bandung,25000,4.5,0.035472



Kueri: wisata murah di Semarang
Kota: Semarang
Kategori: None


,Place_Name,Category,City,Price,Rating,similarity_score
0,Desa Wisata Lembah Kalipancur,Taman Hiburan,Semarang,0,3.9,0.026347
1,Wisata Mangrove Tapak,Cagar Alam,Semarang,5000,4.3,0.021532
2,Wisata Alam Wana Wisata Penggaron,Cagar Alam,Semarang,10000,4.1,0.019983
3,Wisata Lereng Kelir,Cagar Alam,Semarang,7000,4.3,0.019834
4,Hutan Wisata Tinjomoyo Semarang,Cagar Alam,Semarang,3000,4.3,0.018902



Kueri: budaya di Bandung
Kota: Bandung
Kategori: Budaya


,Place_Name,Category,City,Price,Rating,similarity_score
0,Taman Budaya Jawa Barat,Budaya,Bandung,0,4.3,0.078045
1,Roemah Seni Sarasvati,Budaya,Bandung,0,4.4,0.059253
2,Museum Pendidikan Nasional,Budaya,Bandung,5000,4.6,0.053799
3,Monumen Perjuangan Rakyat Jawa Barat,Budaya,Bandung,0,4.5,0.047905
4,Taman Sejarah Bandung,Budaya,Bandung,0,4.6,0.047114


In [84]:
from pathlib import Path
import pandas as pd

baris_audit = []

for kasus in kasus_uji_kategori:
    query = kasus["query"]
    kota = kasus["kota"]
    kategori = kasus["kategori"]

    harga_maks = (
        20000
        if "murah" in query.lower().split()
        else None
    )

    hasil = cari_destinasi_eksperimen_v3(
        query,
        kota=kota,
        kategori=kategori,
        harga_maks=harga_maks,
        top_k=5,
    )

    for peringkat, (_, row) in enumerate(hasil.iterrows(), start=1):
        info = df_perbaikan.loc[
            df_perbaikan["Place_Name"] == row["Place_Name"]
        ].iloc[0]

        baris_audit.append({
            "query": query,
            "peringkat": peringkat,
            "Place_Name": row["Place_Name"],
            "Category": row["Category"],
            "City": row["City"],
            "Price": row["Price"],
            "similarity_score": row["similarity_score"],
            "Description": info["Description"],
            "relevant": "",
            "catatan_manual": "",
        })

audit_independen = pd.DataFrame(baris_audit)

folder_output = Path("..") / "data" / "processed"
folder_output.mkdir(parents=True, exist_ok=True)

path_output = folder_output / "audit_relevansi_independen.csv"
audit_independen.to_csv(path_output, index=False, encoding="utf-8-sig")

print("Jumlah kandidat:", len(audit_independen))
print("Berkas tersimpan di:", path_output.resolve())

display(audit_independen.head(10))

Jumlah kandidat: 38
Berkas tersimpan di: D:\WisataFinder\data\processed\audit_relevansi_independen.csv


,query,peringkat,Place_Name,Category,City,Price,similarity_score,Description,relevant,catatan_manual
0,waterpark di Bandung,1,Tektona Waterpark,Taman Hiburan,Bandung,60000,0.158543,Tektona Waterpark sebuah wahana yang memberika...,,
1,waterpark di Bandung,2,Pesona Nirwana Waterpark & Cottages,Taman Hiburan,Bandung,50000,0.144690,Pesona Nirwana Waterpark merupakan sebuah obje...,,
2,waterpark di Bandung,3,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,0.109092,Bandung Indah Waterpark merupakan salah satu k...,,
3,museum di Jakarta,1,Museum Wayang,Budaya,Jakarta,5000,0.228504,Museum Wayang adalah sebuah museum yang berlok...,,
4,museum di Jakarta,2,Museum Nasional,Budaya,Jakarta,5000,0.205613,Museum Nasional Republik Indonesia atau Museum...,,
5,museum di Jakarta,3,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,0.185302,Museum Seni Rupa dan Keramik (Indonesia: Museu...,,
6,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,0.164647,Museum Seni Modern dan Kontemporer di Nusantar...,,
7,museum di Jakarta,5,Museum Bahari Jakarta,Budaya,Jakarta,2000,0.160919,Museum Bahari adalah museum yang menyimpan kol...,,
8,pantai di Yogyakarta,1,Pantai Glagah,Bahari,Yogyakarta,5000,0.179568,Pantai Glagah Indah adalah salah satu pantai y...,,
9,pantai di Yogyakarta,2,Pantai Timang,Bahari,Yogyakarta,10000,0.165433,Pantai Timang adalah objek wisata berupa panta...,,


In [85]:
from pathlib import Path
import pandas as pd

path_audit = (
    Path("..") / "data" / "processed"
    / "audit_relevansi_independen.csv"
)

audit = pd.read_csv(path_audit)

audit["relevant"] = pd.to_numeric(
    audit["relevant"], errors="coerce"
)

hasil_evaluasi = []

for query, grup in audit.groupby("query"):
    grup = grup.sort_values("peringkat")

    # Evaluasi hanya jika semua hasil yang tersedia
    # pada lima peringkat teratas sudah diberi label.
    top5 = grup.head(5)

    if top5["relevant"].isna().any():
        continue

    precision_at_5 = top5["relevant"].sum() / 5

    hasil_evaluasi.append({
        "query": query,
        "jumlah_hasil_top5": len(top5),
        "Precision@5": precision_at_5,
    })

evaluasi = pd.DataFrame(hasil_evaluasi)

display(evaluasi)

if not evaluasi.empty:
    print(
        "Macro-average Precision@5:",
        round(evaluasi["Precision@5"].mean(), 4)
    )
else:
    print("Belum ada kueri dengan label top-5 lengkap.")

""


Belum ada kueri dengan label top-5 lengkap.


In [86]:
from pathlib import Path
import pandas as pd

path_audit = (
    Path("..") / "data" / "processed"
    / "audit_relevansi_independen.csv"
)

audit = pd.read_csv(path_audit)

print("Total kandidat:", len(audit))
print("Label sudah terisi:", audit["relevant"].notna().sum())
print("Label masih kosong:", audit["relevant"].isna().sum())

display(
    audit.groupby("query").agg(
        jumlah_kandidat=("Place_Name", "size"),
        label_terisi=("relevant", "count"),
    ).reset_index()
)

Total kandidat: 38
Label sudah terisi: 0
Label masih kosong: 38


,query,jumlah_kandidat,label_terisi
0,budaya di Bandung,5,0
1,museum di Jakarta,5,0
2,pantai di Yogyakarta,5,0
3,taman bermain di Bandung,5,0
4,tempat rekreasi keluarga di Bandung,5,0
5,waterpark di Bandung,3,0
6,wisata alam di Semarang,5,0
7,wisata murah di Semarang,5,0


In [87]:
audit = pd.read_csv(
    Path("..") / "data" / "processed"
    / "audit_relevansi_independen.csv"
)

print("Total label terisi:", audit["relevant"].notna().sum())
print("Total label kosong:", audit["relevant"].isna().sum())

display(
    audit[audit["relevant"].notna()][
        ["query", "Place_Name", "relevant"]
    ]
)

Total label terisi: 0
Total label kosong: 38


,query,Place_Name,relevant


In [88]:
from pathlib import Path
import pandas as pd

path_audit = (
    Path("..") / "data" / "processed"
    / "audit_relevansi_independen.csv"
)

audit = pd.read_csv(path_audit)

label_awal = {
    "Tektona Waterpark": 1,
    "Pesona Nirwana Waterpark & Cottages": 1,
    "Water Park Bandung Indah": 1,
    "Museum Wayang": 1,
    "Museum Nasional": 1,
    "Museum Seni Rupa dan Kramik": 1,
    "Museum Macan (Modern and Contemporary Art in N...": 1,
    "Museum Bahari Jakarta": 1,
}

for nama, label in label_awal.items():
    mask = audit["Place_Name"].eq(nama)
    audit.loc[mask, "relevant"] = label

audit.to_csv(path_audit, index=False, encoding="utf-8-sig")

print("Label terisi:", audit["relevant"].notna().sum())
print("Label kosong:", audit["relevant"].isna().sum())

display(
    audit[audit["relevant"].notna()][
        ["query", "Place_Name", "relevant"]
    ]
)

Label terisi: 8
Label kosong: 30


,query,Place_Name,relevant
0,waterpark di Bandung,Tektona Waterpark,1.0
1,waterpark di Bandung,Pesona Nirwana Waterpark & Cottages,1.0
2,waterpark di Bandung,Water Park Bandung Indah,1.0
3,museum di Jakarta,Museum Wayang,1.0
4,museum di Jakarta,Museum Nasional,1.0
5,museum di Jakarta,Museum Seni Rupa dan Kramik,1.0
7,museum di Jakarta,Museum Bahari Jakarta,1.0
24,tempat rekreasi keluarga di Bandung,Water Park Bandung Indah,1.0


In [89]:
display(
    audit.loc[
        audit["query"].eq("pantai di Yogyakarta"),
        [
            "query",
            "peringkat",
            "Place_Name",
            "Category",
            "Description",
            "relevant",
            "catatan_manual",
        ],
    ]
)

,query,peringkat,Place_Name,Category,Description,relevant,catatan_manual
8,pantai di Yogyakarta,1,Pantai Glagah,Bahari,Pantai Glagah Indah adalah salah satu pantai y...,NaN,NaN
9,pantai di Yogyakarta,2,Pantai Timang,Bahari,Pantai Timang adalah objek wisata berupa panta...,NaN,NaN
10,pantai di Yogyakarta,3,Pantai Samas,Bahari,"Pantai Samas (bahasa Jawa: ꦥꦱꦶꦱꦶꦂ​ꦱꦩꦱ꧀, transl...",NaN,NaN
11,pantai di Yogyakarta,4,Pantai Patihan,Bahari,"Pantai ini terletak di daerah Srigading, Sande...",NaN,NaN
12,pantai di Yogyakarta,5,Pantai Depok Jogja,Bahari,"Pantai Depok (bahasa Jawa: ꦥꦱꦶꦱꦶꦂ ꦝꦺꦥꦺꦴꦏ꧀, tra...",NaN,NaN


In [90]:
display(
    audit.loc[
        audit["query"].eq("pantai di Yogyakarta"),
        ["Place_Name", "City", "Description"]
    ]
)

,Place_Name,City,Description
8,Pantai Glagah,Yogyakarta,Pantai Glagah Indah adalah salah satu pantai y...
9,Pantai Timang,Yogyakarta,Pantai Timang adalah objek wisata berupa panta...
10,Pantai Samas,Yogyakarta,"Pantai Samas (bahasa Jawa: ꦥꦱꦶꦱꦶꦂ​ꦱꦩꦱ꧀, transl..."
11,Pantai Patihan,Yogyakarta,"Pantai ini terletak di daerah Srigading, Sande..."
12,Pantai Depok Jogja,Yogyakarta,"Pantai Depok (bahasa Jawa: ꦥꦱꦶꦱꦶꦂ ꦝꦺꦥꦺꦴꦏ꧀, tra..."


In [91]:
mask = audit["query"].eq("pantai di Yogyakarta")

audit.loc[mask, "relevant"] = 1
audit.loc[mask, "catatan_manual"] = (
    "Destinasi pantai dan kota sesuai kueri"
)

audit.to_csv(
    path_audit,
    index=False,
    encoding="utf-8-sig"
)

print("Total kandidat:", len(audit))
print("Label terisi:", audit["relevant"].notna().sum())
print("Label kosong:", audit["relevant"].isna().sum())

Total kandidat: 38
Label terisi: 13
Label kosong: 25


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21672\147652548.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Destinasi pantai dan kota sesuai kueri' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  audit.loc[mask, "catatan_manual"] = (


In [92]:
# Pastikan kolom catatan_manual bertipe teks
audit["catatan_manual"] = audit["catatan_manual"].astype("string")

# Simpan kembali file audit
audit.to_csv(
    path_audit,
    index=False,
    encoding="utf-8-sig"
)

print("Total kandidat:", len(audit))
print("Label terisi:", audit["relevant"].notna().sum())
print("Label kosong:", audit["relevant"].isna().sum())

Total kandidat: 38
Label terisi: 13
Label kosong: 25


In [93]:
display(
    audit.loc[
        audit["query"].eq("wisata alam di Semarang"),
        [
            "query",
            "peringkat",
            "Place_Name",
            "Category",
            "City",
            "Description",
            "relevant",
        ],
    ]
)

,query,peringkat,Place_Name,Category,City,Description,relevant
13,wisata alam di Semarang,1,Watu Gunung Ungaran,Cagar Alam,Semarang,Objek wisata alam memang sangat cocok dikunjun...,NaN
14,wisata alam di Semarang,2,Umbul Sidomukti,Cagar Alam,Semarang,Kawasan wisata umbul Sidomukti merupakan salah...,NaN
15,wisata alam di Semarang,3,Air Terjun Kali Pancur,Cagar Alam,Semarang,Kali Pancur Getasan merupakan kawasan wisata a...,NaN
16,wisata alam di Semarang,4,Wisata Alam Wana Wisata Penggaron,Cagar Alam,Semarang,Berada sekitar 2 KM dari Kota Ungaran atau sek...,NaN
17,wisata alam di Semarang,5,Wisata Eling Bening,Cagar Alam,Semarang,Sebuah fasilitas wisata yang dikelola dengan c...,NaN


In [94]:
mask = audit["query"].eq("wisata alam di Semarang")

audit.loc[mask, "relevant"] = 1
audit.loc[mask, "catatan_manual"] = (
    "Destinasi sesuai kueri wisata alam di Semarang"
)

audit["catatan_manual"] = audit["catatan_manual"].astype("string")

audit.to_csv(path_audit, index=False, encoding="utf-8-sig")

print("Total kandidat:", len(audit))
print("Label terisi:", audit["relevant"].notna().sum())
print("Label kosong:", audit["relevant"].isna().sum())

Total kandidat: 38
Label terisi: 18
Label kosong: 20


In [95]:
display(
    audit.loc[
        audit["query"].eq("taman bermain di Bandung"),
        [
            "query",
            "peringkat",
            "Place_Name",
            "Category",
            "City",
            "Description",
            "relevant",
        ],
    ]
)

,query,peringkat,Place_Name,Category,City,Description,relevant
18,taman bermain di Bandung,1,Trans Studio Bandung,Taman Hiburan,Bandung,Trans Studio Bandung adalah kawasan wisata ter...,NaN
19,taman bermain di Bandung,2,Taman Jomblo,Taman Hiburan,Bandung,Taman Jomblo atau Taman Pasupati adalah taman ...,NaN
20,taman bermain di Bandung,3,Panama Park 825,Taman Hiburan,Bandung,Panama Park Bandung menjadi salah satu tempat ...,NaN
21,taman bermain di Bandung,4,Taman Lalu Lintas Ade Irma Suryani Nasution,Taman Hiburan,Bandung,Taman Lalu-lintas Ade Irma Suryani adalah sebu...,NaN
22,taman bermain di Bandung,5,Taman Balai Kota Bandung,Taman Hiburan,Bandung,Taman Balai Kota Bandung merupakan sebuah tama...,NaN


In [96]:
mask = audit["query"].eq("taman bermain di Bandung")

label_taman = {
    "Trans Studio Bandung": 1,
    "Taman Jomblo": 0,
    "Panama Park 825": 1,
    "Taman Lalu Lintas Ade Irma Suryani Nasution": 1,
    "Taman Balai Kota Bandung": 0,
}

for nama, nilai in label_taman.items():
    baris = mask & audit["Place_Name"].eq(nama)
    audit.loc[baris, "relevant"] = nilai
    audit.loc[baris, "catatan_manual"] = (
        "Sesuai kueri taman bermain"
        if nilai == 1
        else "Belum cukup sesuai dengan maksud kueri taman bermain"
    )

audit["catatan_manual"] = audit["catatan_manual"].astype("string")

audit.to_csv(path_audit, index=False, encoding="utf-8-sig")

print("Total kandidat:", len(audit))
print("Label terisi:", audit["relevant"].notna().sum())
print("Label kosong:", audit["relevant"].isna().sum())

Total kandidat: 38
Label terisi: 23
Label kosong: 15


In [97]:
display(
    audit.loc[
        audit["query"].eq("tempat rekreasi keluarga di Bandung"),
        [
            "query",
            "peringkat",
            "Place_Name",
            "Category",
            "City",
            "Description",
            "relevant",
        ],
    ]
)

,query,peringkat,Place_Name,Category,City,Description,relevant
23,tempat rekreasi keluarga di Bandung,1,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,Panghegar Waterboom Bandung merupakan salah sa...,NaN
24,tempat rekreasi keluarga di Bandung,2,Water Park Bandung Indah,Taman Hiburan,Bandung,Bandung Indah Waterpark merupakan salah satu k...,1.0
25,tempat rekreasi keluarga di Bandung,3,Rainbow Garden,Cagar Alam,Bandung,Rainbow Garden Harapan Indah salah satu taman ...,NaN
26,tempat rekreasi keluarga di Bandung,4,Tafso Barn,Cagar Alam,Bandung,Nama Punclut mungkin sudah cukup akrab di teli...,NaN
27,tempat rekreasi keluarga di Bandung,5,Dusun Bambu,Taman Hiburan,Bandung,Dusun Bambu merupakan taman rekreasi berbasis ...,NaN


In [98]:
mask = audit["query"].eq("tempat rekreasi keluarga di Bandung")

label_keluarga = {
    "Panghegar Waterboom Bandung": 1,
    "Water Park Bandung Indah": 1,
    "Rainbow Garden": 1,
    "Tafso Barn": 1,
    "Dusun Bambu": 1,
}

for nama, nilai in label_keluarga.items():
    baris = mask & audit["Place_Name"].eq(nama)
    audit.loc[baris, "relevant"] = nilai
    audit.loc[baris, "catatan_manual"] = (
        "Relevan untuk rekreasi keluarga"
    )

audit["catatan_manual"] = audit["catatan_manual"].astype("string")
audit.to_csv(path_audit, index=False, encoding="utf-8-sig")

print("Total kandidat:", len(audit))
print("Label terisi:", audit["relevant"].notna().sum())
print("Label kosong:", audit["relevant"].isna().sum())

Total kandidat: 38
Label terisi: 27
Label kosong: 11


In [99]:
display(
    audit.loc[
        audit["query"].eq("wisata murah di Semarang"),
        [
            "query",
            "peringkat",
            "Place_Name",
            "Category",
            "City",
            "Price",
            "Description",
            "relevant",
        ],
    ]
)

,query,peringkat,Place_Name,Category,City,Price,Description,relevant
28,wisata murah di Semarang,1,Desa Wisata Lembah Kalipancur,Taman Hiburan,Semarang,0,Wisata alam tengah menjadi sorotan bagi dunia ...,NaN
29,wisata murah di Semarang,2,Wisata Mangrove Tapak,Cagar Alam,Semarang,5000,Wisata hutan mangrove Semarang hampir tak pern...,NaN
30,wisata murah di Semarang,3,Wisata Alam Wana Wisata Penggaron,Cagar Alam,Semarang,10000,Berada sekitar 2 KM dari Kota Ungaran atau sek...,NaN
31,wisata murah di Semarang,4,Wisata Lereng Kelir,Cagar Alam,Semarang,7000,Wisata yang ada di kawasan gunung Kelir ini me...,NaN
32,wisata murah di Semarang,5,Hutan Wisata Tinjomoyo Semarang,Cagar Alam,Semarang,3000,Awalnya taman wisata hutan Tinjomoyo Semarang ...,NaN


In [101]:
mask = audit["query"].eq("wisata murah di Semarang")

label_murah = {
    "Desa Wisata Lembah Kalipancur": 1,
    "Wisata Mangrove Tapak": 1,
    "Wisata Alam Wana Wisata Penggaron": 1,
    "Wisata Lereng Kelir": 1,
    "Hutan Wisata Tinjomoyo Semarang": 1,
}

for nama, nilai in label_murah.items():
    baris = mask & audit["Place_Name"].eq(nama)
    audit.loc[baris, "relevant"] = nilai
    audit.loc[baris, "catatan_manual"] = (
        "Sesuai kota dan batas harga; harga nol perlu diverifikasi"
        if nama == "Desa Wisata Lembah Kalipancur"
        else "Sesuai kota dan batas harga Rp20.000"
    )

audit["catatan_manual"] = audit["catatan_manual"].astype("string")
audit.to_csv(path_audit, index=False, encoding="utf-8-sig")

print("Total kandidat:", len(audit))
print("Label terisi:", audit["relevant"].notna().sum())
print("Label kosong:", audit["relevant"].isna().sum())

Total kandidat: 38
Label terisi: 32
Label kosong: 6


In [102]:
display(
    audit.loc[
        audit["query"].eq("budaya di Bandung"),
        [
            "query",
            "peringkat",
            "Place_Name",
            "Category",
            "City",
            "Description",
            "relevant",
        ],
    ]
)

,query,peringkat,Place_Name,Category,City,Description,relevant
33,budaya di Bandung,1,Taman Budaya Jawa Barat,Budaya,Bandung,Taman Budaya Jawa Barat (TBJB) didirikan pada ...,NaN
34,budaya di Bandung,2,Roemah Seni Sarasvati,Budaya,Bandung,Roemah Seni Sarasvati adalah sebuah galeri yan...,NaN
35,budaya di Bandung,3,Museum Pendidikan Nasional,Budaya,Bandung,Museum Pendidikan Nasional Universitas Pendidi...,NaN
36,budaya di Bandung,4,Monumen Perjuangan Rakyat Jawa Barat,Budaya,Bandung,Monumen Perjuangan Rakyat Jawa Barat (Monju) a...,NaN
37,budaya di Bandung,5,Taman Sejarah Bandung,Budaya,Bandung,Taman Sejarah Bandung adalah taman yang terlet...,NaN


In [103]:
mask = audit["query"].eq("budaya di Bandung")

label_budaya = {
    "Taman Budaya Jawa Barat": 1,
    "Roemah Seni Sarasvati": 1,
    "Museum Pendidikan Nasional": 1,
    "Monumen Perjuangan Rakyat Jawa Barat": 1,
    "Taman Sejarah Bandung": 1,
}

for nama, nilai in label_budaya.items():
    baris = mask & audit["Place_Name"].eq(nama)
    audit.loc[baris, "relevant"] = nilai
    audit.loc[baris, "catatan_manual"] = (
        "Destinasi wisata budaya di Bandung"
    )

audit["catatan_manual"] = audit["catatan_manual"].astype("string")
audit.to_csv(path_audit, index=False, encoding="utf-8-sig")

print("Total kandidat:", len(audit))
print("Label terisi:", audit["relevant"].notna().sum())
print("Label kosong:", audit["relevant"].isna().sum())

Total kandidat: 38
Label terisi: 37
Label kosong: 1


In [104]:
display(
    audit.loc[
        audit["relevant"].isna(),
        [
            "query",
            "peringkat",
            "Place_Name",
            "Category",
            "City",
            "Price",
            "Description",
        ],
    ]
)

,query,peringkat,Place_Name,Category,City,Price,Description
6,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in N...,Budaya,Jakarta,100000,Museum Seni Modern dan Kontemporer di Nusantar...


In [105]:
mask = (
    audit["query"].eq("museum di Jakarta")
    & audit["Place_Name"].str.startswith("Museum Macan", na=False)
)

audit.loc[mask, "relevant"] = 1
audit.loc[mask, "catatan_manual"] = (
    "Museum seni modern dan kontemporer di Jakarta"
)

audit["catatan_manual"] = audit["catatan_manual"].astype("string")

audit.to_csv(
    path_audit,
    index=False,
    encoding="utf-8-sig"
)

print("Total kandidat:", len(audit))
print("Label terisi:", audit["relevant"].notna().sum())
print("Label kosong:", audit["relevant"].isna().sum())

Total kandidat: 38
Label terisi: 38
Label kosong: 0


In [106]:
import pandas as pd

hasil_evaluasi = []

for query, grup in audit.groupby("query", sort=False):
    grup = grup.sort_values("peringkat")

    if grup["relevant"].isna().any():
        print(f"Label belum lengkap: {query}")
        continue

    k = min(5, len(grup))
    jumlah_relevan = grup.head(k)["relevant"].sum()
    precision = jumlah_relevan / k

    hasil_evaluasi.append({
        "query": query,
        "jumlah_hasil": len(grup),
        "k": k,
        "jumlah_relevan": int(jumlah_relevan),
        "precision": precision,
    })

evaluasi = pd.DataFrame(hasil_evaluasi)

display(evaluasi.round(3))

print(
    "Macro-average Precision:",
    round(evaluasi["precision"].mean(), 3)
)

,query,jumlah_hasil,k,jumlah_relevan,precision
0,waterpark di Bandung,3,3,3,1.0
1,museum di Jakarta,5,5,5,1.0
2,pantai di Yogyakarta,5,5,5,1.0
3,wisata alam di Semarang,5,5,5,1.0
4,taman bermain di Bandung,5,5,3,0.6
5,tempat rekreasi keluarga di Bandung,5,5,5,1.0
6,wisata murah di Semarang,5,5,5,1.0
7,budaya di Bandung,5,5,5,1.0


Macro-average Precision: 0.95


In [107]:
print("Jumlah kandidat:", len(audit))
print("Label kosong:", audit["relevant"].isna().sum())
print("Nilai label:", sorted(audit["relevant"].dropna().unique()))

display(
    audit.groupby("query").agg(
        jumlah_hasil=("Place_Name", "count"),
        jumlah_relevan=("relevant", "sum"),
        label_kosong=("relevant", lambda s: s.isna().sum()),
    ).reset_index()
)

Jumlah kandidat: 38
Label kosong: 0
Nilai label: [np.float64(0.0), np.float64(1.0)]


,query,jumlah_hasil,jumlah_relevan,label_kosong
0,budaya di Bandung,5,5.0,0
1,museum di Jakarta,5,5.0,0
2,pantai di Yogyakarta,5,5.0,0
3,taman bermain di Bandung,5,3.0,0
4,tempat rekreasi keluarga di Bandung,5,5.0,0
5,waterpark di Bandung,3,3.0,0
6,wisata alam di Semarang,5,5.0,0
7,wisata murah di Semarang,5,5.0,0


In [108]:
display(
    audit.loc[
        audit["relevant"].eq(0),
        [
            "query",
            "peringkat",
            "Place_Name",
            "Category",
            "City",
            "Description",
            "relevant",
            "catatan_manual",
        ],
    ]
)

,query,peringkat,Place_Name,Category,City,Description,relevant,catatan_manual
19,taman bermain di Bandung,2,Taman Jomblo,Taman Hiburan,Bandung,Taman Jomblo atau Taman Pasupati adalah taman ...,0.0,Belum cukup sesuai dengan maksud kueri taman b...
22,taman bermain di Bandung,5,Taman Balai Kota Bandung,Taman Hiburan,Bandung,Taman Balai Kota Bandung merupakan sebuah tama...,0.0,Belum cukup sesuai dengan maksud kueri taman b...


In [109]:
pd.set_option("display.max_colwidth", None)

display(
    audit.loc[
        audit["Place_Name"].isin([
            "Taman Jomblo",
            "Taman Balai Kota Bandung",
        ]),
        ["Place_Name", "Description", "relevant", "catatan_manual"],
    ]
)

,Place_Name,Description,relevant,catatan_manual
19,Taman Jomblo,Taman Jomblo atau Taman Pasupati adalah taman publik di Bandung.,0.0,Belum cukup sesuai dengan maksud kueri taman bermain
22,Taman Balai Kota Bandung,"Taman Balai Kota Bandung merupakan sebuah taman kota yang terletak di Kota Bandung. Taman ini berada di sebelah selatan Balai Kota Bandung. Saat ini, Taman Balai Kota Bandung terdiri dari dua taman yang menyatu, antara lain Taman Dewi Sartika dan Taman Badak. Kedua taman tersebut diresmikan oleh Walikota Bandung saat itu, Ridwan Kamil, pada tanggal 19 Desember 2017.",0.0,Belum cukup sesuai dengan maksud kueri taman bermain


In [110]:
catatan_perbaikan = {
    "Taman Jomblo": (
        "Label 0 sementara: deskripsi hanya menyebut taman publik "
        "dan tidak menjelaskan fasilitas bermain."
    ),
    "Taman Balai Kota Bandung": (
        "Label 0 sementara: deskripsi menyebut taman kota, "
        "tetapi tidak menjelaskan fasilitas bermain."
    ),
}

for nama, catatan in catatan_perbaikan.items():
    mask = audit["Place_Name"].eq(nama) & audit["query"].eq(
        "taman bermain di Bandung"
    )
    audit.loc[mask, "catatan_manual"] = catatan

display(
    audit.loc[
        audit["query"].eq("taman bermain di Bandung"),
        ["peringkat", "Place_Name", "relevant", "catatan_manual"],
    ].sort_values("peringkat")
)

,peringkat,Place_Name,relevant,catatan_manual
18,1,Trans Studio Bandung,1.0,Sesuai kueri taman bermain
19,2,Taman Jomblo,0.0,Label 0 sementara: deskripsi hanya menyebut taman publik dan tidak menjelaskan fasilitas bermain.
20,3,Panama Park 825,1.0,Sesuai kueri taman bermain
21,4,Taman Lalu Lintas Ade Irma Suryani Nasution,1.0,Sesuai kueri taman bermain
22,5,Taman Balai Kota Bandung,0.0,"Label 0 sementara: deskripsi menyebut taman kota, tetapi tidak menjelaskan fasilitas bermain."


In [111]:
hasil_evaluasi = []

for query, grup in audit.groupby("query", sort=False):
    grup = grup.sort_values("peringkat")

    k = min(5, len(grup))
    jumlah_relevan = grup.head(k)["relevant"].sum()

    hasil_evaluasi.append({
        "query": query,
        "jumlah_hasil": len(grup),
        "k": k,
        "jumlah_relevan": int(jumlah_relevan),
        "precision": jumlah_relevan / k,
    })

evaluasi = pd.DataFrame(hasil_evaluasi)

display(evaluasi.round(3))
print(
    "Macro-average Precision:",
    round(evaluasi["precision"].mean(), 3)
)

,query,jumlah_hasil,k,jumlah_relevan,precision
0,waterpark di Bandung,3,3,3,1.0
1,museum di Jakarta,5,5,5,1.0
2,pantai di Yogyakarta,5,5,5,1.0
3,wisata alam di Semarang,5,5,5,1.0
4,taman bermain di Bandung,5,5,3,0.6
5,tempat rekreasi keluarga di Bandung,5,5,5,1.0
6,wisata murah di Semarang,5,5,5,1.0
7,budaya di Bandung,5,5,5,1.0


Macro-average Precision: 0.95


In [112]:
from pathlib import Path

folder_output = Path("..") / "data" / "processed"
folder_output.mkdir(parents=True, exist_ok=True)

evaluasi.to_csv(
    folder_output / "evaluasi_awal.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Evaluasi awal berhasil disimpan.")
print("Jumlah kueri:", len(evaluasi))
print(
    "Macro-average Precision:",
    round(evaluasi["precision"].mean(), 3)
)

Evaluasi awal berhasil disimpan.
Jumlah kueri: 8
Macro-average Precision: 0.95


In [113]:

from pathlib import Path
import pandas as pd

kueri_uji = [
    "waterpark di Bandung",
    "museum di Jakarta",
    "pantai di Yogyakarta",
    "wisata alam di Semarang",
    "taman bermain di Bandung",
    "tempat rekreasi keluarga di Bandung",
    "wisata murah di Semarang",
    "budaya di Bandung",
]

hasil_baseline = []

for query in kueri_uji:
    hasil = cari_destinasi_v3(query, top_k=5)

    if hasil is None or len(hasil) == 0:
        print(f"Tidak ada hasil: {query}")
        continue

    hasil = pd.DataFrame(hasil).copy()
    hasil["query"] = query
    hasil["peringkat"] = range(1, len(hasil) + 1)

    hasil_baseline.append(hasil)

if hasil_baseline:
    kandidat_baseline = pd.concat(
        hasil_baseline,
        ignore_index=True
    )

    kolom_tampil = [
        kolom for kolom in [
            "query",
            "peringkat",
            "Place_Name",
            "Category",
            "City",
            "Price",
            "similarity_score",
        ]
        if kolom in kandidat_baseline.columns
    ]

    display(kandidat_baseline[kolom_tampil])

    folder_output = Path("..") / "data" / "processed"
    folder_output.mkdir(parents=True, exist_ok=True)

    kandidat_baseline.to_csv(
        folder_output / "kandidat_baseline.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print("Kandidat baseline berhasil disimpan.")
    print("Jumlah kandidat:", len(kandidat_baseline))


Kota terdeteksi: Bandung
Kategori terdeteksi: Tidak ada
Batas harga: Tidak ada
Kota terdeteksi: Jakarta
Kategori terdeteksi: Budaya
Batas harga: Tidak ada
Kota terdeteksi: Yogyakarta
Kategori terdeteksi: Bahari
Batas harga: Tidak ada
Kota terdeteksi: Semarang
Kategori terdeteksi: Cagar Alam
Batas harga: Tidak ada
Kota terdeteksi: Bandung
Kategori terdeteksi: Tidak ada
Batas harga: Tidak ada
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada
Kota terdeteksi: Semarang
Kategori terdeteksi: Tidak ada
Batas harga: Rp20,000
Kota terdeteksi: Bandung
Kategori terdeteksi: Budaya
Batas harga: Tidak ada


,query,peringkat,Place_Name,Category,City,Price,similarity_score
0,waterpark di Bandung,1,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,0.1772
1,waterpark di Bandung,2,Pesona Nirwana Waterpark & Cottages,Taman Hiburan,Bandung,50000,0.1306
2,waterpark di Bandung,3,Tektona Waterpark,Taman Hiburan,Bandung,60000,0.1267
3,waterpark di Bandung,4,Panama Park 825,Taman Hiburan,Bandung,50000,0.1151
4,waterpark di Bandung,5,NuArt Sculpture Park,Taman Hiburan,Bandung,50000,0.0892
5,museum di Jakarta,1,Museum Seni Rupa dan Kramik,Budaya,Jakarta,5000,0.2207
6,museum di Jakarta,2,Museum Nasional,Budaya,Jakarta,5000,0.1666
7,museum di Jakarta,3,Museum Wayang,Budaya,Jakarta,5000,0.1436
8,museum di Jakarta,4,Museum Macan (Modern and Contemporary Art in Nusantara),Budaya,Jakarta,100000,0.1393
9,museum di Jakarta,5,Museum Joang 45,Budaya,Jakarta,2000,0.1366


Kandidat baseline berhasil disimpan.
Jumlah kandidat: 40


In [114]:

# Gunakan label relevansi dari audit versi perbaikan
label_lama = audit[
    ["query", "Place_Name", "relevant", "catatan_manual"]
].drop_duplicates(subset=["query", "Place_Name"])

baseline_audit = kandidat_baseline.merge(
    label_lama,
    on=["query", "Place_Name"],
    how="left",
)

# Tambahkan deskripsi destinasi agar penilaian manual lebih mudah
if "Description" not in baseline_audit.columns and "Description" in df.columns:
    deskripsi = df[["Place_Name", "Description"]].drop_duplicates(
        subset=["Place_Name"]
    )
    baseline_audit = baseline_audit.merge(
        deskripsi,
        on="Place_Name",
        how="left",
    )

# Pastikan kandidat yang belum dinilai tetap kosong
print("Total kandidat baseline:", len(baseline_audit))
print("Sudah memiliki label:", baseline_audit["relevant"].notna().sum())
print("Belum memiliki label:", baseline_audit["relevant"].isna().sum())

kolom_audit = [
    c for c in [
        "query", "peringkat", "Place_Name", "Category",
        "City", "Price", "Description", "similarity_score",
        "relevant", "catatan_manual",
    ]
    if c in baseline_audit.columns
]

display(
    baseline_audit.loc[
        baseline_audit["relevant"].isna(), kolom_audit
    ].sort_values(["query", "peringkat"])
)

baseline_audit.to_csv(
    Path("..") / "data" / "processed" / "audit_baseline.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Template audit baseline berhasil disimpan.")


Total kandidat baseline: 40
Sudah memiliki label: 20
Belum memiliki label: 20


,query,peringkat,Place_Name,Category,City,Price,Description,similarity_score,relevant,catatan_manual
36,budaya di Bandung,2,Monumen Bandung Lautan Api,Budaya,Bandung,0,"Monumen Bandung Lautan Api, merupakan monumen yang menjadi markah tanah Bandung. Monumen ini setinggi 45 meter, memiliki sisi sebanyak 9 bidang.\nMonumen ini dibangun untuk memperingati peristiwa Bandung Lautan Api, dimana terjadi pembumihangusan Bandung Selatan yang dipimpin oleh Muhammad Toha.\nMonumen ini berada di tengah-tengah kota yaitu terletak di kawasan Lapangan Tegallega. Monumen ini menjadi salah satu monumen terkenal di Bandung.\nMonumen ini menjadi pusat perhatian setiap tanggal 23 Maret mengenang peristiwa Bandung Lautan Api.",0.0726,NaN,<NA>
37,budaya di Bandung,3,Amazing Art World,Budaya,Bandung,75000,"Amazing Art World Bandung, sebuah objek wisata yang terletak di Bandung, yang menegaskan bahwa destinasi wisata itu bukan hanya sekedar tentang pantai, air terjun, atau wisata alam lainnya saja. Hasil karya lukisanpun ternyata mampu menjadi sebuah destinasi wisata yang sangat menyenangkan. Apalagi lukisan 3D, selain keren, tetapi mampu membuat manipulasi bagi mata para pengunjung. Luasnya Amazing Art World Bandung menjadikannya sebagai salah-satu museum lukisan 3D terbesar, bukan hanya di tingkat nasional, tetapi di dunia.",0.0665,NaN,<NA>
39,budaya di Bandung,5,Bandros City Tour,Budaya,Bandung,40000,"Bandros atau Bus Wisata Bandung adalah bus wisata yang beroperasi di Bandung, Jawa Barat, Indonesia.Digunakan oleh wisatawan di Kota Bandung. Bus tersebut diresmikan oleh Wali Kota Bandung Ridwan Kamil pada malam tahun baru 2014, dan pada 18 Februari 2018 diumumkan bus wisata akan melayani kota Bandung. Seorang pemimpin tur lokal akan menemani turis di dalam bus. Bandros dikelola oleh dua pihak: Dinas Perhubungan Bandung (yang terkait dengan Anggaran Pendapatan dan Belanja Daerah) dan Mang Dudung yang merupakan bagian dari Tanggung Jawab Sosial Perusahaan. Bus memiliki kapasitas 22 orang. Ada enam bus yang dioperasikan oleh divisi Corporate Social Responsibility, dan 12 bus dioperasikan oleh Anggaran Pendapatan dan Belanja Daerah yang dikelola oleh BLUD UPT Dinas Perhubungan Kota Bandung.",0.0569,NaN,<NA>
9,museum di Jakarta,5,Museum Joang 45,Budaya,Jakarta,2000,"Gedung Joang '45 atau Museum Joang 45 adalah salah satu museum yang berada di Jakarta. Saat ini pengelolaannya dilaksanakan oleh Dinas Pariwisata dan Kebudayaan Provinsi DKI Jakarta. Museum ini terletak di Jalan Menteng Raya 31, Kelurahan Kebon Sirih, Kecamatan Menteng, Jakarta Pusat. Museum ini diresmikan pada tahun 1974 oleh Presiden Soeharto, setelah dilakukan renovasi.",0.1366,NaN,<NA>
11,pantai di Yogyakarta,2,Pantai Parangtritis,Bahari,Yogyakarta,10000,"Pantai Parangtritis (bahasa Jawa: ꦥꦱꦶꦱꦶꦂ ꦥꦫꦁꦠꦿꦶꦠꦶꦱ꧀, translit. Pasisir Parangtritis) adalah tempat wisata yang terletak di Desa Parangtritis, Kretek, Kabupaten Bantul, Daerah Istimewa Yogyakarta. Jaraknya kurang lebih 27 km dari pusat Kota Yogyakarta. Pantai ini boleh terbilang adalah salah satu destinasi wisata terkenal di Yogyakarta dan paling favorit diantara para wisatawan sejak lama, bahkan Pantai Parangtritis telah menjadi ikon pariwisata di Yogyakarta. Pantai Parangtritis mempunyai nilai simbolis yang merupakan garis yang bersifat magis yang menghubungkan Panggung Krapyak, Keraton Yogyakarta, Tugu Yogyakarta dan Gunung Merapi yang dikenal sebagai Garis Imajiner Yogyakarta.\nPantai yang terletak di sisi timur Pantai Parangkusumo ini terkenal diantara para wisatawan karena memiliki legenda yang melekat dengan Ratu Kidul sebagai penguasa laut selatan dan keindahannya. Pantai ini merupakan pantai yang cukup luas di Yogyakarta, berbeda dengan pantai-pantai di kawasan Yogyakarta lainya seperti Pantai di Gunungkidul yang ukurannya relatif kecil.",0.2034,NaN,<NA>
13,pantai di Yogyakarta,4,Pantai Sepanjang,Bahari,Yogyakarta,10000,"Siapa sangka, ternyata Yogyakarta pun punya “Kuta”-nya sendiri Ialah Pantai Sepanjang, salah satu pantai di K

Template audit baseline berhasil disimpan.


In [115]:

label_saran = {
    ("budaya di Bandung", "Monumen Bandung Lautan Api"): 1,
    ("budaya di Bandung", "Amazing Art World"): 1,
    ("budaya di Bandung", "Bandros City Tour"): 0,
    ("museum di Jakarta", "Museum Joang 45"): 1,
    ("pantai di Yogyakarta", "Pantai Parangtritis"): 1,
    ("pantai di Yogyakarta", "Pantai Sepanjang"): 1,
    ("pantai di Yogyakarta", "Pantai Jungwok"): 1,
    ("taman bermain di Bandung", "Happyfarm Ciwidey"): 1,
    ("tempat rekreasi keluarga di Bandung", "Taman Jomblo"): 0,
    ("tempat rekreasi keluarga di Bandung", "Panama Park 825"): 1,
    ("tempat rekreasi keluarga di Bandung", "Teras Cikapundung BBWS"): 1,
    ("waterpark di Bandung", "Panama Park 825"): 0,
    ("waterpark di Bandung", "NuArt Sculpture Park"): 0,
    ("wisata alam di Semarang", "Brown Canyon"): 1,
    ("wisata alam di Semarang", "Danau Rawa Pening"): 1,
    ("wisata murah di Semarang", "Pura Giri Natha"): 1,
    ("wisata murah di Semarang", "Brown Canyon"): 0,
    ("wisata murah di Semarang", "Semarang Contemporary Art Gallery"): 1,
    ("wisata murah di Semarang", "Lawang Sewu"): 1,
    ("wisata murah di Semarang", "Grand Maerakaca"): 1,
}

for (query, nama), label in label_saran.items():
    mask = (
        baseline_audit["query"].eq(query)
        & baseline_audit["Place_Name"].eq(nama)
        & baseline_audit["relevant"].isna()
    )
    baseline_audit.loc[mask, "relevant"] = label
    baseline_audit.loc[mask, "catatan_manual"] = (
        "Label awal berdasarkan deskripsi dataset; "
        "perlu verifikasi jika informasi belum cukup."
    )

print("Jumlah kandidat:", len(baseline_audit))
print("Label kosong:", baseline_audit["relevant"].isna().sum())
print("Jumlah label 1:", (baseline_audit["relevant"] == 1).sum())
print("Jumlah label 0:", (baseline_audit["relevant"] == 0).sum())

display(
    baseline_audit.loc[
        baseline_audit["relevant"].isna(),
        ["query", "peringkat", "Place_Name"]
    ]
)


Jumlah kandidat: 40
Label kosong: 0
Jumlah label 1: 34
Jumlah label 0: 6


,query,peringkat,Place_Name


In [116]:

hasil_baseline_eval = []

for query, grup in baseline_audit.groupby("query", sort=False):
    grup = grup.sort_values("peringkat")

    k = min(5, len(grup))
    jumlah_relevan = int(grup.head(k)["relevant"].sum())

    hasil_baseline_eval.append({
        "query": query,
        "jumlah_hasil": len(grup),
        "k": k,
        "jumlah_relevan": jumlah_relevan,
        "precision": jumlah_relevan / k,
    })

evaluasi_baseline = pd.DataFrame(hasil_baseline_eval)

display(evaluasi_baseline.round(3))

macro_precision_baseline = evaluasi_baseline["precision"].mean()

print(
    "Macro-average Precision baseline:",
    round(macro_precision_baseline, 3)
)


,query,jumlah_hasil,k,jumlah_relevan,precision
0,waterpark di Bandung,5,5,3,0.6
1,museum di Jakarta,5,5,5,1.0
2,pantai di Yogyakarta,5,5,5,1.0
3,wisata alam di Semarang,5,5,5,1.0
4,taman bermain di Bandung,5,5,4,0.8
5,tempat rekreasi keluarga di Bandung,5,5,4,0.8
6,wisata murah di Semarang,5,5,4,0.8
7,budaya di Bandung,5,5,4,0.8


Macro-average Precision baseline: 0.85


In [117]:

# Periksa label pada destinasi yang muncul di kedua versi
label_baseline = baseline_audit[
    ["query", "Place_Name", "relevant"]
].rename(columns={"relevant": "relevant_baseline"})

label_perbaikan = audit[
    ["query", "Place_Name", "relevant"]
].rename(columns={"relevant": "relevant_perbaikan"})

perbandingan_label = label_baseline.merge(
    label_perbaikan,
    on=["query", "Place_Name"],
    how="inner",
)

label_berbeda = perbandingan_label[
    perbandingan_label["relevant_baseline"]
    != perbandingan_label["relevant_perbaikan"]
]

print("Destinasi-kueri yang muncul di kedua versi:",
      len(perbandingan_label))
print("Label yang berbeda:", len(label_berbeda))

display(label_berbeda)


Destinasi-kueri yang muncul di kedua versi: 20
Label yang berbeda: 0


,query,Place_Name,relevant_baseline,relevant_perbaikan


In [118]:

perbandingan_precision = evaluasi_baseline[
    ["query", "precision"]
].rename(columns={"precision": "precision_baseline"}).merge(
    evaluasi[["query", "precision"]].rename(
        columns={"precision": "precision_perbaikan"}
    ),
    on="query",
    how="inner",
)

perbandingan_precision["selisih"] = (
    perbandingan_precision["precision_perbaikan"]
    - perbandingan_precision["precision_baseline"]
)

display(perbandingan_precision.round(3))

print(
    "Rata-rata baseline:",
    round(perbandingan_precision["precision_baseline"].mean(), 3)
)
print(
    "Rata-rata versi perbaikan:",
    round(perbandingan_precision["precision_perbaikan"].mean(), 3)
)
print(
    "Rata-rata selisih:",
    round(perbandingan_precision["selisih"].mean(), 3)
)


,query,precision_baseline,precision_perbaikan,selisih
0,waterpark di Bandung,0.6,1.0,0.4
1,museum di Jakarta,1.0,1.0,0.0
2,pantai di Yogyakarta,1.0,1.0,0.0
3,wisata alam di Semarang,1.0,1.0,0.0
4,taman bermain di Bandung,0.8,0.6,-0.2
5,tempat rekreasi keluarga di Bandung,0.8,1.0,0.2
6,wisata murah di Semarang,0.8,1.0,0.2
7,budaya di Bandung,0.8,1.0,0.2


Rata-rata baseline: 0.85
Rata-rata versi perbaikan: 0.95
Rata-rata selisih: 0.1


In [119]:
display(
    audit.loc[
        audit["query"].eq("taman bermain di Bandung"),
        [
            "peringkat",
            "Place_Name",
            "Category",
            "similarity_score",
            "relevant",
            "catatan_manual",
        ],
    ].sort_values("peringkat")
)

,peringkat,Place_Name,Category,similarity_score,relevant,catatan_manual
18,1,Trans Studio Bandung,Taman Hiburan,0.110372,1.0,Sesuai kueri taman bermain
19,2,Taman Jomblo,Taman Hiburan,0.053001,0.0,Label 0 sementara: deskripsi hanya menyebut taman publik dan tidak menjelaskan fasilitas bermain.
20,3,Panama Park 825,Taman Hiburan,0.050225,1.0,Sesuai kueri taman bermain
21,4,Taman Lalu Lintas Ade Irma Suryani Nasution,Taman Hiburan,0.049727,1.0,Sesuai kueri taman bermain
22,5,Taman Balai Kota Bandung,Taman Hiburan,0.034218,0.0,"Label 0 sementara: deskripsi menyebut taman kota, tetapi tidak menjelaskan fasilitas bermain."


In [120]:

from sklearn.metrics.pairwise import cosine_similarity

# Kueri asli dan kueri yang diperluas
query_asli = "taman bermain di Bandung"
query_bersih = bersihkan_kueri(query_asli, kota="Bandung")

query_perluasan = (
    query_bersih
    + " playground wahana anak rekreasi keluarga"
)

print("Kueri bersih:", query_bersih)
print("Kueri diperluas:", query_perluasan)

# Hitung kemiripan menggunakan TF-IDF yang sudah ada
vektor_kueri = vectorizer_v2.transform([query_perluasan])
skor = cosine_similarity(
    vektor_kueri,
    tfidf_matrix_v2
).flatten()

# Susun hasil untuk eksperimen
hasil_expansion = df.copy()
hasil_expansion["similarity_score"] = skor

# Filter kota dan hilangkan hasil dengan skor nol
hasil_expansion = hasil_expansion[
    (hasil_expansion["City"].eq("Bandung"))
    & (hasil_expansion["similarity_score"] > 0)
].copy()

hasil_expansion = hasil_expansion.sort_values(
    "similarity_score",
    ascending=False
).head(5)

display(
    hasil_expansion[
        [
            "Place_Name",
            "Category",
            "City",
            "similarity_score",
            "Description",
        ]
    ]
)


Kueri bersih: taman bermain
Kueri diperluas: taman bermain playground wahana anak rekreasi keluarga


,Place_Name,Category,City,similarity_score,Description
266,Panama Park 825,Taman Hiburan,Bandung,0.120687,"Panama Park Bandung menjadi salah satu tempat bermain di Bandung, Jawa Barat. Wahana rekreasi yang disebut-sebut terbesar di Bandung ini dibuka sejak Mei 2019 lalu. Ada beberapa permainan yang ditawarkan di arena ini, misalnya playground hingga snow park yang berisi salju. Pengguna pun bisa merasakan sensasi salju sebenarnya."
290,Water Park Bandung Indah,Taman Hiburan,Bandung,0.110764,Bandung Indah Waterpark merupakan salah satu kolam renang yang ada di Bandung. Menyediakan berbagai wahana permainan air yang cocok bagi keluarga. Terdapat kolam renang untuk dewasa dan anak-anak. Pengunjung dapat melakukan olahraga renang dan rekreasi.
236,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,0.109042,"Panghegar Waterboom Bandung merupakan salah satu destinasi wisata yang wajib dikunjungi bagi wisatawan yang hendak mengajak buah hatinya bermain air. Terletak di Buah Batu, lokasi ini kerap kali diincar saat akhir pekan. Tak hanya menyuguhkan kolam renang, tempat ini juga menyediakan berbagai wahana menarik yang sayang jika dilewatkan. Terdiri dari seluncuran air yang dibagi untuk 2 kategori yaitu seluncuran orang dewasa dan seluncuran anak-anak. Destinasi rekreasi keluarga menawarkan beragam kegiatan seru, baik untuk anak-anak maupun dewasa. Terdapat beberapa jenis kolam yang bisa dicoba: kolam ombak, kolam mini, kolam arus, dan lain sebagainya. Ada pun yang paling baru di Panghegar Waterboom Bandung yaitu kolam air panas yang disajikan bagi wisatawan yang hendak berendam."
280,Tektona Waterpark,Taman Hiburan,Bandung,0.089725,"Tektona Waterpark sebuah wahana yang memberikan sensasi bermain air bersama keluarga dan sahabat anda, memberikan keceriaan bagi anak-anak maupun pengunjung dewasa dengan fasilitas modern diantaranya Kolam Ombak dan Kolam Arus, Kolam Anak dan Balita, Semi Olympic serta Mini Slider untuk anak-anak dan Mega Slider untuk dewasa dengan ketinggian hampir dari 30 meter, yang akan memicu adrenaline para pengunjung. Selain itu wahana Tektona Waterpark juga menyediakan pojok kuliner, gazebo serta jasa pompa dan foto aksi yang dapat diperoleh di sudut wahana Tektona Waterpark, dan juga selalu tersedia ruang ganti serta ruang bilas di dalam kawasan Tektona Waterpark."
331,Rainbow Garden,Cagar Alam,Bandung,0.087786,"Rainbow Garden Harapan Indah salah satu taman rekreasi yang terutama cocok untuk keluarga. Berkonsep ‘eat and play’, Rainbow Garden Bekasi memadukan arena permainan anak dengan wisata kuliner. Berbagai wahana dan aktivitas tersedia di taman wisata penuh warna seluas 3000 m2 ini."


In [121]:

# Label awal untuk eksperimen query expansion.
# Rainbow Garden diberi 0 sementara karena lokasi pada deskripsi
# tidak konsisten dengan kueri Bandung.
label_expansion = [1, 1, 1, 1, 0]

hasil_expansion = hasil_expansion.sort_values(
    "similarity_score", ascending=False
).head(5).copy()

hasil_expansion["relevant_exp"] = label_expansion

display(
    hasil_expansion[
        [
            "Place_Name",
            "City",
            "similarity_score",
            "relevant_exp",
        ]
    ]
)

precision_expansion = hasil_expansion["relevant_exp"].sum() / 5

print("Precision@5 eksperimen:", round(precision_expansion, 3))


,Place_Name,City,similarity_score,relevant_exp
266,Panama Park 825,Bandung,0.120687,1
290,Water Park Bandung Indah,Bandung,0.110764,1
236,Panghegar Waterboom Bandung,Bandung,0.109042,1
280,Tektona Waterpark,Bandung,0.089725,1
331,Rainbow Garden,Bandung,0.087786,0


Precision@5 eksperimen: 0.8


In [122]:

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Tiga kueri untuk menguji query expansion
skenario_uji = [
    {
        "query": "tempat rekreasi keluarga di Bandung",
        "kota": "Bandung",
        "tambahan": "keluarga wahana anak bermain rekreasi"
    },
    {
        "query": "wisata alam di Semarang",
        "kota": "Semarang",
        "tambahan": "alam hutan pegunungan air terjun pemandangan"
    },
    {
        "query": "budaya di Bandung",
        "kota": "Bandung",
        "tambahan": "budaya museum sejarah seni tradisi monumen"
    }
]

hasil_semua_expansion = []

for skenario in skenario_uji:
    query = skenario["query"]
    kota = skenario["kota"]

    query_bersih = bersihkan_kueri(query, kota=kota)
    query_diperluas = (
        query_bersih + " " + skenario["tambahan"]
    )

    vektor_query = vectorizer_v2.transform([query_diperluas])
    skor = cosine_similarity(
        vektor_query, tfidf_matrix_v2
    ).flatten()

    hasil = df.copy()
    hasil["similarity_score"] = skor

    hasil = hasil[
        (hasil["City"].eq(kota))
        & (hasil["similarity_score"] > 0)
    ].copy()

    hasil = hasil.sort_values(
        "similarity_score", ascending=False
    ).head(5).copy()

    hasil["query"] = query
    hasil["peringkat"] = range(1, len(hasil) + 1)

    hasil_semua_expansion.append(hasil)

    print("\n" + "=" * 70)
    print("KUERI:", query)
    print("Kueri bersih:", query_bersih)
    print("Kueri diperluas:", query_diperluas)

    display(
        hasil[
            [
                "peringkat",
                "Place_Name",
                "Category",
                "City",
                "similarity_score",
                "Description"
            ]
        ]
    )

hasil_semua_expansion = pd.concat(
    hasil_semua_expansion,
    ignore_index=True
)

print("Total kandidat hasil eksperimen:",
      len(hasil_semua_expansion))



KUERI: tempat rekreasi keluarga di Bandung
Kueri bersih: rekreasi keluarga
Kueri diperluas: rekreasi keluarga keluarga wahana anak bermain rekreasi


,peringkat,Place_Name,Category,City,similarity_score,Description
290,1,Water Park Bandung Indah,Taman Hiburan,Bandung,0.128471,Bandung Indah Waterpark merupakan salah satu kolam renang yang ada di Bandung. Menyediakan berbagai wahana permainan air yang cocok bagi keluarga. Terdapat kolam renang untuk dewasa dan anak-anak. Pengunjung dapat melakukan olahraga renang dan rekreasi.
236,2,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,0.116306,"Panghegar Waterboom Bandung merupakan salah satu destinasi wisata yang wajib dikunjungi bagi wisatawan yang hendak mengajak buah hatinya bermain air. Terletak di Buah Batu, lokasi ini kerap kali diincar saat akhir pekan. Tak hanya menyuguhkan kolam renang, tempat ini juga menyediakan berbagai wahana menarik yang sayang jika dilewatkan. Terdiri dari seluncuran air yang dibagi untuk 2 kategori yaitu seluncuran orang dewasa dan seluncuran anak-anak. Destinasi rekreasi keluarga menawarkan beragam kegiatan seru, baik untuk anak-anak maupun dewasa. Terdapat beberapa jenis kolam yang bisa dicoba: kolam ombak, kolam mini, kolam arus, dan lain sebagainya. Ada pun yang paling baru di Panghegar Waterboom Bandung yaitu kolam air panas yang disajikan bagi wisatawan yang hendak berendam."
331,3,Rainbow Garden,Cagar Alam,Bandung,0.099748,"Rainbow Garden Harapan Indah salah satu taman rekreasi yang terutama cocok untuk keluarga. Berkonsep ‘eat and play’, Rainbow Garden Bekasi memadukan arena permainan anak dengan wisata kuliner. Berbagai wahana dan aktivitas tersedia di taman wisata penuh warna seluas 3000 m2 ini."
280,4,Tektona Waterpark,Taman Hiburan,Bandung,0.089931,"Tektona Waterpark sebuah wahana yang memberikan sensasi bermain air bersama keluarga dan sahabat anda, memberikan keceriaan bagi anak-anak maupun pengunjung dewasa dengan fasilitas modern diantaranya Kolam Ombak dan Kolam Arus, Kolam Anak dan Balita, Semi Olympic serta Mini Slider untuk anak-anak dan Mega Slider untuk dewasa dengan ketinggian hampir dari 30 meter, yang akan memicu adrenaline para pengunjung. Selain itu wahana Tektona Waterpark juga menyediakan pojok kuliner, gazebo serta jasa pompa dan foto aksi yang dapat diperoleh di sudut wahana Tektona Waterpark, dan juga selalu tersedia ruang ganti serta ruang bilas di dalam kawasan Tektona Waterpark."
266,5,Panama Park 825,Taman Hiburan,Bandung,0.070239,"Panama Park Bandung menjadi salah satu tempat bermain di Bandung, Jawa Barat. Wahana rekreasi yang disebut-sebut terbesar di Bandung ini dibuka sejak Mei 2019 lalu. Ada beberapa permainan yang ditawarkan di arena ini, misalnya playground hingga snow park yang berisi salju. Pengguna pun bisa merasakan sensasi salju sebenarnya."



KUERI: wisata alam di Semarang
Kueri bersih: wisata alam
Kueri diperluas: wisata alam alam hutan pegunungan air terjun pemandangan


,peringkat,Place_Name,Category,City,similarity_score,Description
385,1,Air Terjun Semirang,Cagar Alam,Semarang,0.181678,"Terletak di lereng Gunung Ungaran bagian utara, terdapat sebuah air terjun yang eksotis dengan kolam renang yang berwarna hijau yang dikelilingi popohonan dan kesejukan udara pegunungan. Nama lokasi tersebut adalah Air Terjun Semirang atau disebut juga Curug Semirang. Dan masyarakat Semarang menyebutnya dengan Curug Semarang. Karena keindahannya lokasi wisata ini sering dikunjungi oleh para wisatawan pada saat hari-hari biasa, apalagi saat liburan tiba. Air terjun Semirang atau Curug Semirang berada di kawasan hutan lindung dengan luas 10 hektar di bawah pengelolaan perum perhutani. Adapun air terjun ini memiliki ketinggian sekitar 45 meter. Jarak lokasi parkiran dengan lokasi air terjun sekitar 500 meter. Dan selama perjalanan, para pengunjung akan disuguhkan pemandangan alam yang masih asri dengan kesegaran udara yang sangat sejuk."
359,2,Air Terjun Kali Pancur,Cagar Alam,Semarang,0.170618,"Kali Pancur Getasan merupakan kawasan wisata alam Air Terjun Kali Pancur ini berada di Desa Nogosaren, Kecamatan Getasan, Kabupaten Semarang, sekitar 14 km sebelah barat Kota Salatiga. Untuk menuju ke lokasi objek wisata ini tidak ada sarana transportasi umum, jadi harus membawa kendaraan sendiri."
339,3,Desa Wisata Lembah Kalipancur,Taman Hiburan,Semarang,0.166099,"Wisata alam tengah menjadi sorotan bagi dunia pariwisata. Wisata alam tidak hanya pegunungan, danau, hutan, air terjun, atau pantai. Tapi juga merambah ke konsep pedesaan, seperti Desa Wisata Lembah Kalipancur. Tempat wisata ini menawarkan rekreasi berkonsep pedesaan yang ada di Kota Semarang. Meskipun berkonsep pedesaan, lokasinya tidak jauh dari pusat kota. Dan objek wisata yang sudah lama berdiri ini, masih tetap diminati oleh wisatawan."
354,4,Umbul Sidomukti,Cagar Alam,Semarang,0.088720,"Kawasan wisata umbul Sidomukti merupakan salah satu Wisata Alam Pegunungan di Semarang, berada di Desa Sidomukti Kecamatan Bandungan Kabupaten Semarang. Kawasan wisata ini dengan didukung fasiltas & Servis: Outbond Training, Adrenalin Games, Taman Renang Alam, Camping Ground, Pondok Wisata, Pondok Lesehan, serta Meeting Room."
356,5,Wisata Alam Wana Wisata Penggaron,Cagar Alam,Semarang,0.075525,"Berada sekitar 2 KM dari Kota Ungaran atau sekitar 18 KM dari Kota Semarang, Kawasan Wana Wisata Penggaron secara administratif berada di Desa Susukan, Kecamatan Ungaran, Kabupaten semarang. Area ini merupakan salah satu hutan binaan Kesatuan Bisnis Mandiri Wisata, Benih dan Usaha Lain (KBM WBU I) Perum Perhutani Unit I Jateng. Wana Wisata Penggaron atau Wisata Alam Wana Wisata Penggaron merupakan sebuah tempat wisata hutan yang bisa anda gunakan untuk berlibur bersama keluarga anda, salah satunya untuk camping. Tempat ini banyak di jadikan sebagai tempat untuk camping atau berlibur oleh masyarakat Semarang. Wana Wisata Penggaron sebuah hutan yang eksotis, sayangnya surga di kota semarang ini jarang tersentuh mata. Bagi pengunjung yang bertanya kepada penjaga loket, dengan jawaban yang sederhana dan apa adanya “hutan biasa yang sering dipake kemah, tidak ada yang istimewa”, tentunya akan merasa kecewa."



KUERI: budaya di Bandung
Kueri bersih: budaya
Kueri diperluas: budaya budaya museum sejarah seni tradisi monumen


,peringkat,Place_Name,Category,City,similarity_score,Description
258,1,Monumen Perjuangan Rakyat Jawa Barat,Budaya,Bandung,0.162920,"Monumen Perjuangan Rakyat Jawa Barat (Monju) adalah Museum Sejarah Perjuangan Rakyat Jawa Barat, di Tatar Pasundan atau Parahyangan. Monumen diresmikan oleh Gubernur Jawa Barat, Raden Nana Nuriana pada tanggal 23 Agustus 1995."
254,2,Museum Barli,Budaya,Bandung,0.098355,"Museum Barli adalah sebuah museum yang terletak di Kota Bandung. Museum ini didedikasikan untuk mengenang dan memamerkan hasil karya pelukis Barli Sasmitawinata. Museum ini didirikan pada tahun 1990 dan diresmikan pada Oktober 1992 oleh Soesilo Soedarman, Menteri Pariwisata, Pos, dan Telekomunikasi pada waktu itu. Awalnya museum ini tidak dibuka untuk umum. Namun, saat ini museum dapat dikunjungi oleh siapa saja. Museum ini juga menjadi tempat dari berbagai kegiatan seni seperti pameran karya seni rupa, penjualan karya, diskusi dan sarasehan, pelatihan studio keramik dan lukis."
271,3,Taman Budaya Jawa Barat,Budaya,Bandung,0.078147,"Taman Budaya Jawa Barat (TBJB) didirikan pada tahun 1991 di Kawasan Dago Tea House. TBJB memiliki fasilitas gedung teater tertutup, teater terbuka, ruang pameran, sekertariat, sanggar olah seni, wisma seni, dan cafetaria. Semua fasilitas tersebut diperuntunkan untuk kelancaran beraktifitas dan berkarya para warga Bandung, khususnya para seniman. TBJB menghelat berbagai macam agenda yang menarik seperti pementasan karya seni, lomba seni, eksperimentasi karya seni, seminar, workshop, sarasehan seni dan budaya, penerbitan buku, dsb."
220,4,Museum Geologi Bandung,Budaya,Bandung,0.076966,"Museum Geologi didirikan pada tanggal 16 Mei 1929 Museum ini telah direnovasi dengan dana bantuan dari JICA (Japan International Cooperation Agency). Setelah mengalami renovasi, Museum Geologi dibuka kembali dan diresmikan oleh Wakil Presiden RI, Megawati Soekarnoputri pada tanggal 23 Agustus 2000. Sebagai salah satu monumen bersejarah, museum berada di bawah perlindungan pemerintah dan merupakan peninggalan nasional. Dalam Museum ini, tersimpan dan dikelola materi-materi geologi yang berlimpah, seperti fosil, batuan, mineral. Kesemuanya itu dikumpulkan selama kerja lapangan di Indonesia sejak 1850."
256,5,Monumen Bandung Lautan Api,Budaya,Bandung,0.069277,"Monumen Bandung Lautan Api, merupakan monumen yang menjadi markah tanah Bandung. Monumen ini setinggi 45 meter, memiliki sisi sebanyak 9 bidang.\nMonumen ini dibangun untuk memperingati peristiwa Bandung Lautan Api, dimana terjadi pembumihangusan Bandung Selatan yang dipimpin oleh Muhammad Toha.\nMonumen ini berada di tengah-tengah kota yaitu terletak di kawasan Lapangan Tegallega. Monumen ini menjadi salah satu monumen terkenal di Bandung.\nMonumen ini menjadi pusat perhatian setiap tanggal 23 Maret mengenang peristiwa Bandung Lautan Api."


Total kandidat hasil eksperimen: 15


In [123]:

# Label relevansi manual berdasarkan deskripsi yang diperiksa.
# 1 = relevan, 0 = tidak relevan.

label_manual = {
    "tempat rekreasi keluarga di Bandung": {
        "Water Park Bandung Indah": 1,
        "Panghegar Waterboom Bandung": 1,
        "Rainbow Garden": 0,  # lokasi perlu diverifikasi
        "Tektona Waterpark": 1,
        "Panama Park 825": 1,
    },
    "wisata alam di Semarang": {
        "Air Terjun Semirang": 1,
        "Air Terjun Kali Pancur": 1,
        "Desa Wisata Lembah Kalipancur": 1,
        "Umbul Sidomukti": 1,
        "Wisata Alam Wana Wisata Penggaron": 1,
    },
    "budaya di Bandung": {
        "Monumen Perjuangan Rakyat Jawa Barat": 1,
        "Museum Barli": 1,
        "Taman Budaya Jawa Barat": 1,
        "Museum Geologi Bandung": 1,
        "Monumen Bandung Lautan Api": 1,
    },
}

evaluasi_expansion = hasil_semua_expansion.copy()

evaluasi_expansion["relevant"] = evaluasi_expansion.apply(
    lambda baris: label_manual[baris["query"]][baris["Place_Name"]],
    axis=1
)

ringkasan_expansion = (
    evaluasi_expansion.groupby("query")
    .agg(
        jumlah_hasil=("Place_Name", "count"),
        jumlah_relevan=("relevant", "sum"),
    )
    .reset_index()
)

ringkasan_expansion["Precision@5"] = (
    ringkasan_expansion["jumlah_relevan"]
    / 5
)

display(ringkasan_expansion)

macro_precision = ringkasan_expansion["Precision@5"].mean()
print("Macro-average Precision@5:", round(macro_precision, 3))


,query,jumlah_hasil,jumlah_relevan,Precision@5
0,budaya di Bandung,5,5,1.0
1,tempat rekreasi keluarga di Bandung,5,4,0.8
2,wisata alam di Semarang,5,5,1.0


Macro-average Precision@5: 0.933


In [126]:

kueri_uji = [
    "tempat rekreasi keluarga di Bandung",
    "wisata alam di Semarang",
    "budaya di Bandung",
]

hasil_baseline_baru = []

for query in kueri_uji:
    print("\n" + "=" * 60)
    print("KUERI:", query)

    hasil = cari_destinasi_v3(query, top_k=5).copy()

    # Tambahkan Description dari dataset utama jika belum tersedia
    if "Description" not in hasil.columns:
        hasil = hasil.merge(
            df[["Place_Name", "Description"]].drop_duplicates(
                subset=["Place_Name"]
            ),
            on="Place_Name",
            how="left"
        )

    # Pastikan nama kolom hasil pencarian
    print("Kolom tersedia:", hasil.columns.tolist())

    kolom_tampil = [
        kolom for kolom in [
            "Place_Name",
            "Category",
            "City",
            "similarity_score",
            "Description"
        ]
        if kolom in hasil.columns
    ]

    display(hasil[kolom_tampil])

    hasil["query"] = query
    hasil["peringkat"] = range(1, len(hasil) + 1)
    hasil_baseline_baru.append(hasil)

if hasil_baseline_baru:
    hasil_baseline_baru = pd.concat(
        hasil_baseline_baru,
        ignore_index=True
    )
    print("\nTotal hasil baseline:", len(hasil_baseline_baru))



KUERI: tempat rekreasi keluarga di Bandung
Kota terdeteksi: Bandung
Kategori terdeteksi: Taman Hiburan
Batas harga: Tidak ada
Kolom tersedia: ['Place_Name', 'Category', 'City', 'Price', 'Rating', 'similarity_score', 'Description']


,Place_Name,Category,City,similarity_score,Description
0,Taman Jomblo,Taman Hiburan,Bandung,0.1120,"Taman Jomblo dapat mengacu kepada:\n\nTaman Pasupati, sebuah taman yang didirikan oleh Ridwan Kamil di Bandung\nPedestrian Jomblo, sebuah tempat rekreasi yang didirikan di Jambi\nTaman Jomblo (Mataram), sebuah taman yang didirikan di Mataram, Nusa Tenggara Barat"
1,Water Park Bandung Indah,Taman Hiburan,Bandung,0.1032,Bandung Indah Waterpark merupakan salah satu kolam renang yang ada di Bandung. Menyediakan berbagai wahana permainan air yang cocok bagi keluarga. Terdapat kolam renang untuk dewasa dan anak-anak. Pengunjung dapat melakukan olahraga renang dan rekreasi.
2,Panama Park 825,Taman Hiburan,Bandung,0.0899,"Panama Park Bandung menjadi salah satu tempat bermain di Bandung, Jawa Barat. Wahana rekreasi yang disebut-sebut terbesar di Bandung ini dibuka sejak Mei 2019 lalu. Ada beberapa permainan yang ditawarkan di arena ini, misalnya playground hingga snow park yang berisi salju. Pengguna pun bisa merasakan sensasi salju sebenarnya."
3,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,0.0731,"Panghegar Waterboom Bandung merupakan salah satu destinasi wisata yang wajib dikunjungi bagi wisatawan yang hendak mengajak buah hatinya bermain air. Terletak di Buah Batu, lokasi ini kerap kali diincar saat akhir pekan. Tak hanya menyuguhkan kolam renang, tempat ini juga menyediakan berbagai wahana menarik yang sayang jika dilewatkan. Terdiri dari seluncuran air yang dibagi untuk 2 kategori yaitu seluncuran orang dewasa dan seluncuran anak-anak. Destinasi rekreasi keluarga menawarkan beragam kegiatan seru, baik untuk anak-anak maupun dewasa. Terdapat beberapa jenis kolam yang bisa dicoba: kolam ombak, kolam mini, kolam arus, dan lain sebagainya. Ada pun yang paling baru di Panghegar Waterboom Bandung yaitu kolam air panas yang disajikan bagi wisatawan yang hendak berendam."
4,Teras Cikapundung BBWS,Taman Hiburan,Bandung,0.0548,"Teras Cikapundung Bandung sebelumnya merupakan daerah yang kumuh dan tak terurus yang berhasil diubah menjadi tempat wisata. Cikapundung adalah sungai yang membelah Kota Bandung yang dimulai dari daerah Bandung Utara dan bermuara di Sungai Citarum, Bandung Selatan. Sebelumnya, taman ini merupakan tempat berkumpulnya orang-orang kreatif seperti para seniman yang ada di Bandung. Terdapat beberapa sanggar seni dan ramainya warung makan di tempat ini, sampai pada akhirnya terlantar dan dibiarkan kotor. Kini Teras Cikapundung Bandung telah berubah menjadi tempat wisata terbuka untuk umum dengan konsep ekologi dan urban. Berbagai fasilitas menarik tersedia di Teras Cikapundung dan bisa digunakan pada saat rekreasi ke obyek wisata yang satu ini. Dilengkapi dengan kolam ikan dan lampu-lampu, membuat taman ini sangat cantik dikunjungi pada malam hari. Terlebih suasananya yang ramai membuat para pengunjung lupa bahwa dulunya tempat ini pernah dijuluki sebagai “tempat jin buang anak”."



KUERI: wisata alam di Semarang
Kota terdeteksi: Semarang
Kategori terdeteksi: Cagar Alam
Batas harga: Tidak ada
Kolom tersedia: ['Place_Name', 'Category', 'City', 'Price', 'Rating', 'similarity_score', 'Description']


,Place_Name,Category,City,similarity_score,Description
0,Umbul Sidomukti,Cagar Alam,Semarang,0.1521,"Kawasan wisata umbul Sidomukti merupakan salah satu Wisata Alam Pegunungan di Semarang, berada di Desa Sidomukti Kecamatan Bandungan Kabupaten Semarang. Kawasan wisata ini dengan didukung fasiltas & Servis: Outbond Training, Adrenalin Games, Taman Renang Alam, Camping Ground, Pondok Wisata, Pondok Lesehan, serta Meeting Room."
1,Brown Canyon,Cagar Alam,Semarang,0.1060,"Di Indonesia, Brown Canyon adalah sebuah bekas penambangan tanah di Meteseh, Tembalang, Semarang. Lokasi ini menjadi salah satu lokasi (bukan tempat wisata) yang populer di Semarang dan dianggap mirip dengan Grand Canyon di Amerika Serikat. Kemiripan ini terletak pada bukit-bukit yang berubah menjadi tebing-tebing curam."
2,Danau Rawa Pening,Cagar Alam,Semarang,0.0982,"Rawa Pening (pening adalah salah satu varian bahasa Jawa dari kata ""wening"" yang artinya hening, tenang, damai) (bahasa Jawa: ꦫꦮ​ꦥꦼꦤꦶꦁ, translit. Rawa Pening) adalah danau alam di Kabupaten Semarang, Jawa Tengah. Dengan luas 2.670 hektare ia menempati wilayah Kecamatan Ambarawa, Bawen, Tuntang, dan Banyubiru.Rawa Pening terletak di cekungan terendah lereng Gunung Merbabu, Gunung Telomoyo, dan Gunung Ungaran. Danau ini dangkal dan menjadi hulu bagi Sungai Tuntang."
3,Air Terjun Kali Pancur,Cagar Alam,Semarang,0.0929,"Kali Pancur Getasan merupakan kawasan wisata alam Air Terjun Kali Pancur ini berada di Desa Nogosaren, Kecamatan Getasan, Kabupaten Semarang, sekitar 14 km sebelah barat Kota Salatiga. Untuk menuju ke lokasi objek wisata ini tidak ada sarana transportasi umum, jadi harus membawa kendaraan sendiri."
4,Watu Gunung Ungaran,Cagar Alam,Semarang,0.0919,"Objek wisata alam memang sangat cocok dikunjungi, terlebih saat akhir pekan. Ditambah lagi, lokasi objek wisata yang tidak terlalu jauh dari pusat kota. Seperti objek wisata Watu Gunung yang ada di Ungaran, Kota Semarang. Berkunjung ke wisata alam dapat membuat pikiran menjadi segar. Terlebih, objek wisata ini menawarkan wisata alam dan air yang menawan. Selain itu, dipadukan juga dengan nuansa adat Jawa yang khas. Dulunya, kawasan Watu Gunung ini bukanlah suatu objek wisata. Hanya tanah yang ditumbuhi pohon pala dan cengkeh yang lebat. Sayangnya, kawasan tersebut gersang dan susah mendapatkan air. Kemudian, di tahun 2004 sang pemilik berinisiatif untuk mendirikan villa pribadi. Dan lama kelamaan, dibuatlah objek wisata yang dibuka untuk umum. Dengan tetap mempertahankan bangunan joglo sebagai ciri khasnya."



KUERI: budaya di Bandung
Kota terdeteksi: Bandung
Kategori terdeteksi: Budaya
Batas harga: Tidak ada
Kolom tersedia: ['Place_Name', 'Category', 'City', 'Price', 'Rating', 'similarity_score', 'Description']


,Place_Name,Category,City,similarity_score,Description
0,Museum Pendidikan Nasional,Budaya,Bandung,0.0768,"Museum Pendidikan Nasional Universitas Pendidikan Indonesia diresmikan oleh Gubernur Jawa Barat, H. Ahmad Heryawan, Lc. Pembangunan Museum Pendidikan Nasional ini merupakan salah satu bentuk tanggung jawab UPI sebagai perguruan tinggi yang memiliki kepedulian terhadap kelestarian warisan sejarah budaya bangsa khususnya dibidang pendidikan. Museum pada umumnya memamerkan benda koleksi yang berasal dari masa lalu, namun hal tersebut berbeda dengan Museum Pendidikan Nasional Universitas Pendidikan Indonesia. Museum ini menyajikan bagaimana pendidikan dimasa yang akan datang, karena saat ini pendidikan menjadi bersifat forecasting atau ramalan. Keberadaan Museum Pendidikan Nasional diharapkan mampu menjadi wahana pusat penelitian, serta sumber belajar dan pembelajaran, yang dapat meningkatkan kompetensi pendidik maupun wawasan pembelajar dalam mengupayakan peningkatan mutu pendidikan, sekaligus menjadi tujuan wisata budaya di Jawa Barat dan Nasional. Karena pada dasarnya manusia tidak mempelajari masa lalu, melainkan masa depan."
1,Monumen Bandung Lautan Api,Budaya,Bandung,0.0726,"Monumen Bandung Lautan Api, merupakan monumen yang menjadi markah tanah Bandung. Monumen ini setinggi 45 meter, memiliki sisi sebanyak 9 bidang.\nMonumen ini dibangun untuk memperingati peristiwa Bandung Lautan Api, dimana terjadi pembumihangusan Bandung Selatan yang dipimpin oleh Muhammad Toha.\nMonumen ini berada di tengah-tengah kota yaitu terletak di kawasan Lapangan Tegallega. Monumen ini menjadi salah satu monumen terkenal di Bandung.\nMonumen ini menjadi pusat perhatian setiap tanggal 23 Maret mengenang peristiwa Bandung Lautan Api."
2,Amazing Art World,Budaya,Bandung,0.0665,"Amazing Art World Bandung, sebuah objek wisata yang terletak di Bandung, yang menegaskan bahwa destinasi wisata itu bukan hanya sekedar tentang pantai, air terjun, atau wisata alam lainnya saja. Hasil karya lukisanpun ternyata mampu menjadi sebuah destinasi wisata yang sangat menyenangkan. Apalagi lukisan 3D, selain keren, tetapi mampu membuat manipulasi bagi mata para pengunjung. Luasnya Amazing Art World Bandung menjadikannya sebagai salah-satu museum lukisan 3D terbesar, bukan hanya di tingkat nasional, tetapi di dunia."
3,Taman Sejarah Bandung,Budaya,Bandung,0.0586,"Taman Sejarah Bandung adalah taman yang terletak di belakang Balai Kota Bandung, kompleks Pemerintahan Kota Bandung. Taman ini berfungsi untuk mengenalkan sejarah di Kota Bandung. Taman ini dilengkapi informasi mengenai pemimpin kota Bandung dari masa ke masa. Pembangunan taman ini menghabiskan anggaran sekitar 3,5 miliar rupiah."
4,Bandros City Tour,Budaya,Bandung,0.0569,"Bandros atau Bus Wisata Bandung adalah bus wisata yang beroperasi di Bandung, Jawa Barat, Indonesia.Digunakan oleh wisatawan di Kota Bandung. Bus tersebut diresmikan oleh Wali Kota Bandung Ridwan Kamil pada malam tahun baru 2014, dan pada 18 Februari 2018 diumumkan bus wisata akan melayani kota Bandung. Seorang pemimpin tur lokal akan menemani turis di dalam bus. Bandros dikelola oleh dua pihak: Dinas Perhubungan Bandung (yang terkait dengan Anggaran Pendapatan dan Belanja Daerah) dan Mang Dudung yang merupakan bagian dari Tanggung Jawab Sosial Perusahaan. Bus memiliki kapasitas 22 orang. Ada enam bus yang dioperasikan oleh divisi Corporate Social Responsibility, dan 12 bus dioperasikan oleh Anggaran Pendapatan dan Belanja Daerah yang dikelola oleh BLUD UPT Dinas Perhubungan Kota Bandung."



Total hasil baseline: 15


In [127]:

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

skenario_uji = [
    {
        "query": "tempat rekreasi keluarga di Bandung",
        "kota": "Bandung",
    },
    {
        "query": "wisata alam di Semarang",
        "kota": "Semarang",
    },
    {
        "query": "budaya di Bandung",
        "kota": "Bandung",
    },
]

hasil_baseline_adil = []

for skenario in skenario_uji:
    query = skenario["query"]
    kota = skenario["kota"]

    query_bersih = bersihkan_kueri(query, kota=kota)

    # Baseline: tidak menambahkan kata ekspansi
    vektor_query = vectorizer_v2.transform([query_bersih])
    skor = cosine_similarity(
        vektor_query, tfidf_matrix_v2
    ).flatten()

    hasil = df.copy()
    hasil["similarity_score"] = skor

    # Kondisi filter harus sama dengan eksperimen expansion
    hasil = hasil[
        (hasil["City"].eq(kota))
        & (hasil["similarity_score"] > 0)
    ].copy()

    hasil = hasil.sort_values(
        "similarity_score", ascending=False
    ).head(5).copy()

    hasil["query"] = query
    hasil["peringkat"] = range(1, len(hasil) + 1)

    hasil_baseline_adil.append(hasil)

    print("\n" + "=" * 60)
    print("KUERI:", query)
    print("Kueri baseline:", query_bersih)

    display(
        hasil[
            [
                "peringkat",
                "Place_Name",
                "Category",
                "City",
                "similarity_score",
                "Description",
            ]
        ]
    )

hasil_baseline_adil = pd.concat(
    hasil_baseline_adil,
    ignore_index=True
)

print("\nTotal kandidat baseline:", len(hasil_baseline_adil))



KUERI: tempat rekreasi keluarga di Bandung
Kueri baseline: rekreasi keluarga


,peringkat,Place_Name,Category,City,similarity_score,Description
236,1,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,0.089849,"Panghegar Waterboom Bandung merupakan salah satu destinasi wisata yang wajib dikunjungi bagi wisatawan yang hendak mengajak buah hatinya bermain air. Terletak di Buah Batu, lokasi ini kerap kali diincar saat akhir pekan. Tak hanya menyuguhkan kolam renang, tempat ini juga menyediakan berbagai wahana menarik yang sayang jika dilewatkan. Terdiri dari seluncuran air yang dibagi untuk 2 kategori yaitu seluncuran orang dewasa dan seluncuran anak-anak. Destinasi rekreasi keluarga menawarkan beragam kegiatan seru, baik untuk anak-anak maupun dewasa. Terdapat beberapa jenis kolam yang bisa dicoba: kolam ombak, kolam mini, kolam arus, dan lain sebagainya. Ada pun yang paling baru di Panghegar Waterboom Bandung yaitu kolam air panas yang disajikan bagi wisatawan yang hendak berendam."
290,2,Water Park Bandung Indah,Taman Hiburan,Bandung,0.079010,Bandung Indah Waterpark merupakan salah satu kolam renang yang ada di Bandung. Menyediakan berbagai wahana permainan air yang cocok bagi keluarga. Terdapat kolam renang untuk dewasa dan anak-anak. Pengunjung dapat melakukan olahraga renang dan rekreasi.
331,3,Rainbow Garden,Cagar Alam,Bandung,0.068628,"Rainbow Garden Harapan Indah salah satu taman rekreasi yang terutama cocok untuk keluarga. Berkonsep ‘eat and play’, Rainbow Garden Bekasi memadukan arena permainan anak dengan wisata kuliner. Berbagai wahana dan aktivitas tersedia di taman wisata penuh warna seluas 3000 m2 ini."
313,4,Tafso Barn,Cagar Alam,Bandung,0.044439,"Nama Punclut mungkin sudah cukup akrab di telinga wisatawan. Kawasan dataran tinggi di Bandung yang belakangan populer sebagai destinasi wisata. Selain menawarkan pemandangan yang indah, kawasan ini juga memiliki banyak tempat wisata untuk rekreasi. Salah satunya adalah kawasan wisata sekaligus restoran Tafso dan Boda Barn atau sering dikenal juga dengan Tafso Barn. Tempat rekreasi ini memadukan tempat makan dengan nuansa taman dan gudang yang unik. Pemandangan perbukitan yang dipotong garis horison menjadi incaran utama pengunjung. Sangat tepat nongkrong dan bersantai ketika berlibur di Bandung."
219,5,Dusun Bambu,Taman Hiburan,Bandung,0.035472,"Dusun Bambu merupakan taman rekreasi berbasis ecotourism di Jawa Barat dengan konsep ‘Private Sanctuary Lifestyle’ (Konsep Wisata Sehat). Melalui konsep ini, Dusun Bambu siap menghadirkan sarana rekreasi yang aman dan nyaman dengan senantiasa menerapkan standar protokol kesehatan yang ketat. Memadukan keindahan alam dan budaya tradisional, fasilitas di Dusun Bambu didesain untuk pengalaman wisata alam yang paling berkesan, mulai dari berjalan-jalan santai, bersepeda, menikmati kuliner lezat di tengah pesona alam, hingga mengunjungi replika pedesaan tradisional. Terdapat pula kapasitas kunjungan harian ke Dusun Bambu untuk memastikan bahwa anjuran social distancing tetap dijalankan. Dengan begitu, Dusun Bambu siap menjamin keselamatan kunjungan wisata Anda menyajikan pengalaman kunjungan yang tak tergantikan dan penjelajahan rasa yang memuaskan hati."



KUERI: wisata alam di Semarang
Kueri baseline: wisata alam


,peringkat,Place_Name,Category,City,similarity_score,Description
339,1,Desa Wisata Lembah Kalipancur,Taman Hiburan,Semarang,0.147602,"Wisata alam tengah menjadi sorotan bagi dunia pariwisata. Wisata alam tidak hanya pegunungan, danau, hutan, air terjun, atau pantai. Tapi juga merambah ke konsep pedesaan, seperti Desa Wisata Lembah Kalipancur. Tempat wisata ini menawarkan rekreasi berkonsep pedesaan yang ada di Kota Semarang. Meskipun berkonsep pedesaan, lokasinya tidak jauh dari pusat kota. Dan objek wisata yang sudah lama berdiri ini, masih tetap diminati oleh wisatawan."
387,2,Watu Gunung Ungaran,Cagar Alam,Semarang,0.136283,"Objek wisata alam memang sangat cocok dikunjungi, terlebih saat akhir pekan. Ditambah lagi, lokasi objek wisata yang tidak terlalu jauh dari pusat kota. Seperti objek wisata Watu Gunung yang ada di Ungaran, Kota Semarang. Berkunjung ke wisata alam dapat membuat pikiran menjadi segar. Terlebih, objek wisata ini menawarkan wisata alam dan air yang menawan. Selain itu, dipadukan juga dengan nuansa adat Jawa yang khas. Dulunya, kawasan Watu Gunung ini bukanlah suatu objek wisata. Hanya tanah yang ditumbuhi pohon pala dan cengkeh yang lebat. Sayangnya, kawasan tersebut gersang dan susah mendapatkan air. Kemudian, di tahun 2004 sang pemilik berinisiatif untuk mendirikan villa pribadi. Dan lama kelamaan, dibuatlah objek wisata yang dibuka untuk umum. Dengan tetap mempertahankan bangunan joglo sebagai ciri khasnya."
354,3,Umbul Sidomukti,Cagar Alam,Semarang,0.130445,"Kawasan wisata umbul Sidomukti merupakan salah satu Wisata Alam Pegunungan di Semarang, berada di Desa Sidomukti Kecamatan Bandungan Kabupaten Semarang. Kawasan wisata ini dengan didukung fasiltas & Servis: Outbond Training, Adrenalin Games, Taman Renang Alam, Camping Ground, Pondok Wisata, Pondok Lesehan, serta Meeting Room."
359,4,Air Terjun Kali Pancur,Cagar Alam,Semarang,0.115640,"Kali Pancur Getasan merupakan kawasan wisata alam Air Terjun Kali Pancur ini berada di Desa Nogosaren, Kecamatan Getasan, Kabupaten Semarang, sekitar 14 km sebelah barat Kota Salatiga. Untuk menuju ke lokasi objek wisata ini tidak ada sarana transportasi umum, jadi harus membawa kendaraan sendiri."
356,5,Wisata Alam Wana Wisata Penggaron,Cagar Alam,Semarang,0.105413,"Berada sekitar 2 KM dari Kota Ungaran atau sekitar 18 KM dari Kota Semarang, Kawasan Wana Wisata Penggaron secara administratif berada di Desa Susukan, Kecamatan Ungaran, Kabupaten semarang. Area ini merupakan salah satu hutan binaan Kesatuan Bisnis Mandiri Wisata, Benih dan Usaha Lain (KBM WBU I) Perum Perhutani Unit I Jateng. Wana Wisata Penggaron atau Wisata Alam Wana Wisata Penggaron merupakan sebuah tempat wisata hutan yang bisa anda gunakan untuk berlibur bersama keluarga anda, salah satunya untuk camping. Tempat ini banyak di jadikan sebagai tempat untuk camping atau berlibur oleh masyarakat Semarang. Wana Wisata Penggaron sebuah hutan yang eksotis, sayangnya surga di kota semarang ini jarang tersentuh mata. Bagi pengunjung yang bertanya kepada penjaga loket, dengan jawaban yang sederhana dan apa adanya “hutan biasa yang sering dipake kemah, tidak ada yang istimewa”, tentunya akan merasa kecewa."



KUERI: budaya di Bandung
Kueri baseline: budaya


,peringkat,Place_Name,Category,City,similarity_score,Description
271,1,Taman Budaya Jawa Barat,Budaya,Bandung,0.078045,"Taman Budaya Jawa Barat (TBJB) didirikan pada tahun 1991 di Kawasan Dago Tea House. TBJB memiliki fasilitas gedung teater tertutup, teater terbuka, ruang pameran, sekertariat, sanggar olah seni, wisma seni, dan cafetaria. Semua fasilitas tersebut diperuntunkan untuk kelancaran beraktifitas dan berkarya para warga Bandung, khususnya para seniman. TBJB menghelat berbagai macam agenda yang menarik seperti pementasan karya seni, lomba seni, eksperimentasi karya seni, seminar, workshop, sarasehan seni dan budaya, penerbitan buku, dsb."
285,2,Roemah Seni Sarasvati,Budaya,Bandung,0.059253,"Roemah Seni Sarasvati adalah sebuah galeri yang berlokasi di Jl. Jendral Sudirman 137, Bandung. RSS merupakan project utama Sarasvati Arts Management yang dibangun dengan tujuan untuk memperkenalkan berbagai keragaman seni budaya. RSS juga merupakan tempat bertukar gagasan dan kepedulian untuk menciptakan identitas unik bagi Kota Bandung serta merawat kreativitas dan pengetahuan. Salah satu program kerja yang sedang dirancang adalah Grote Postweg Art Project."
289,3,Museum Pendidikan Nasional,Budaya,Bandung,0.053799,"Museum Pendidikan Nasional Universitas Pendidikan Indonesia diresmikan oleh Gubernur Jawa Barat, H. Ahmad Heryawan, Lc. Pembangunan Museum Pendidikan Nasional ini merupakan salah satu bentuk tanggung jawab UPI sebagai perguruan tinggi yang memiliki kepedulian terhadap kelestarian warisan sejarah budaya bangsa khususnya dibidang pendidikan. Museum pada umumnya memamerkan benda koleksi yang berasal dari masa lalu, namun hal tersebut berbeda dengan Museum Pendidikan Nasional Universitas Pendidikan Indonesia. Museum ini menyajikan bagaimana pendidikan dimasa yang akan datang, karena saat ini pendidikan menjadi bersifat forecasting atau ramalan. Keberadaan Museum Pendidikan Nasional diharapkan mampu menjadi wahana pusat penelitian, serta sumber belajar dan pembelajaran, yang dapat meningkatkan kompetensi pendidik maupun wawasan pembelajar dalam mengupayakan peningkatan mutu pendidikan, sekaligus menjadi tujuan wisata budaya di Jawa Barat dan Nasional. Karena pada dasarnya manusia tidak mempelajari masa lalu, melainkan masa depan."
258,4,Monumen Perjuangan Rakyat Jawa Barat,Budaya,Bandung,0.047905,"Monumen Perjuangan Rakyat Jawa Barat (Monju) adalah Museum Sejarah Perjuangan Rakyat Jawa Barat, di Tatar Pasundan atau Parahyangan. Monumen diresmikan oleh Gubernur Jawa Barat, Raden Nana Nuriana pada tanggal 23 Agustus 1995."
284,5,Taman Sejarah Bandung,Budaya,Bandung,0.047114,"Taman Sejarah Bandung adalah taman yang terletak di belakang Balai Kota Bandung, kompleks Pemerintahan Kota Bandung. Taman ini berfungsi untuk mengenalkan sejarah di Kota Bandung. Taman ini dilengkapi informasi mengenai pemimpin kota Bandung dari masa ke masa. Pembangunan taman ini menghabiskan anggaran sekitar 3,5 miliar rupiah."



Total kandidat baseline: 15


In [128]:

# Label relevansi baseline berdasarkan deskripsi destinasi.
# Gunakan kriteria yang sama dengan eksperimen query expansion.

label_baseline_manual = {
    "tempat rekreasi keluarga di Bandung": {
        "Panghegar Waterboom Bandung": 1,
        "Water Park Bandung Indah": 1,
        "Rainbow Garden": 0,  # lokasi perlu diverifikasi
        "Tafso Barn": 1,
        "Dusun Bambu": 1,
    },
    "wisata alam di Semarang": {
        "Desa Wisata Lembah Kalipancur": 1,
        "Watu Gunung Ungaran": 1,
        "Umbul Sidomukti": 1,
        "Air Terjun Kali Pancur": 1,
        "Wisata Alam Wana Wisata Penggaron": 1,
    },
    "budaya di Bandung": {
        "Taman Budaya Jawa Barat": 1,
        "Roemah Seni Sarasvati": 1,
        "Museum Pendidikan Nasional": 1,
        "Monumen Perjuangan Rakyat Jawa Barat": 1,
        "Taman Sejarah Bandung": 1,
    },
}

baseline_dinilai = hasil_baseline_adil.copy()

baseline_dinilai["relevant"] = baseline_dinilai.apply(
    lambda baris: label_baseline_manual[
        baris["query"]
    ][baris["Place_Name"]],
    axis=1
)

evaluasi_baseline_baru = (
    baseline_dinilai.groupby("query")
    .agg(
        jumlah_hasil=("Place_Name", "count"),
        jumlah_relevan=("relevant", "sum"),
    )
    .reset_index()
)

evaluasi_baseline_baru["Precision@5"] = (
    evaluasi_baseline_baru["jumlah_relevan"] / 5
)

display(evaluasi_baseline_baru)

macro_baseline_baru = (
    evaluasi_baseline_baru["Precision@5"].mean()
)

print(
    "Macro-average Precision@5 baseline:",
    round(macro_baseline_baru, 3)
)


,query,jumlah_hasil,jumlah_relevan,Precision@5
0,budaya di Bandung,5,5,1.0
1,tempat rekreasi keluarga di Bandung,5,4,0.8
2,wisata alam di Semarang,5,5,1.0


Macro-average Precision@5 baseline: 0.933


In [129]:

# Rekap hasil evaluasi tiga kueri yang sudah diuji
perbandingan_tiga_kueri = evaluasi_baseline_baru[
    ["query", "Precision@5"]
].merge(
    ringkasan_expansion[["query", "Precision@5"]],
    on="query",
    suffixes=("_baseline", "_expansion")
)

perbandingan_tiga_kueri["selisih"] = (
    perbandingan_tiga_kueri["Precision@5_expansion"]
    - perbandingan_tiga_kueri["Precision@5_baseline"]
)

display(perbandingan_tiga_kueri)

print(
    "Macro-average baseline:",
    round(perbandingan_tiga_kueri["Precision@5_baseline"].mean(), 3)
)

print(
    "Macro-average expansion:",
    round(perbandingan_tiga_kueri["Precision@5_expansion"].mean(), 3)
)


,query,Precision@5_baseline,Precision@5_expansion,selisih
0,budaya di Bandung,1.0,1.0,0.0
1,tempat rekreasi keluarga di Bandung,0.8,0.8,0.0
2,wisata alam di Semarang,1.0,1.0,0.0


Macro-average baseline: 0.933
Macro-average expansion: 0.933


In [131]:

from sklearn.metrics.pairwise import cosine_similarity

def cari_eksperimen(teks_kueri, kota, top_k=5):
    vektor = vectorizer_v2.transform([teks_kueri])
    skor = cosine_similarity(
        vektor, tfidf_matrix_v2
    ).flatten()

    hasil = df.copy()
    hasil["similarity_score"] = skor

    hasil = hasil[
        (hasil["City"].eq(kota)) &
        (hasil["similarity_score"] > 0)
    ].copy()

    hasil = hasil.sort_values(
        "similarity_score",
        ascending=False
    ).head(top_k)

    kolom = [
        "Place_Name", "Category", "City",
        "Price", "Rating", "similarity_score"
    ]

    return hasil[kolom].reset_index(drop=True)

print("Fungsi cari_eksperimen berhasil dibuat.")


Fungsi cari_eksperimen berhasil dibuat.


In [132]:

query = "taman bermain di Bandung"
kota = "Bandung"
tambahan = "taman bermain playground wahana anak rekreasi keluarga"

query_bersih = bersihkan_kueri(query, kota=kota)

hasil_baseline_taman = cari_eksperimen(
    query_bersih, kota, top_k=5
)

hasil_expansion_taman = cari_eksperimen(
    f"{query_bersih} {tambahan}", kota, top_k=5
)

print("=== BASELINE ===")
display(hasil_baseline_taman)

print("=== QUERY EXPANSION ===")
display(hasil_expansion_taman)


=== BASELINE ===


,Place_Name,Category,City,Price,Rating,similarity_score
0,Trans Studio Bandung,Taman Hiburan,Bandung,280000,4.5,0.110372
1,Taman Jomblo,Taman Hiburan,Bandung,10000,4.1,0.053001
2,Panama Park 825,Taman Hiburan,Bandung,50000,4.4,0.050225
3,Taman Lalu Lintas Ade Irma Suryani Nasution,Taman Hiburan,Bandung,7000,4.4,0.049727
4,Happyfarm Ciwidey,Cagar Alam,Bandung,15000,4.2,0.035647


=== QUERY EXPANSION ===


,Place_Name,Category,City,Price,Rating,similarity_score
0,Panama Park 825,Taman Hiburan,Bandung,50000,4.4,0.111764
1,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,75000,4.3,0.097271
2,Trans Studio Bandung,Taman Hiburan,Bandung,280000,4.5,0.094971
3,Water Park Bandung Indah,Taman Hiburan,Bandung,50000,4.0,0.092212
4,Tektona Waterpark,Taman Hiburan,Bandung,60000,3.8,0.082095


In [133]:

# Label relevansi awal: 1 = relevan, 0 = tidak relevan
label_baseline_taman = [1, 0, 1, 1, 1]
label_expansion_taman = [1, 1, 1, 1, 1]

p5_baseline_taman = sum(label_baseline_taman) / 5
p5_expansion_taman = sum(label_expansion_taman) / 5

print("Precision@5 baseline:", round(p5_baseline_taman, 3))
print("Precision@5 expansion:", round(p5_expansion_taman, 3))
print(
    "Selisih:",
    round(p5_expansion_taman - p5_baseline_taman, 3)
)


Precision@5 baseline: 0.8
Precision@5 expansion: 1.0
Selisih: 0.2


In [134]:

import pandas as pd

rekap_evaluasi = pd.DataFrame([
    {
        "query": "budaya di Bandung",
        "Precision@5_baseline": 1.0,
        "Precision@5_expansion": 1.0,
    },
    {
        "query": "tempat rekreasi keluarga di Bandung",
        "Precision@5_baseline": 0.8,
        "Precision@5_expansion": 0.8,
    },
    {
        "query": "wisata alam di Semarang",
        "Precision@5_baseline": 1.0,
        "Precision@5_expansion": 1.0,
    },
    {
        "query": "taman bermain di Bandung",
        "Precision@5_baseline": 0.8,
        "Precision@5_expansion": 1.0,
    },
])

rekap_evaluasi["selisih"] = (
    rekap_evaluasi["Precision@5_expansion"]
    - rekap_evaluasi["Precision@5_baseline"]
)

display(rekap_evaluasi)

print(
    "Macro-average baseline:",
    round(rekap_evaluasi["Precision@5_baseline"].mean(), 3)
)
print(
    "Macro-average expansion:",
    round(rekap_evaluasi["Precision@5_expansion"].mean(), 3)
)


,query,Precision@5_baseline,Precision@5_expansion,selisih
0,budaya di Bandung,1.0,1.0,0.0
1,tempat rekreasi keluarga di Bandung,0.8,0.8,0.0
2,wisata alam di Semarang,1.0,1.0,0.0
3,taman bermain di Bandung,0.8,1.0,0.2


Macro-average baseline: 0.9
Macro-average expansion: 0.95


In [135]:

# Periksa deskripsi destinasi hasil query expansion
nama_destinasi = hasil_expansion_taman["Place_Name"].tolist()

kolom = ["Place_Name", "Category", "City", "Description"]

audit_taman = df.loc[
    df["Place_Name"].isin(nama_destinasi),
    kolom
].copy()

display(audit_taman)


,Place_Name,Category,City,Description
213,Trans Studio Bandung,Taman Hiburan,Bandung,"Trans Studio Bandung adalah kawasan wisata terpadu di Kota Bandung, Indonesia. Trans Studio dibangun pada areal seluas 4,2 hektare dengan investasi mencapai Rp 2 triliun sehingga menjadikan sebagai kawasan hiburan terluas dan terlengkap di Bandung. Wahana yang terdapat menawarkan petualangan dan sensasi baru yang tidak kalah menantang seperti pendahulunya yaitu Trans Studio Resort Makassar yang dibuka pada tahun 2009. Trans Studio Bandung dibuka untuk umum pada tanggal 18 Juni 2011. Trans Studio Bandung adalah taman bermain yang paling komplet di kota Bandung"
236,Panghegar Waterboom Bandung,Taman Hiburan,Bandung,"Panghegar Waterboom Bandung merupakan salah satu destinasi wisata yang wajib dikunjungi bagi wisatawan yang hendak mengajak buah hatinya bermain air. Terletak di Buah Batu, lokasi ini kerap kali diincar saat akhir pekan. Tak hanya menyuguhkan kolam renang, tempat ini juga menyediakan berbagai wahana menarik yang sayang jika dilewatkan. Terdiri dari seluncuran air yang dibagi untuk 2 kategori yaitu seluncuran orang dewasa dan seluncuran anak-anak. Destinasi rekreasi keluarga menawarkan beragam kegiatan seru, baik untuk anak-anak maupun dewasa. Terdapat beberapa jenis kolam yang bisa dicoba: kolam ombak, kolam mini, kolam arus, dan lain sebagainya. Ada pun yang paling baru di Panghegar Waterboom Bandung yaitu kolam air panas yang disajikan bagi wisatawan yang hendak berendam."
266,Panama Park 825,Taman Hiburan,Bandung,"Panama Park Bandung menjadi salah satu tempat bermain di Bandung, Jawa Barat. Wahana rekreasi yang disebut-sebut terbesar di Bandung ini dibuka sejak Mei 2019 lalu. Ada beberapa permainan yang ditawarkan di arena ini, misalnya playground hingga snow park yang berisi salju. Pengguna pun bisa merasakan sensasi salju sebenarnya."
280,Tektona Waterpark,Taman Hiburan,Bandung,"Tektona Waterpark sebuah wahana yang memberikan sensasi bermain air bersama keluarga dan sahabat anda, memberikan keceriaan bagi anak-anak maupun pengunjung dewasa dengan fasilitas modern diantaranya Kolam Ombak dan Kolam Arus, Kolam Anak dan Balita, Semi Olympic serta Mini Slider untuk anak-anak dan Mega Slider untuk dewasa dengan ketinggian hampir dari 30 meter, yang akan memicu adrenaline para pengunjung. Selain itu wahana Tektona Waterpark juga menyediakan pojok kuliner, gazebo serta jasa pompa dan foto aksi yang dapat diperoleh di sudut wahana Tektona Waterpark, dan juga selalu tersedia ruang ganti serta ruang bilas di dalam kawasan Tektona Waterpark."
290,Water Park Bandung Indah,Taman Hiburan,Bandung,Bandung Indah Waterpark merupakan salah satu kolam renang yang ada di Bandung. Menyediakan berbagai wahana permainan air yang cocok bagi keluarga. Terdapat kolam renang untuk dewasa dan anak-anak. Pengunjung dapat melakukan olahraga renang dan rekreasi.


In [136]:

from pathlib import Path

folder_laporan = Path("../reports")
folder_laporan.mkdir(parents=True, exist_ok=True)

rekap_evaluasi.to_csv(
    folder_laporan / "evaluasi_query_expansion.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Laporan evaluasi berhasil disimpan.")
print("Lokasi: reports/evaluasi_query_expansion.csv")


Laporan evaluasi berhasil disimpan.
Lokasi: reports/evaluasi_query_expansion.csv
